# Position

Lets see if we can do interesting small boards

In [ ]:
#| default_exp game/mechanics/position

In [ ]:
import sys
import math
from fastcore.basics import patch
from fasthtml.common import *
from fasthtml.jupyter import *

In [ ]:
from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord 
from HexMagic.primitives import HexGrid, HexPosition ,  HexRegion, GosperCurve, windy_edge , unique_windy_edge

import numpy as np

from HexMagic.terrain import Terrain
from HexMagic.climate import ClimatePreset, Climate, TerraDemo, TerrainFactory
from HexMagic.geology import Geology, DrainageBasins, SoilSystem
from HexMagic.overlay import  TerrainDisplay, TerrainOverlay, ClimateOverlay, TerraDemo, DrainageBasins, OverlaySpec
from HexMagic.overlay import TerrainDisplay, CreamOverlay, RiverOverlay, ClimateOverlay

In [ ]:
from HexMagic.game.globals import appRoutes,  webMe, globalStore, ensure_user,new_game_page, create_game, create_world, invalidate_cache, showUsers, logging
from HexMagic.game.data import Settlement, Kingdom, Piece, TradeRoute, GameBoard, ActiveGame, CountryFlag, _map_point
from HexMagic.game.data import Piece, PieceType, Instruction, InstructionList, Squad
from HexMagic.game.data import RANK_ORDER 
from HexMagic.game.piece import CountryFlag, DiagramGlyphs,PieceBoard, PieceStep, _move_cost,PieceBoardPlan
from HexMagic.game.piece import SquadPlanOverlay, PieceList, FormationType, SurroundMode, TroopPath, _facing_toward
from HexMagic.game.piece import GroupedPieceList, SquadPlanOverlay, piece_plan_overlay, SquadSymbolOverlay #, SquadVisionOverlay
from HexMagic.game.piece import FormationType, _formation_offsets

from HexMagic.game.piece import pieces_center, StatBar, SquadVisionOverlay, GameParts,SettlementOverlay, KingdomNamesOverlay, PieceOverlay

In [ ]:
from HexMagic.game.food import FoodOverlay, FoodYield, PieceOverlay, PieceDataOverlay, PieceArrowOverlay, SquadSymbolOverlay, food_triage, pre_march_distribute, compose_callbacks

In [ ]:
from HexMagic.game.mechanics.simulator import TurnReport, SimEventType, FoodSimulator, squad_chart, SquadFoodTrailOverlay


In [ ]:
import pyomo.environ as pyo
import optuna

## Substainabilty

In [ ]:
def sustain_cost(hex_idx, grid, food_tiers, elevations,
                 queen_diet=1.9, pawn_diet=1.9, pawn_harvest=2):
    """How many pawns needed to sustain a queen camped at hex_idx.
    Returns int (1-99) or -1 for unsustainable.
    """
    # Best food tier in ring-1 (where pawns would sit)
    best_tier = 0
    for nb in grid.neighborsOf(hex_idx):
        if 0 <= nb < len(food_tiers) and elevations[nb] > 0:
            best_tier = max(best_tier, int(food_tiers[nb]))
    
    # Net food a pawn can export per turn (harvest every other turn)
    net = 0.5 * pawn_harvest * best_tier - pawn_diet
    
    if net <= 0:
        return -1  # unsustainable
    
    return math.ceil(queen_diet / net)


In [ ]:
def SustainOverlay(queen, pawn_template=None, **kw) -> OverlaySpec:
    """Color hexes by number of pawns needed to sustain the queen there.
    
    Green (1-2), Yellow (3-4), Orange (5-6), Red (7+), Gray (impossible).
    """
    COLORS = {
        1: '#27ae60', 2: '#2ecc71',  # great
        3: '#f1c40f', 4: '#f39c12',  # expensive
        5: '#e67e22', 6: '#d35400',  # costly
    }
    DEFAULT = '#e74c3c'  # 7+
    IMPOSSIBLE = '#444444'
    
    def render(ctx):
        grid = ctx.grid
        fy = FoodYield(ctx.terrain, ctx.basins)
        fy.compute()
        food_tiers = fy.tiers
        elevs = ctx.terrain.elevations
        
        parts = []
        for i in range(len(elevs)):
            if elevs[i] <= 0:
                continue
            cost = sustain_cost(i, grid, food_tiers, elevs,
                                queen_diet=queen.diet)
            if cost == -1:
                color = IMPOSSIBLE
            else:
                color = COLORS.get(cost, DEFAULT)
            
            h = grid.hexes[i]
            pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in h.vertices())
            parts.append(
                f'<polygon points="{pts}" fill="{color}" opacity="0.3"/>')
        
        return '\n'.join(parts)
    
    return OverlaySpec("sustain_cost", render, priority=45)


## Camp Assign

In [ ]:
def _solver_ok(result):
    return (result.solver.status == pyo.SolverStatus.ok and
            result.solver.termination_condition == pyo.TerminationCondition.optimal)


In [ ]:
@patch
def camp_assign(squad:Squad, target_hex, grid, elevations, food_tiers,
                countries=None, max_ring=3,
                elevation_mult=0.005) -> dict:
    """Facility-location MIP: station feeders on fertile tiles to cover all demand.
    
    Minimizes total combat value assigned to feeding duty,
    keeping the best fighters free.
    """
    alive = squad.alive
    if not alive: return {}
    
    # ── Candidate tiles (facilities) ──
    tiles = []  # (hex_idx, tier)
    occupied = {target_hex}
    for ring in range(1, max_ring + 1):
        for hp in HexPosition.origin().ring(ring):
            idx = grid.hexposition_to_index(hp, target_hex)
            if idx < 0 or idx in grid.invalidRegion: continue
            if elevations[idx] <= 0: continue
            if countries is not None and countries[idx] < 0: continue
            tier = int(food_tiers[idx]) if 0 <= idx < len(food_tiers) else 0
            if tier < 1: continue
            tiles.append((idx, tier))
    
    if not tiles: return {}
    
    N = len(alive)      # pieces (potential feeders AND consumers)
    T = len(tiles)       # candidate tiles
    
    # ── Precompute: which (tile, facing) can see which piece locations ──
    # For each tile, compute sight cone facing target_hex
    visible = np.zeros((T, N), dtype=bool)
    for t, (tidx, _) in enumerate(tiles):
        facing = _facing_toward(grid, tidx, target_hex)
        facing_dir = HexPosition.directions()[facing]
        elev = max(0, elevations[tidx])
        eff_sight = 3 + elev * elevation_mult
        fov = set()
        for hp in HexPosition.origin().field_of_view(facing_dir, int(eff_sight)):
            fov_idx = grid.hexposition_to_index(hp, tidx)
            if fov_idx >= 0: fov.add(fov_idx)
        # Target hex and ring-1 around it are where consumers will be
        for c, piece in enumerate(alive):
            # Consumer location = target_hex (they're camped)
            if target_hex in fov:
                visible[t, c] = True
    
    # ── Production capacity per (feeder, tile) ──
    # feed_turns = move_strength - 1 harvest actions per turn
    production = np.zeros((N, T))
    for f, piece in enumerate(alive):
        for t, (tidx, tier) in enumerate(tiles):
            production[f, t] = piece.harvest_strength * tier
    
    diet = np.array([p.diet for p in alive])
    combat = np.array([getattr(p, 'attack_strength', 0) for p in alive])
    
    # ── Pyomo model ──
    m = pyo.ConcreteModel()
    m.F = pyo.RangeSet(0, N - 1)    # feeders (any piece)
    m.T = pyo.RangeSet(0, T - 1)    # tiles
    m.C = pyo.RangeSet(0, N - 1)    # consumers (same pieces)
    
    # assign[f,t] = 1 if piece f is stationed on tile t
    m.assign = pyo.Var(m.F, m.T, domain=pyo.Binary)
    
    # flow[f,t,c] = food from feeder f on tile t to consumer c
    m.flow = pyo.Var(m.F, m.T, m.C, domain=pyo.NonNegativeReals)
    
    # ── Objective: minimize combat value of assigned feeders ──
    m.obj = pyo.Objective(
        expr=sum(combat[f] * m.assign[f, t] for f in m.F for t in m.T),
        sense=pyo.minimize)
    
    # ── Each piece assigned to at most 1 tile ──
    m.one_tile = pyo.Constraint(m.F, rule=lambda m, f:
        sum(m.assign[f, t] for t in m.T) <= 1)
    
    # ── Each tile has at most 1 piece ──
    m.one_per_tile = pyo.Constraint(m.T, rule=lambda m, t:
        sum(m.assign[f, t] for f in m.F) <= 1)
    
    # ── Flow only if assigned AND visible ──
    M = max(diet) * 2  # big-M
    def flow_limit(m, f, t, c):
        if not visible[t, c]:
            return m.flow[f, t, c] == 0
        return m.flow[f, t, c] <= M * m.assign[f, t]
    m.flow_cap = pyo.Constraint(m.F, m.T, m.C, rule=flow_limit)
    
    # ── Production capacity per feeder-tile ──
    m.prod_cap = pyo.Constraint(m.F, m.T, rule=lambda m, f, t:
        sum(m.flow[f, t, c] for c in m.C) <= production[f, t] * m.assign[f, t])
    
    # ── Every consumer's diet must be met ──
    m.demand = pyo.Constraint(m.C, rule=lambda m, c:
        sum(m.flow[f, t, c] for f in m.F for t in m.T) >= diet[c])
    
    # ── Solve ──
    solver = pyo.SolverFactory('appsi_highs')
    result = solver.solve(m)
    
    if not _solver_ok(result):
        return {}
    
    # ── Extract assignments ──
    orders = {}
    for f in m.F:
        piece = alive[f]
        assigned_tile = None
        for t in m.T:
            if pyo.value(m.assign[f, t]) > 0.5:
                assigned_tile = t
                break
        
        if assigned_tile is None:
            # Free piece — hold position at target
            orders[piece.id] = InstructionList(
                [Instruction.PAUSE.value], cursor=0, patrol=True)
        else:
            tidx, _ = tiles[assigned_tile]
            facing = _facing_toward(grid, tidx, target_hex)
            
            tp = piece.pathfind(tidx, grid, elevations, countries)
            rules = tp.to_rules(grid, piece.facing) if tp else []
            
            # Rotate to face target
            cur = piece.facing if not rules else _facing_toward(
                grid, tp.hexes[-2] if len(tp.hexes) >= 2 else piece.location, tidx)
            diff = (facing - cur) % 6
            if diff <= 3: rules.extend([Instruction.ROT_R.value] * diff)
            else: rules.extend([Instruction.ROT_L.value] * (6 - diff))
            
            rules.extend([Instruction.HARVEST.value, Instruction.GIVE.value])
            orders[piece.id] = InstructionList(rules, cursor=0, patrol=True)
    
        # ── Extract assignments (add to existing camp_assign) ──
    feeder_ids = set()
    tile_for = {}  # piece_id → hex_idx
    
    for f in m.F:
        piece = alive[f]
        for t in m.T:
            if pyo.value(m.assign[f, t]) > 0.5:
                feeder_ids.add(piece.id)
                tile_for[piece.id] = tiles[t][0]
                break
    
    orders['feeder_ids'] = feeder_ids
    orders['tile_for'] = tile_for
    orders['total_production'] = sum(
        alive[f].harvest_strength * tiles[t][1]
        for f in m.F for t in m.T
        if pyo.value(m.assign[f, t]) > 0.5)
    
    return orders



In [ ]:
@patch
def supply_assign(squad:Squad, target_hex, grid, elevations, food_tiers,
                countries=None, max_ring=3):
    """Facility-location MIP: pick who feeds, on which tile, minimizing combat loss."""
    alive = squad.alive
    if not alive: return {'feasible': False}
    
    # ── Candidate tiles around camp center ──
    tiles = []
    for ring in range(1, max_ring + 1):
        for hp in HexPosition.origin().ring(ring):
            idx = grid.hexposition_to_index(hp, target_hex)
            if idx < 0 or idx in grid.invalidRegion: continue
            if elevations[idx] <= 0: continue
            if countries is not None and countries[idx] < 0: continue
            tier = int(food_tiers[idx]) if 0 <= idx < len(food_tiers) else 0
            if tier < 1: continue
            tiles.append((idx, tier))
    
    if not tiles: return {'feasible': False}
    
    N, T = len(alive), len(tiles)
    prod = np.zeros((N, T))
    for f, p in enumerate(alive):
        for t, (_, tier) in enumerate(tiles):
            prod[f, t] = p.harvest_strength * tier
    
    total_diet = sum(p.diet for p in alive)
    combat = np.array([getattr(p, 'attack_strength', 0) for p in alive])
    
    m = pyo.ConcreteModel()
    m.F = pyo.RangeSet(0, N - 1)
    m.T = pyo.RangeSet(0, T - 1)
    m.x = pyo.Var(m.F, m.T, domain=pyo.Binary)
    
    m.obj = pyo.Objective(
        expr=sum(combat[f] * m.x[f, t] for f in m.F for t in m.T),
        sense=pyo.minimize)
    
    m.one_tile = pyo.Constraint(m.F, rule=lambda m, f:
        sum(m.x[f, t] for t in m.T) <= 1)
    m.one_per = pyo.Constraint(m.T, rule=lambda m, t:
        sum(m.x[f, t] for f in m.F) <= 1)
    m.demand = pyo.Constraint(
        expr=sum(prod[f, t] * m.x[f, t] for f in m.F for t in m.T) >= total_diet)
    
    solver = pyo.SolverFactory('appsi_highs')
    result = solver.solve(m)
    if not _solver_ok(result):
        return {'feasible': False}
    
    feeder_ids, tile_for = set(), {}
    total_prod = 0
    for f in m.F:
        for t in m.T:
            if pyo.value(m.x[f, t]) > 0.5:
                p = alive[f]
                feeder_ids.add(p.id)
                tile_for[p.id] = tiles[t][0]
                total_prod += prod[f, t]
    
    return dict(feasible=True, feeder_ids=feeder_ids, tile_for=tile_for,
                total_production=total_prod, total_diet=total_diet,
                net_surplus=total_prod - total_diet, tiles=tiles)


In [ ]:
def smart_placement(settlement, squad, terrain, food_tiers, grid,
                    max_ring=10, pawn_ring=3):
    """Place queen on best sustainable hex, assign pawns via MIP."""
    center = settlement.location
    elevs = terrain.elevations
    ranked = squad.by_rank()
    queen, pawns = ranked[0], ranked[1:]

    # ── Best queen hex (cheap greedy scan) ──
    best_idx, best_cost = center, 99
    for hp in HexPosition.origin().spiral(max_ring):
        idx = grid.hexposition_to_index(hp, origin_index=center)
        if idx < 0 or elevs[idx] <= 0: continue
        cost = sustain_cost(idx, grid, food_tiers, elevs, queen_diet=queen.diet)
        if 0 < cost < best_cost:
            best_cost, best_idx = cost, idx

    queen.location = best_idx
    if not pawns: return best_cost

    # ── Candidate tiles (same pattern as camp_assign) ──
    tiles = []
    for ring in range(1, pawn_ring + 1):
        for hp in HexPosition.origin().ring(ring):
            idx = grid.hexposition_to_index(hp, origin_index=best_idx)
            if idx < 0 or idx in grid.invalidRegion or elevs[idx] <= 0: continue
            tier = int(food_tiers[idx]) if 0 <= idx < len(food_tiers) else 0
            if tier >= 1: tiles.append((idx, tier))

    if not tiles: return best_cost

    N, T = len(pawns), len(tiles)
    diet    = np.array([p.diet for p in pawns])
    combat  = np.array([getattr(p, 'attack_strength', 0) for p in pawns])
    harvest = np.array([p.harvest_strength for p in pawns])
    tiers   = np.array([t[1] for t in tiles])

    # ── MIP: assign pawns to tiles ──
    m = pyo.ConcreteModel()
    m.P = pyo.RangeSet(0, N - 1)
    m.T = pyo.RangeSet(0, T - 1)
    m.assign = pyo.Var(m.P, m.T, domain=pyo.Binary)

    # Maximize food surplus, tiebreak: keep strong fighters unassigned
    α = 10
    m.obj = pyo.Objective(
        expr=sum(α * harvest[p] * tiers[t] * m.assign[p, t]
                 for p in m.P for t in m.T)
           - sum(combat[p] * sum(m.assign[p, t] for t in m.T)
                 for p in m.P),
        sense=pyo.maximize)

    # Each pawn → at most 1 tile
    m.one_tile = pyo.Constraint(m.P, rule=lambda m, p:
        sum(m.assign[p, t] for t in m.T) <= 1)

    # Each tile → at most 1 pawn
    m.one_per_tile = pyo.Constraint(m.T, rule=lambda m, t:
        sum(m.assign[p, t] for p in m.P) <= 1)

    # Total harvest must cover everyone's diet
    total_diet = queen.diet + float(diet.sum())
    m.sustain = pyo.Constraint(rule=
        sum(harvest[p] * tiers[t] * m.assign[p, t]
            for p in m.P for t in m.T) >= total_diet)

    solver = pyo.SolverFactory('appsi_highs')
    result = solver.solve(m)
    if not _solver_ok(result): return best_cost

    # ── Extract placements ──
    for p in m.P:
        for t in m.T:
            if pyo.value(m.assign[p, t]) > 0.5:
                tidx = tiles[t][0]
                pawns[p].location = tidx
                pawns[p].facing = _facing_toward(grid, tidx, best_idx)
                break

    return best_cost


In [ ]:
@patch
def camp_economics(squad:Squad, camp_result, tiles, food_tiers):
    """Compute per-turn surplus from a camp_assign solution.
    
    Returns (total_production, total_diet, net_surplus, deficit_to_fill)
    """
    alive = squad.alive
    total_diet = sum(p.diet for p in alive)
    
    # Sum production from assigned feeders
    total_prod = 0
    for p in alive:
        tile_idx = camp_result.get('tile_for', {}).get(p.id)
        if tile_idx is not None:
            tier = int(food_tiers[tile_idx])
            total_prod += p.harvest_strength * tier
    
    deficit = sum(max(0, p.food_capacity - p.food) for p in alive)
    net = total_prod - total_diet
    
    return dict(production=total_prod, diet=total_diet,
                net_surplus=net, deficit=deficit,
                turns_to_fill=math.ceil(deficit / net) if net > 0 else float('inf'))


In [ ]:
@patch
def leapfrog_plan(self: Squad, leader_path: TroopPath,
                  grid: HexGrid, elevations: np.ndarray,
                  formation: FormationType = None,
                  countries: np.ndarray = None,
                  food_tiers: np.ndarray = None,
                  min_food_tier: int = 2,
                  max_camp_turns: int = 30,
                  capacity_pct: float = 0.95,
                  max_legs: int = 20,
                  leader: Piece = None,
                  elevation_mult: float = 0.005) -> list[dict]:
    """Leapfrog march using camp_assign MIP at each waypoint.
    
    At each stop:
      1. camp_assign solves facility-location for optimal feeder placement
      2. Analytically compute recovery turns until capacity_pct reached
      3. Apply food gains, redistribute, depart
    """
    if leader is None:
        leader = self.by_rank()[0]
    
    legs = []
    pos = 0
    
    for _ in range(max_legs):
        if pos >= len(leader_path) - 1:
            break
        
        # ── Phase 0: Bishop tops up before departure ──
        pre_march_distribute(self)
        
        # ── Phase 1: Farthest reachable waypoint ──
        sub_hexes = leader_path.hexes[pos:]
        sub_cost = sum(_move_cost(elevations, sub_hexes[k], sub_hexes[k+1])
                       for k in range(len(sub_hexes) - 1))
        sub_path = TroopPath(sub_hexes, sub_cost)
        
        leg = self.march_range(sub_path, grid, elevations,
                               formation=formation, countries=countries,
                               food_tiers=food_tiers,
                               min_food_tier=min_food_tier, leader=leader)
        
        if not leg['feasible'] or leg['waypoint'] == 0:
            break
        
        # ── Phase 2: Deduct travel food ──
        wp_hex = leg['hex_idx']
        for p in self.alive:
            tp = leg['paths'].get(p.id)
            turns = math.ceil(tp.cost / max(p.move_strength, 1)) if tp else 0
            p.food = max(0, p.food - p.diet * turns)
            p.location = wp_hex
        
        # ── Phase 3: Camp — solve facility-location MIP ──
        camp = self.camp_assign( wp_hex, grid, elevations, food_tiers,
                          countries=countries, elevation_mult=elevation_mult)
        
        # ── Phase 4: Analytical recovery ──
        econ = self.camp_economics( camp, food_tiers=food_tiers)
        net = econ['net_surplus']
        
        if net <= 0:
            # Can't sustain here — record and hope next leg is better
            leg['camp'] = camp
            leg['econ'] = econ
            leg['camp_turns'] = 0
            leg['leg_number'] = len(legs)
            legs.append(leg)
            pos += leg['waypoint']
            continue
        
        # How many turns until everyone is at capacity_pct?
        target_food = {p.id: p.food_capacity * capacity_pct for p in self.alive}
        deficit = sum(max(0, target_food[p.id] - p.food) for p in self.alive)
        camp_turns = math.ceil(deficit / net)
        camp_turns = min(camp_turns, max_camp_turns)
        
        # Apply food gains proportionally (each piece gets diet + share of surplus)
        # Feeders harvest, eat, share. Non-feeders just eat what's given.
        feeder_ids = camp.get('feeder_ids', set())
        for t in range(camp_turns):
            # Feeders harvest
            for p in self.alive:
                if p.id in feeder_ids:
                    tile = camp['tile_for'].get(p.id)
                    if tile is not None:
                        tier = int(food_tiers[tile])
                        gained = p.harvest_strength * tier
                        p.food = min(p.food + gained, p.food_capacity)
            # Everyone eats
            for p in self.alive:
                p.food = max(0, p.food - p.diet)
            # Feeders share surplus to neediest
            pre_march_distribute(self)
        
        # ── Phase 5: Final redistribute before next leg ──
        pre_march_distribute(self)
        
        leg['camp'] = camp
        leg['econ'] = econ
        leg['camp_turns'] = camp_turns
        leg['leg_number'] = len(legs)
        legs.append(leg)
        
        pos += leg['waypoint']
    
    return legs


In [ ]:
@patch
def apply_food_profile(self: GameParts, soil_scale: float = 1.0):
    """Set food stats on all pieces based on balanced profile.
    
    Design intent:
      - Queen is expensive to feed (3.5× pawn), can't self-sustain
      - Pawns are field workers: decent harvest, small pack
      - Bishops are supply wagons: bad harvest, huge capacity
      - Knights are fast but hungry scouts
      - Rooks are heavy but carry well
      - King eats like a queen (royalty!)
    """
    PROFILES = {
        #                  diet  harvest  capacity
        PieceType.PAWN:   (2.0,  2.0,     12),
        PieceType.BISHOP: (2.5,  1.0,    100),
        PieceType.KNIGHT: (3.0,  1.5,     20),
        PieceType.ROOK:   (3.5,  1.0,     40),
        PieceType.QUEEN:  (7.0,  0.5,     30),
        PieceType.KING:   (6.0,  0.5,     25),
    }
    
    count = {t.name: 0 for t in PieceType}
    for country in self.board.kingdoms:
        for city in country.settlements:
            for piece in city.citizens:
                profile = PROFILES.get(piece.piece_type)
                if profile is None:
                    continue
                piece.diet, piece.harvest_strength, piece.food_capacity = profile
                piece.food = piece.food_capacity  # start full
                count[piece.piece_type.name] += 1
    
    # Apply soil scaling if needed

    fy = FoodYield(self.terr, self.basin)
    if soil_scale != 1.0:
        fy.soil_mult = [s * soil_scale for s in fy.soil_mult]
    fy.compute()

    
    print(f"Food profile applied:")
    for ptype, (d, h, c) in PROFILES.items():
        n = count.get(ptype.name, 0)
        net_t5 = 0.5 * h * 5 - d  # net at tier 5
        label = "✅" if net_t5 > 0 else "❌"
        print(f"  {ptype.name:8s} ×{n:2d}: diet={d:.1f} harvest={h:.1f} "
              f"cap={c:3d}  net@tier5={net_t5:+.1f} {label}")
    
    queen_d = PROFILES[PieceType.QUEEN][0]
    pawn_net = 0.5 * PROFILES[PieceType.PAWN][1] * 5 - PROFILES[PieceType.PAWN][0]
    bish_gap = PROFILES[PieceType.BISHOP][2] / (PROFILES[PieceType.BISHOP][0] + queen_d)
    print(f"\n  Pawns to sustain queen @tier5: {math.ceil(queen_d / max(pawn_net, 0.01))}")
    print(f"  Bishop gap crossing: ~{int(bish_gap)} barren hexes")
    return fy


In [ ]:
@patch
def playability_check(self: GameBoard, max_pawns=5):
    """Check if all settlements are connected via sustainable corridors.
    
    Returns dict with:
      - connected: bool
      - coverage: float (fraction of land that's sustainable)
      - corridors: dict of settlement pairs → path exists
      - sustainable_count / land_count: raw numbers
    """
    fy = FoodYield(self.terrain, self.basins); fy.compute()
    food_tiers = fy.tiers
    elevs = self.terrain.elevations
    grid = self.grid
    
    # Build sustainable hex set
    sustainable = set()
    for i in range(len(elevs)):
        if elevs[i] <= 0:
            continue
        cost = sustain_cost(i, grid, food_tiers, elevs)
        if 0 < cost <= max_pawns:
            sustainable.add(i)
    
    land = sum(1 for e in elevs if e > 0)
    coverage = len(sustainable) / max(land, 1)
    
    # BFS connectivity between settlements using only sustainable hexes
    def bfs_connected(start, end):
        if start not in sustainable or end not in sustainable:
            return False
        visited = {start}
        queue = [start]
        while queue:
            current = queue.pop(0)
            if current == end:
                return True
            for nb in grid.neighborsOf(current):
                if nb in sustainable and nb not in visited:
                    visited.add(nb)
                    queue.append(nb)
        return False
    
    settlements = self.settlements
    corridors = {}
    locs = [s.location for s in settlements]
    all_connected = True
    for i in range(len(locs)):
        for j in range(i+1, len(locs)):
            ok = bfs_connected(locs[i], locs[j])
            corridors[(i, j)] = ok
            if not ok:
                all_connected = False
    
    return {
        'connected': all_connected,
        'coverage': coverage,
        'corridors': corridors,
        'sustainable_count': len(sustainable),
        'land_count': land,
    }


In [ ]:
@patch
def loadGeology(self: GameParts, geo: Geology):
    self.terr = geo.terrain
    self.grid = self.terr.hexGrid
    self.builder = self.grid.builder

    self.board = GameBoard(self.terr, 2)
    self.basin = geo.basins

    self.country = self.board.kingdoms[0]
    self.home = self.country.settlements[0]
    self.squads = Squad.squads(6, self.country.flag)
    self.queenGuard = self.squads[0]
    for pType in [PieceType.PAWN, PieceType.PAWN, PieceType.PAWN, PieceType.PAWN,
                  PieceType.PAWN, PieceType.BISHOP, PieceType.BISHOP, PieceType.QUEEN]:
        piece = self.home.recruit(pType, flag=self.queenGuard.flag)
        self.queenGuard.pieces.append(piece)

    # Initial placement so pieces have locations
    self.home.place_squad_spiral(self.queenGuard, self.terr)
    self.queen = self.queenGuard.by_rank()[0]

    # Set diet/harvest/capacity on all pieces
    self.apply_food_profile()

    # Compute food tiers, then re-place using sustainability logic
    fy = FoodYield(self.terr, self.basin)
    fy.compute()
    best_cost = smart_placement(self.home, self.queenGuard, self.terr, fy.tiers, self.grid)
    print(f"Smart placement: queen at {self.queen.location}, sustain cost = {best_cost} pawns")


### Show Demo

In [ ]:
showDemo = True

In [ ]:
showDemo = False

In [ ]:
myStuff = GameParts()

In [ ]:
def demoTerr():

    world = TerrainFactory.create_world(
        bounds= MapRect(MapCord(0, 0), MapSize(300, 300)),
        preset='temperate',
        name='Maiden Lane',
        radius=15,
        lon_span=10.0,
        num_plates=8,
        subdivisions=3,
        ocean_fraction=0.3,
        oceanic_sides=['N'],  # Ocean on east and west
        terrain_age='young',  # Sharp, dramatic features
        formation_type='ridge',  # Creates ridge formations
        elevation_scale=1.5,  # Exaggerate the heights
        erosion_age=0.1,  # Minimal erosion for sharp peaks
        num_lakes=0,
        seed=23,
        debug=True
    )

    sampleMap = world.terrain
    sampleMap.repalette()

    for i in range(len(sampleMap.hexGrid.hexes)):
        sampleMap.hexGrid.hexes[i].label = str(i)

    myStuff.loadGeology(world)
    myStuff.grid.adjustRadius(40)

    return myStuff

In [ ]:
demoTerr()

In [ ]:
for i in range(len(myStuff.grid.hexes)):
    myStuff.grid.hexes[i].label = str(i)

In [ ]:
TerrainDisplay(
    TerrainOverlay(),
    SettlementOverlay(),
    SquadVisionOverlay([myStuff.queenGuard]),
    SustainOverlay(queen=myStuff.queen),
    RiverOverlay(max_width=5),
    terrain=myStuff.terr,
    board=myStuff.board,
    basins=myStuff.basin,
    debug=not showDemo
    )

In [ ]:
GroupedPieceList(myStuff.queenGuard, grid= myStuff.grid)

fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()
food_tiers = fy.tiers
grid, elevs = myStuff.grid, myStuff.terr.elevations
countries = myStuff.terr.fields.get('country')
queen = myStuff.queenGuard.by_rank()[0]

# Queen pathfinds west as far as she can
far_west = 236

route = queen.pathfind(far_west, grid, elevs, countries)
print(f"Leader route: {len(route)} hexes, cost {route.cost:.0f}")

# Compare formation vs broken
analysis = myStuff.queenGuard.march_analysis(
    route, grid, elevs,
    formation=FormationType.COLUMN,
    countries=countries,
    food_tiers=food_tiers,
    min_food_tier=2)

print(f"\nFormation range: {analysis['formation_range']} hexes (COLUMN)")
print(f"Broken range:    {analysis['broken_range']} hexes")
print(f"Bonus for breaking formation: +{analysis['range_bonus']} hexes")

if analysis['formation']['feasible']:
    print(f"\nFormation arrivals:")
    print(analysis['formation']['arrivals'].to_string(index=False))

if analysis['broken']['feasible']:
    print(f"\nBroken arrivals:")
    print(analysis['broken']['arrivals'].to_string(index=False))

# Leapfrog in broken formation
legs = myStuff.queenGuard.leapfrog_plan(
    route, grid, elevs,
    formation=None, countries=countries,
    food_tiers=food_tiers, recovery_turns=4)

for leg in legs:
    fc = leg['feed_cover']
    print(f"\nLeg {leg['leg_number']}: {leg['waypoint']} hexes → hex {leg['hex_idx']} "
          f"(cost {leg['path_cost']})")
    print(f"  Feeders: {[p.name for p in fc['feeders']]}")
    print(f"  Free:    {[p.name for p in fc['free']]}")

total = sum(l['waypoint'] for l in legs)
print(f"\nTotal coverage: {total}/{len(route)} hexes in {len(legs)} legs")

I want to see the plan for the queen going to 236 with her squad in support. They stop when they need to feed and set up so that queen can be fed.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

You've already got the route and leapfrog legs computed. To visualize the full march-and-feed plan, I'd suggest running a `FoodSimulator` with `food_triage` as a callback so pieces auto-harvest when hungry, then layering a `SquadFoodTrailOverlay` on your terrain display.

The key steps would be:

1. **Fresh state** — re-run `demoTerr()` so piece food/locations are pristine
2. **Set march orders** — use `march_to(236, ...)` or manually set instructions from the leapfrog plan
3. **Simulate** — `FoodSimulator(..., callbacks={squad.id: food_triage}).run(30)`
4. **Display** — add `SquadFoodTrailOverlay(sim, squad)` to your `TerrainDisplay`

Want me to write that up as a code cell, or would you prefer to build it yourself?

setting the march order for the pieces to their intructions

orders = myStuff.queenGuard.march_to(
    236, FormationType.BOX,
    grid=myStuff.grid, elevations=myStuff.terr.elevations,
    leader=myStuff.queen, countries=myStuff.terr.fields.get('country'),
    food_tiers=food_tiers, sync=True)

for p in myStuff.queenGuard.alive:
    if p.id in orders:
        p.instructions = orders[p.id]
        print(f"{p.name:10s} ({p.piece_type.name:6s}): {len(p.instructions.rules)} instructions")


In [ ]:
showDemo = False

In [ ]:
TerrainDisplay(
    TerrainOverlay(),
    SettlementOverlay(),
    SquadVisionOverlay([myStuff.queenGuard]),
    SquadPlanOverlay([myStuff.queenGuard]),
    RiverOverlay(max_width=5),
    terrain=myStuff.terr,
    board=myStuff.board,
    basins=myStuff.basin,
    debug=not showDemo
    )

So can we simulate it so that it stops to feed?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes! The `FoodSimulator` + `food_triage` callback is exactly this. `food_triage` watches average food turns remaining and auto-prepends HARVEST/GIVE instructions when the squad gets hungry.

```python
# Run the simulator with food_triage so the squad stops to feed
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()
food_tiers = fy.tiers

sim = FoodSimulator(
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=food_tiers,
    pieces=myStuff.queenGuard.alive,
    countries=myStuff.terr.fields.get('country'),
    squads=[myStuff.queenGuard],
    callbacks={myStuff.queenGuard.id: food_triage},
)

sim.run(num_turns=40)
sim.summary()
```

That will show you the summary. Then to see the charts:

```python
sim  # renders the alive/health/food line charts via __ft__
```

And to see the food trail on the map (pips showing food level at each turn, arrows for shares, X for deaths):

```python
TerrainDisplay(
    TerrainOverlay(),
    SettlementOverlay(),
    FoodOverlay(),
    SquadFoodTrailOverlay(sim, myStuff.queenGuard),
    PieceOverlay([myStuff.queenGuard]),
    RiverOverlay(max_width=5),
    terrain=myStuff.terr,
    board=myStuff.board,
    basins=myStuff.basin,
    debug=not showDemo
)
```

The first cell is the one to try — it'll tell you how many turns until they arrive (or run out of food), and whether `food_triage` kicked in to make them stop and harvest along the way.

In [ ]:
# Run the simulator with food_triage so the squad stops to feed
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()
food_tiers = fy.tiers

sim = FoodSimulator(
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=food_tiers,
    pieces=myStuff.queenGuard.alive,
    countries=myStuff.terr.fields.get('country'),
    squads=[myStuff.queenGuard],
    callbacks={myStuff.queenGuard.id: food_triage},
)

sim.run(num_turns=40)
sim.summary()


In [ ]:
show(sim)

In [ ]:
GroupedPieceList(myStuff.queenGuard, grid= myStuff.grid)

So we need something at the end of the route that can put the pieces into a substainable food pattern.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Exactly — right now `food_triage` is reactive (prepends harvest/give when hungry), but once they arrive at 236 they need a **proactive camp pattern**: feeders stationed on good tiles, facing the queen, running HARVEST→GIVE on loop.

The pieces you need are already there in the codebase — `feed_cover` picks who should be feeders, and `supply_line` does the "station on fertile tile facing a path" logic. What's missing is a simpler **"set up camp"** function for the destination. Here's the shape:

```python
def setup_camp(squad, target_hex, grid, elevations, food_tiers,
               countries=None, max_ring=3, leader=None):
    """After arrival: assign feeders to nearby fertile tiles with HARVEST→GIVE patrol.
    
    1. feed_cover picks optimal feeders vs free pieces
    2. Each feeder gets best unclaimed food tile in ring 1..max_ring, facing target
    3. Feeders get [HARVEST, GIVE] patrol; free pieces get [PAUSE] patrol
    
    Returns dict of piece.id → InstructionList
    """
    if leader is None:
        leader = squad.by_rank()[0]
    
    cover = squad.feed_cover(food_tiers, objective='attack')
    feeder_ids = {p.id for p in cover['feeders']}
    
    # Find candidate tiles sorted by food tier (best first)
    candidates = []
    for ring in range(1, max_ring + 1):
        for hp in HexPosition.origin().ring(ring):
            idx = grid.hexposition_to_index(hp, target_hex)
            if idx < 0 or idx in grid.invalidRegion: continue
            if elevations[idx] <= 0: continue
            if countries is not None and countries[idx] < 0: continue
            tier = int(food_tiers[idx]) if 0 <= idx < len(food_tiers) else 0
            if tier < 1: continue
            candidates.append((idx, tier, ring))
    candidates.sort(key=lambda c: (-c[1], c[2]))  # best tier, closest
    
    claimed = {target_hex}
    orders = {}
    
    for piece in squad.alive:
        if piece.id not in feeder_ids:
            # Free piece — queen, fighters — hold position
            orders[piece.id] = InstructionList(
                [Instruction.PAUSE.value], cursor=0, patrol=True)
            continue
        
        # Find best unclaimed tile for this feeder
        best = None
        for idx, tier, ring in candidates:
            if idx not in claimed:
                best = (idx, tier)
                claimed.add(idx)
                break
        
        if best is None:
            # No tile available — harvest in place
            orders[piece.id] = InstructionList(
                [Instruction.HARVEST.value, Instruction.GIVE.value],
                cursor=0, patrol=True)
            continue
        
        station_idx, _ = best
        facing = _facing_toward(grid, station_idx, target_hex)
        
        # Build: pathfind to station → rotate to face queen → harvest/give loop
        tp = piece.pathfind(station_idx, grid, elevations, countries)
        if tp:
            rules = tp.to_rules(grid, piece.facing)
        else:
            rules = []
        
        # Rotate to face target
        end_facing = piece.facing if not tp else _facing_toward(
            grid, tp.hexes[-2] if len(tp.hexes) >= 2 else piece.location, station_idx)
        diff = (facing - end_facing) % 6
        if diff <= 3:
            rules.extend([Instruction.ROT_R.value] * diff)
        else:
            rules.extend([Instruction.ROT_L.value] * (6 - diff))
        
        # Station loop
        rules.extend([Instruction.HARVEST.value, Instruction.GIVE.value])
        orders[piece.id] = InstructionList(rules, cursor=0, patrol=True)
    
    return orders
```

The idea: once the march instructions are exhausted (everyone arrives at 236), call `setup_camp` and assign the new orders. You could either:

1. **Append** camp orders to the end of the march instructions, or
2. Use a **callback** that detects arrival and swaps in camp orders

Option 2 is cleaner — a callback like:

```python
def camp_on_arrival(target_hex, grid, elevations, food_tiers, countries=None):
    """Callback that switches to camp mode when squad reaches target."""
    _setup_done = set()
    def cb(report: TurnReport):
        squad = report.squad
        if squad.id in _setup_done: return
        leader = squad.by_rank()[0]
        if leader.location != target_hex: return
        # Leader arrived — set up camp
        orders = setup_camp(squad, target_hex, grid, elevations, food_tiers, countries)
        for p in squad.alive:
            if p.id in orders:
                p.instructions = orders[p.id]
        _setup_done.add(squad.id)
    return cb
```

Then your sim becomes:

```python
sim = FoodSimulator(
    ...,
    callbacks={myStuff.queenGuard.id: compose_callbacks(
        food_triage,
        camp_on_arrival(236, myStuff.grid, myStuff.terr.elevations, 
                       food_tiers, myStuff.terr.fields.get('country'))
    )},
)
```

Want to try implementing `setup_camp` yourself, or shall I write the full version ready to paste?

Should we take advantage of having pyomo. This feels like a wherehouse location problem.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great instinct — it *is* a capacitated facility location problem! The analogy maps perfectly:

| Warehouse/Facility Problem | Camp Setup |
|---|---|
| Warehouse sites | Candidate food tiles (ring 1..n around target) |
| Customers | Pieces needing food (queen, pawns, etc.) |
| Warehouse capacity | `harvest_strength × tier` per feeder on that tile |
| Customer demand | `piece.diet` |
| Coverage/shipping | Line of sight from tile → piece (facing cone) |
| Opening cost | Combat value lost (we want fighters free) |

Here's the Pyomo formulation:

```python
def camp_assign(squad, target_hex, grid, elevations, food_tiers,
                countries=None, max_ring=3,
                elevation_mult=0.005) -> dict:
    """Facility-location MIP: station feeders on fertile tiles to cover all demand.
    
    Minimizes total combat value assigned to feeding duty,
    keeping the best fighters free.
    """
    alive = squad.alive
    if not alive: return {}
    
    # ── Candidate tiles (facilities) ──
    tiles = []  # (hex_idx, tier)
    occupied = {target_hex}
    for ring in range(1, max_ring + 1):
        for hp in HexPosition.origin().ring(ring):
            idx = grid.hexposition_to_index(hp, target_hex)
            if idx < 0 or idx in grid.invalidRegion: continue
            if elevations[idx] <= 0: continue
            if countries is not None and countries[idx] < 0: continue
            tier = int(food_tiers[idx]) if 0 <= idx < len(food_tiers) else 0
            if tier < 1: continue
            tiles.append((idx, tier))
    
    if not tiles: return {}
    
    N = len(alive)      # pieces (potential feeders AND consumers)
    T = len(tiles)       # candidate tiles
    
    # ── Precompute: which (tile, facing) can see which piece locations ──
    # For each tile, compute sight cone facing target_hex
    visible = np.zeros((T, N), dtype=bool)
    for t, (tidx, _) in enumerate(tiles):
        facing = _facing_toward(grid, tidx, target_hex)
        facing_dir = HexPosition.directions()[facing]
        elev = max(0, elevations[tidx])
        eff_sight = 3 + elev * elevation_mult
        fov = set()
        for hp in HexPosition.origin().field_of_view(facing_dir, int(eff_sight)):
            fov_idx = grid.hexposition_to_index(hp, tidx)
            if fov_idx >= 0: fov.add(fov_idx)
        # Target hex and ring-1 around it are where consumers will be
        for c, piece in enumerate(alive):
            # Consumer location = target_hex (they're camped)
            if target_hex in fov:
                visible[t, c] = True
    
    # ── Production capacity per (feeder, tile) ──
    # feed_turns = move_strength - 1 harvest actions per turn
    production = np.zeros((N, T))
    for f, piece in enumerate(alive):
        for t, (tidx, tier) in enumerate(tiles):
            production[f, t] = piece.harvest_strength * tier
    
    diet = np.array([p.diet for p in alive])
    combat = np.array([getattr(p, 'attack_strength', 0) for p in alive])
    
    # ── Pyomo model ──
    m = pyo.ConcreteModel()
    m.F = pyo.RangeSet(0, N - 1)    # feeders (any piece)
    m.T = pyo.RangeSet(0, T - 1)    # tiles
    m.C = pyo.RangeSet(0, N - 1)    # consumers (same pieces)
    
    # assign[f,t] = 1 if piece f is stationed on tile t
    m.assign = pyo.Var(m.F, m.T, domain=pyo.Binary)
    
    # flow[f,t,c] = food from feeder f on tile t to consumer c
    m.flow = pyo.Var(m.F, m.T, m.C, domain=pyo.NonNegativeReals)
    
    # ── Objective: minimize combat value of assigned feeders ──
    m.obj = pyo.Objective(
        expr=sum(combat[f] * m.assign[f, t] for f in m.F for t in m.T),
        sense=pyo.minimize)
    
    # ── Each piece assigned to at most 1 tile ──
    m.one_tile = pyo.Constraint(m.F, rule=lambda m, f:
        sum(m.assign[f, t] for t in m.T) <= 1)
    
    # ── Each tile has at most 1 piece ──
    m.one_per_tile = pyo.Constraint(m.T, rule=lambda m, t:
        sum(m.assign[f, t] for f in m.F) <= 1)
    
    # ── Flow only if assigned AND visible ──
    M = max(diet) * 2  # big-M
    def flow_limit(m, f, t, c):
        if not visible[t, c]:
            return m.flow[f, t, c] == 0
        return m.flow[f, t, c] <= M * m.assign[f, t]
    m.flow_cap = pyo.Constraint(m.F, m.T, m.C, rule=flow_limit)
    
    # ── Production capacity per feeder-tile ──
    m.prod_cap = pyo.Constraint(m.F, m.T, rule=lambda m, f, t:
        sum(m.flow[f, t, c] for c in m.C) <= production[f, t] * m.assign[f, t])
    
    # ── Every consumer's diet must be met ──
    m.demand = pyo.Constraint(m.C, rule=lambda m, c:
        sum(m.flow[f, t, c] for f in m.F for t in m.T) >= diet[c])
    
    # ── Solve ──
    solver = pyo.SolverFactory('appsi_highs')
    result = solver.solve(m)
    
    if not _solver_ok(result):
        return {}
    
    # ── Extract assignments ──
    orders = {}
    for f in m.F:
        piece = alive[f]
        assigned_tile = None
        for t in m.T:
            if pyo.value(m.assign[f, t]) > 0.5:
                assigned_tile = t
                break
        
        if assigned_tile is None:
            # Free piece — hold position at target
            orders[piece.id] = InstructionList(
                [Instruction.PAUSE.value], cursor=0, patrol=True)
        else:
            tidx, _ = tiles[assigned_tile]
            facing = _facing_toward(grid, tidx, target_hex)
            
            tp = piece.pathfind(tidx, grid, elevations, countries)
            rules = tp.to_rules(grid, piece.facing) if tp else []
            
            # Rotate to face target
            cur = piece.facing if not rules else _facing_toward(
                grid, tp.hexes[-2] if len(tp.hexes) >= 2 else piece.location, tidx)
            diff = (facing - cur) % 6
            if diff <= 3: rules.extend([Instruction.ROT_R.value] * diff)
            else: rules.extend([Instruction.ROT_L.value] * (6 - diff))
            
            rules.extend([Instruction.HARVEST.value, Instruction.GIVE.value])
            orders[piece.id] = InstructionList(rules, cursor=0, patrol=True)
    
    return orders
```

The key advantages over the greedy `supply_line` approach:

- **Globally optimal** feeder selection — keeps the highest attack-strength pieces free
- **Respects sight cones** — only assigns food flows along visible paths
- **Capacity-aware** — won't over-assign feeders to weak tiles or under-assign strong ones
- **One solve** — no iterative greedy + Hungarian two-phase

The `camp_on_arrival` callback would just call `camp_assign` instead of `setup_camp`. Want to try it?

So I was thinking that it would make sense for when we leap_frog we would use camp_assign until we are at capacity to travel

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Absolutely — that's a natural fit. At each waypoint you solve the facility-location MIP to optimally station feeders, then analytically compute how many camp turns until the squad is fueled for the next leg. No need to run a full `FoodSimulator` at each stop — the MIP solution gives you exact production rates.

Here's the shape. First, a helper that extracts the camp's per-turn economics from the MIP result:

```python
def camp_economics(squad, camp_result, tiles, food_tiers):
    """Compute per-turn surplus from a camp_assign solution.
    
    Returns (total_production, total_diet, net_surplus, deficit_to_fill)
    """
    alive = squad.alive
    total_diet = sum(p.diet for p in alive)
    
    # Sum production from assigned feeders
    total_prod = 0
    for p in alive:
        tile_idx = camp_result.get('tile_for', {}).get(p.id)
        if tile_idx is not None:
            tier = int(food_tiers[tile_idx])
            total_prod += p.harvest_strength * tier
    
    deficit = sum(max(0, p.food_capacity - p.food) for p in alive)
    net = total_prod - total_diet
    
    return dict(production=total_prod, diet=total_diet,
                net_surplus=net, deficit=deficit,
                turns_to_fill=math.ceil(deficit / net) if net > 0 else float('inf'))
```

Then the updated `leapfrog_plan` — the key change is Phase 3–4, where `squad_recover` is replaced by `camp_assign` + analytical fill:

```python
@patch
def leapfrog_plan(self: Squad, leader_path: TroopPath,
                  grid: HexGrid, elevations: np.ndarray,
                  formation: FormationType = None,
                  countries: np.ndarray = None,
                  food_tiers: np.ndarray = None,
                  min_food_tier: int = 2,
                  max_camp_turns: int = 30,
                  capacity_pct: float = 0.95,
                  max_legs: int = 20,
                  leader: Piece = None,
                  elevation_mult: float = 0.005) -> list[dict]:
    """Leapfrog march using camp_assign MIP at each waypoint.
    
    At each stop:
      1. camp_assign solves facility-location for optimal feeder placement
      2. Analytically compute recovery turns until capacity_pct reached
      3. Apply food gains, redistribute, depart
    """
    if leader is None:
        leader = self.by_rank()[0]
    
    legs = []
    pos = 0
    
    for _ in range(max_legs):
        if pos >= len(leader_path) - 1:
            break
        
        # ── Phase 0: Bishop tops up before departure ──
        pre_march_distribute(self)
        
        # ── Phase 1: Farthest reachable waypoint ──
        sub_hexes = leader_path.hexes[pos:]
        sub_cost = sum(_move_cost(elevations, sub_hexes[k], sub_hexes[k+1])
                       for k in range(len(sub_hexes) - 1))
        sub_path = TroopPath(sub_hexes, sub_cost)
        
        leg = self.march_range(sub_path, grid, elevations,
                               formation=formation, countries=countries,
                               food_tiers=food_tiers,
                               min_food_tier=min_food_tier, leader=leader)
        
        if not leg['feasible'] or leg['waypoint'] == 0:
            break
        
        # ── Phase 2: Deduct travel food ──
        wp_hex = leg['hex_idx']
        for p in self.alive:
            tp = leg['paths'].get(p.id)
            turns = math.ceil(tp.cost / max(p.move_strength, 1)) if tp else 0
            p.food = max(0, p.food - p.diet * turns)
            p.location = wp_hex
        
        # ── Phase 3: Camp — solve facility-location MIP ──
        camp = camp_assign(self, wp_hex, grid, elevations, food_tiers,
                          countries=countries, elevation_mult=elevation_mult)
        
        # ── Phase 4: Analytical recovery ──
        econ = camp_economics(self, camp, food_tiers=food_tiers)
        net = econ['net_surplus']
        
        if net <= 0:
            # Can't sustain here — record and hope next leg is better
            leg['camp'] = camp
            leg['econ'] = econ
            leg['camp_turns'] = 0
            leg['leg_number'] = len(legs)
            legs.append(leg)
            pos += leg['waypoint']
            continue
        
        # How many turns until everyone is at capacity_pct?
        target_food = {p.id: p.food_capacity * capacity_pct for p in self.alive}
        deficit = sum(max(0, target_food[p.id] - p.food) for p in self.alive)
        camp_turns = math.ceil(deficit / net)
        camp_turns = min(camp_turns, max_camp_turns)
        
        # Apply food gains proportionally (each piece gets diet + share of surplus)
        # Feeders harvest, eat, share. Non-feeders just eat what's given.
        feeder_ids = camp.get('feeder_ids', set())
        for t in range(camp_turns):
            # Feeders harvest
            for p in self.alive:
                if p.id in feeder_ids:
                    tile = camp['tile_for'].get(p.id)
                    if tile is not None:
                        tier = int(food_tiers[tile])
                        gained = p.harvest_strength * tier
                        p.food = min(p.food + gained, p.food_capacity)
            # Everyone eats
            for p in self.alive:
                p.food = max(0, p.food - p.diet)
            # Feeders share surplus to neediest
            pre_march_distribute(self)
        
        # ── Phase 5: Final redistribute before next leg ──
        pre_march_distribute(self)
        
        leg['camp'] = camp
        leg['econ'] = econ
        leg['camp_turns'] = camp_turns
        leg['leg_number'] = len(legs)
        legs.append(leg)
        
        pos += leg['waypoint']
    
    return legs
```

And you'd need `camp_assign` to return the extra metadata — add this to the extract section at the bottom:

```python
    # ── Extract assignments (add to existing camp_assign) ──
    feeder_ids = set()
    tile_for = {}  # piece_id → hex_idx
    
    for f in m.F:
        piece = alive[f]
        for t in m.T:
            if pyo.value(m.assign[f, t]) > 0.5:
                feeder_ids.add(piece.id)
                tile_for[piece.id] = tiles[t][0]
                break
    
    orders['feeder_ids'] = feeder_ids
    orders['tile_for'] = tile_for
    orders['total_production'] = sum(
        alive[f].harvest_strength * tiles[t][1]
        for f in m.F for t in m.T
        if pyo.value(m.assign[f, t]) > 0.5)
    
    return orders
```

The flow is now: **march → MIP camp → analytical fill → redistribute → march** at each leg. The MIP ensures the best fighters stay free at every stop, and the analytical fill avoids the overhead of simulating every camp turn. The `capacity_pct` parameter lets you control how full the squad should be before departing — 0.95 means "nearly topped off," which maximizes range per leg.

In [ ]:
TerrainDisplay(
    TerrainOverlay(),
    SettlementOverlay(),
    FoodOverlay(),
    SquadFoodTrailOverlay(sim, myStuff.queenGuard),
    PieceOverlay([myStuff.queenGuard]),
    RiverOverlay(max_width=5),
    terrain=myStuff.terr,
    board=myStuff.board,
    basins=myStuff.basin,
    debug=not showDemo
)


In [ ]:
# Reset to fresh state
demoTerr()
myStuff.apply_food_profile(soil_scale=7)
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()
food_tiers = fy.tiers

# Generate march orders
orders = myStuff.queenGuard.march_to(
    236, FormationType.BOX,
    grid=myStuff.grid, elevations=myStuff.terr.elevations,
    leader=myStuff.queen, countries=myStuff.terr.fields.get('country'),
    food_tiers=food_tiers, sync=True)

# Apply orders BUT append harvest/give loop so they sustain after arrival
for p in myStuff.queenGuard.alive:
    if p.id in orders:
        il = orders[p.id]
        # Append sustenance cycle after the march
        il.rules.extend([Instruction.HARVEST.value, Instruction.GIVE.value])
        il.patrol = True  # loop the harvest/give forever after march completes
        p.instructions = il
        print(f"{p.name:10s} ({p.piece_type.name:6s}): {len(il.rules)} instr (patrol={il.patrol})")


Can you make the above use leap frog and camp_assign until the bishops are full?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Here are two blocks — first `camp_assign`, then the leapfrog loop that camps until bishops are full.

```python
def camp_assign(squad, target_hex, grid, elevations, food_tiers,
                countries=None, max_ring=3):
    """Facility-location MIP: pick who feeds, on which tile, minimizing combat loss."""
    alive = squad.alive
    if not alive: return {'feasible': False}
    
    # ── Candidate tiles around camp center ──
    tiles = []
    for ring in range(1, max_ring + 1):
        for hp in HexPosition.origin().ring(ring):
            idx = grid.hexposition_to_index(hp, target_hex)
            if idx < 0 or idx in grid.invalidRegion: continue
            if elevations[idx] <= 0: continue
            if countries is not None and countries[idx] < 0: continue
            tier = int(food_tiers[idx]) if 0 <= idx < len(food_tiers) else 0
            if tier < 1: continue
            tiles.append((idx, tier))
    
    if not tiles: return {'feasible': False}
    
    N, T = len(alive), len(tiles)
    prod = np.zeros((N, T))
    for f, p in enumerate(alive):
        for t, (_, tier) in enumerate(tiles):
            prod[f, t] = p.harvest_strength * tier
    
    total_diet = sum(p.diet for p in alive)
    combat = np.array([getattr(p, 'attack_strength', 0) for p in alive])
    
    m = pyo.ConcreteModel()
    m.F = pyo.RangeSet(0, N - 1)
    m.T = pyo.RangeSet(0, T - 1)
    m.x = pyo.Var(m.F, m.T, domain=pyo.Binary)
    
    m.obj = pyo.Objective(
        expr=sum(combat[f] * m.x[f, t] for f in m.F for t in m.T),
        sense=pyo.minimize)
    
    m.one_tile = pyo.Constraint(m.F, rule=lambda m, f:
        sum(m.x[f, t] for t in m.T) <= 1)
    m.one_per = pyo.Constraint(m.T, rule=lambda m, t:
        sum(m.x[f, t] for f in m.F) <= 1)
    m.demand = pyo.Constraint(
        expr=sum(prod[f, t] * m.x[f, t] for f in m.F for t in m.T) >= total_diet)
    
    solver = pyo.SolverFactory('appsi_highs')
    result = solver.solve(m)
    if not _solver_ok(result):
        return {'feasible': False}
    
    feeder_ids, tile_for = set(), {}
    total_prod = 0
    for f in m.F:
        for t in m.T:
            if pyo.value(m.x[f, t]) > 0.5:
                p = alive[f]
                feeder_ids.add(p.id)
                tile_for[p.id] = tiles[t][0]
                total_prod += prod[f, t]
    
    return dict(feasible=True, feeder_ids=feeder_ids, tile_for=tile_for,
                total_production=total_prod, total_diet=total_diet,
                net_surplus=total_prod - total_diet, tiles=tiles)
```

Then the leapfrog with camp-until-bishops-full:

```python
# Reset fresh
demoTerr()
myStuff.apply_food_profile(soil_scale=7)
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()
food_tiers = fy.tiers

squad = myStuff.queenGuard
queen = myStuff.queen
grid = myStuff.grid
elevations = myStuff.terr.elevations
countries = myStuff.terr.fields.get('country')

leader_path = queen.pathfind(236, grid, elevations, countries)
print(f"Leader path: {len(leader_path)} hexes, cost {leader_path.cost:.1f}\n")

pos = 0
legs = []
total_camp = 0

for leg_num in range(20):
    if pos >= len(leader_path) - 1:
        print(f"🏁 Arrived at hex 236!")
        break
    
    pre_march_distribute(squad)
    
    sub_hexes = leader_path.hexes[pos:]
    sub_cost = sum(_move_cost(elevations, sub_hexes[k], sub_hexes[k+1])
                   for k in range(len(sub_hexes) - 1))
    sub_path = TroopPath(sub_hexes, sub_cost)
    
    leg = squad.march_range(sub_path, grid, elevations,
                            formation=FormationType.BOX, countries=countries,
                            food_tiers=food_tiers, min_food_tier=2, leader=queen)
    
    if not leg['feasible'] or leg['waypoint'] == 0:
        print(f"❌ Stuck at pos {pos}, hex {leader_path[pos]}")
        break
    
    wp_hex = leg['hex_idx']
    
    # ── Deduct travel food, move everyone to waypoint ──
    for p in squad.alive:
        tp = leg['paths'].get(p.id)
        turns = math.ceil(tp.cost / max(p.move_strength, 1)) if tp else 0
        p.food = max(0, p.food - p.diet * turns)
        p.location = wp_hex
    
    # ── Solve camp MIP ──
    camp = camp_assign(squad, wp_hex, grid, elevations, food_tiers, countries)
    
    if not camp.get('feasible'):
        print(f"  ⚠ Leg {leg_num}: can't sustain at hex {wp_hex}, pressing on")
        pos += leg['waypoint']
        legs.append({**leg, 'camp_turns': 0, 'leg': leg_num})
        continue
    
    # ── Camp until bishops are full ──
    bishops = [p for p in squad.alive if p.piece_type == PieceType.BISHOP]
    camp_turns = 0
    max_camp = 60
    
    while camp_turns < max_camp:
        if bishops and all(b.food >= b.food_capacity * 0.95 for b in bishops):
            break
        
        # Feeders harvest
        for p in squad.alive:
            tile = camp['tile_for'].get(p.id)
            if tile is not None:
                tier = int(food_tiers[tile])
                p.food = min(p.food + p.harvest_strength * tier, p.food_capacity)
        
        # Everyone eats
        for p in squad.alive:
            p.food = max(0, p.food - p.diet)
        
        # Surplus → neediest (bishops fill last since they have huge capacity)
        for donor in sorted(squad.alive, key=lambda d: -d.food):
            reserve = donor.diet * FOOD_RESERVE_TURNS
            surplus = max(0, donor.food - reserve)
            if surplus <= 0: continue
            for recv in sorted(squad.alive, key=lambda r: r.food / max(r.food_capacity, 1)):
                if recv.id == donor.id: continue
                space = recv.food_capacity - recv.food
                give = min(surplus, space)
                if give <= 0: continue
                recv.food += give; donor.food -= give; surplus -= give
                if surplus <= 0: break
        
        camp_turns += 1
    
    total_camp += camp_turns
    pos += leg['waypoint']
    
    bishop_str = ', '.join(f'{b.name}={b.food:.0f}/{b.food_capacity}' for b in bishops)
    print(f"Leg {leg_num}: march {leg['waypoint']} hexes → hex {wp_hex}, "
          f"camp {camp_turns}t (net {camp['net_surplus']:.1f}/t), "
          f"bishops: [{bishop_str}]")
    
    legs.append({**leg, 'camp_turns': camp_turns, 'camp': camp, 'leg': leg_num})

# ── Summary ──
print(f"\n{'='*60}")
print(f"Total: {len(legs)} legs, {total_camp} camp turns")
for p in squad.alive:
    turns_left = p.food / p.diet if p.diet > 0 else float('inf')
    print(f"  {p.name:10s} ({p.piece_type.name:6s}): "
          f"food={p.food:.1f}/{p.food_capacity} ({turns_left:.1f} turns)")
```

The key flow at each waypoint:

1. **MIP picks feeders** — minimizes attack strength lost to feeding, so the queen and best fighters stay free
2. **Feeders harvest** on their assigned tiles each turn
3. **Everyone eats** their diet
4. **Surplus redistributes** from richest → neediest (bishops have 100 capacity, so they soak up the excess like tanks)
5. **Loop until bishops ≥ 95%** — that's the "full tanks, ready to march" signal

The bishops act as fuel tankers — they eat relatively little but carry enormous reserves, which get redistributed via `pre_march_distribute` at the start of each leg.

Do we have two camp_assign? what is the difference between them?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good catch — I proposed two different versions during our conversation. Here's how they differ:

**Version 1** (the earlier, complex one):
- Has **flow variables** `m.flow[f, t, c]` — tracks exactly how much food goes from feeder `f` on tile `t` to consumer `c`
- Models **sight-cone visibility** — a feeder can only feed consumers it can see
- **Per-consumer** demand constraints — every individual piece's diet must be met
- More variables: `O(N × T)` binary + `O(N × T × N)` continuous

**Version 2** (the simpler one I gave with the leapfrog code):
- Just **assignment** variables `m.x[f, t]` — who goes where
- **Aggregate** demand constraint — total production ≥ total diet
- No visibility modeling, no per-consumer flows
- Much fewer variables: `O(N × T)` binary only

For the leapfrog use case, **version 2 is the right one**. At camp everyone is clustered within 1–3 rings of the target hex, so sight cones don't matter — feeders can hand food to adjacent pieces directly. The aggregate demand constraint is sufficient because `pre_march_distribute` handles the actual sharing each turn.

Version 1 would matter for a **spread-out supply line** scenario (like `supply_line` does), where feeders are distant from consumers and can only reach them through facing cones.

I'd say: keep version 2 as `camp_assign` for the leapfrog planner, and if you later need the visibility-aware version for supply lines, give it a different name like `supply_assign`. There's also the existing `feed_cover` in food.py which is even simpler — it just picks *who* should feed (no tile assignment), which is useful for quick "can we sustain here?" checks.

In [ ]:
# Reset fresh
demoTerr()
myStuff.apply_food_profile(soil_scale=7)
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()
food_tiers = fy.tiers

squad = myStuff.queenGuard
queen = myStuff.queen
grid = myStuff.grid
elevations = myStuff.terr.elevations
countries = myStuff.terr.fields.get('country')

leader_path = queen.pathfind(236, grid, elevations, countries)
print(f"Leader path: {len(leader_path)} hexes, cost {leader_path.cost:.1f}\n")

pos = 0
legs = []
total_camp = 0

for leg_num in range(20):
    if pos >= len(leader_path) - 1:
        print(f"🏁 Arrived at hex 236!")
        break
    
    pre_march_distribute(squad)
    
    sub_hexes = leader_path.hexes[pos:]
    sub_cost = sum(_move_cost(elevations, sub_hexes[k], sub_hexes[k+1])
                   for k in range(len(sub_hexes) - 1))
    sub_path = TroopPath(sub_hexes, sub_cost)
    
    leg = squad.march_range(sub_path, grid, elevations,
                            formation=FormationType.BOX, countries=countries,
                            food_tiers=food_tiers, min_food_tier=2, leader=queen)
    
    if not leg['feasible'] or leg['waypoint'] == 0:
        print(f"❌ Stuck at pos {pos}, hex {leader_path[pos]}")
        break
    
    wp_hex = leg['hex_idx']
    
    # ── Deduct travel food, move everyone to waypoint ──
    for p in squad.alive:
        tp = leg['paths'].get(p.id)
        turns = math.ceil(tp.cost / max(p.move_strength, 1)) if tp else 0
        p.food = max(0, p.food - p.diet * turns)
        p.location = wp_hex
    
    # ── Solve camp MIP ──
    camp = squad.supply_assign( wp_hex, grid, elevations, food_tiers, countries)
    
    if not camp.get('feasible'):
        print(f"  ⚠ Leg {leg_num}: can't sustain at hex {wp_hex}, pressing on")
        pos += leg['waypoint']
        legs.append({**leg, 'camp_turns': 0, 'leg': leg_num})
        continue
    
    # ── Camp until bishops are full ──
    bishops = [p for p in squad.alive if p.piece_type == PieceType.BISHOP]
    camp_turns = 0
    max_camp = 60
    
    while camp_turns < max_camp:
        if bishops and all(b.food >= b.food_capacity * 0.95 for b in bishops):
            break
        
        # Feeders harvest
        for p in squad.alive:
            tile = camp['tile_for'].get(p.id)
            if tile is not None:
                tier = int(food_tiers[tile])
                p.food = min(p.food + p.harvest_strength * tier, p.food_capacity)
        
        # Everyone eats
        for p in squad.alive:
            p.food = max(0, p.food - p.diet)
        
        # Surplus → neediest (bishops fill last since they have huge capacity)
        for donor in sorted(squad.alive, key=lambda d: -d.food):
            reserve = donor.diet * FOOD_RESERVE_TURNS
            surplus = max(0, donor.food - reserve)
            if surplus <= 0: continue
            for recv in sorted(squad.alive, key=lambda r: r.food / max(r.food_capacity, 1)):
                if recv.id == donor.id: continue
                space = recv.food_capacity - recv.food
                give = min(surplus, space)
                if give <= 0: continue
                recv.food += give; donor.food -= give; surplus -= give
                if surplus <= 0: break
        
        camp_turns += 1
    
    total_camp += camp_turns
    pos += leg['waypoint']
    
    bishop_str = ', '.join(f'{b.name}={b.food:.0f}/{b.food_capacity}' for b in bishops)
    print(f"Leg {leg_num}: march {leg['waypoint']} hexes → hex {wp_hex}, "
          f"camp {camp_turns}t (net {camp['net_surplus']:.1f}/t), "
          f"bishops: [{bishop_str}]")
    
    legs.append({**leg, 'camp_turns': camp_turns, 'camp': camp, 'leg': leg_num})

# ── Summary ──
print(f"\n{'='*60}")
print(f"Total: {len(legs)} legs, {total_camp} camp turns")
for p in squad.alive:
    turns_left = p.food / p.diet if p.diet > 0 else float('inf')
    print(f"  {p.name:10s} ({p.piece_type.name:6s}): "
          f"food={p.food:.1f}/{p.food_capacity} ({turns_left:.1f} turns)")


In [ ]:


# Run the simulator with food_triage so the squad stops to feed
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()
food_tiers = fy.tiers

sim = FoodSimulator(
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=food_tiers,
    pieces=myStuff.queenGuard.alive,
    countries=myStuff.terr.fields.get('country'),
    squads=[myStuff.queenGuard],
    callbacks={myStuff.queenGuard.id: food_triage},
)

sim.run(num_turns=40)
if showDemo:
    show(sim)
else:
    sim.summary()



What happened to the queen. I am trying to get her to able to get to an oasis.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The queen **starved around turn 8**. Here's what went wrong:

1. **Leapfrog said "0 camp turns"** — because bishops were already at 95/100, the camp condition `all(b.food >= b.food_capacity * 0.95 for b in bishops)` was immediately true. But the queen arrived with only **9.0/30 food (1.3 turns!)**

2. **The post-march patrol (`HARVEST→GIVE` loop) doesn't help her** — queen has `harvest_strength=0.5`, so even on a tier-5 tile she only gets 2.5/turn against a diet of 7.0. She needs pawns feeding *her*, not the other way around.

3. **`food_triage` during sim couldn't save her** — by the time it kicks in she's already nearly dead.

Two fixes needed:

**Fix 1**: The camp exit condition should check the **queen** (or all pieces), not just bishops:

```python
# Replace this:
if bishops and all(b.food >= b.food_capacity * 0.95 for b in bishops):
    break

# With this:
min_food_turns = min(p.food / p.diet for p in squad.alive if p.diet > 0)
if min_food_turns >= 4 and bishops and all(b.food >= b.food_capacity * 0.95 for b in bishops):
    break
```

This ensures **nobody** leaves camp with less than 4 turns of food — especially the queen.

**Fix 2**: The post-arrival instructions need to use `supply_assign` results to make pawns feed the queen, not just blindly loop `HARVEST→GIVE` for everyone:

```python
# After the leapfrog, use the camp assignment for actual instructions
camp = squad.supply_assign(236, grid, elevations, food_tiers, countries)

for p in squad.alive:
    tile = camp.get('tile_for', {}).get(p.id)
    if tile is not None:
        # This piece is a feeder — station on tile, harvest/give loop
        tp = p.pathfind(tile, grid, elevations, countries)
        rules = tp.to_rules(grid, p.facing) if tp else []
        facing = _facing_toward(grid, tile, 236)
        # Add rotation to face queen
        # ... (rotation logic)
        rules.extend([Instruction.HARVEST.value, Instruction.GIVE.value])
        p.instructions = InstructionList(rules, cursor=0, patrol=True)
    else:
        # Not a feeder (queen, fighters) — just pause
        p.instructions = InstructionList(
            [Instruction.PAUSE.value], cursor=0, patrol=True)
```

The root cause is that the **bishops are the fuel tanks but the queen is the fuel guzzler** — camping until bishops are full is necessary but not sufficient. The queen must *also* be fed up before departure, and at the destination pawns must be actively feeding her.

Can you write the proper demo that would get the queen there. 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Just a heads-up: some earlier messages were trimmed from context due to size, so I'm reconstructing from what's visible.

Here's the full demo — leapfrog with proper camp-until-queen-is-safe, then `supply_assign` for sustainable feeding at destination:

```python
# ── Reset fresh ──
demoTerr()
myStuff.apply_food_profile(soil_scale=7)
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()
food_tiers = fy.tiers

squad = myStuff.queenGuard
queen = myStuff.queen
grid = myStuff.grid
elevations = myStuff.terr.elevations
countries = myStuff.terr.fields.get('country')

leader_path = queen.pathfind(236, grid, elevations, countries)
print(f"Leader path: {len(leader_path)} hexes, cost {leader_path.cost:.1f}\n")

# ── Leapfrog march ──
pos = 0
legs = []
total_camp = 0
MIN_DEPART_TURNS = 4  # nobody leaves camp with fewer turns of food than this

for leg_num in range(20):
    if pos >= len(leader_path) - 1:
        print(f"🏁 Arrived at hex 236!")
        break
    
    pre_march_distribute(squad)
    
    sub_hexes = leader_path.hexes[pos:]
    sub_cost = sum(_move_cost(elevations, sub_hexes[k], sub_hexes[k+1])
                   for k in range(len(sub_hexes) - 1))
    sub_path = TroopPath(sub_hexes, sub_cost)
    
    leg = squad.march_range(sub_path, grid, elevations,
                            formation=FormationType.BOX, countries=countries,
                            food_tiers=food_tiers, min_food_tier=2, leader=queen)
    
    if not leg['feasible'] or leg['waypoint'] == 0:
        print(f"❌ Stuck at pos {pos}, hex {leader_path[pos]}")
        break
    
    wp_hex = leg['hex_idx']
    
    # ── Deduct travel food, move everyone ──
    for p in squad.alive:
        tp = leg['paths'].get(p.id)
        turns = math.ceil(tp.cost / max(p.move_strength, 1)) if tp else 0
        p.food = max(0, p.food - p.diet * turns)
        p.location = wp_hex
    
    # ── Camp MIP ──
    camp = squad.supply_assign(wp_hex, grid, elevations, food_tiers, countries)
    
    if not camp.get('feasible'):
        print(f"  ⚠ Leg {leg_num}: can't sustain at hex {wp_hex}, pressing on")
        pos += leg['waypoint']
        legs.append({**leg, 'camp_turns': 0, 'leg': leg_num})
        continue
    
    # ── Camp until BOTH conditions met: ──
    #    1) Every piece has >= MIN_DEPART_TURNS of food
    #    2) Bishops are >= 95% capacity
    bishops = [p for p in squad.alive if p.piece_type == PieceType.BISHOP]
    camp_turns = 0
    max_camp = 80
    
    while camp_turns < max_camp:
        # Check exit: everyone safe AND bishops full
        min_food_turns = min(
            (p.food / p.diet for p in squad.alive if p.diet > 0), default=999)
        bishops_full = not bishops or all(
            b.food >= b.food_capacity * 0.95 for b in bishops)
        
        if min_food_turns >= MIN_DEPART_TURNS and bishops_full:
            break
        
        # Feeders harvest
        for p in squad.alive:
            tile = camp.get('tile_for', {}).get(p.id)
            if tile is not None:
                tier = int(food_tiers[tile])
                p.food = min(p.food + p.harvest_strength * tier, p.food_capacity)
        
        # Everyone eats
        for p in squad.alive:
            p.food = max(0, p.food - p.diet)
        
        # Surplus → neediest
        for donor in sorted(squad.alive, key=lambda d: -d.food):
            reserve = donor.diet * 2
            surplus = max(0, donor.food - reserve)
            if surplus <= 0: continue
            for recv in sorted(squad.alive, key=lambda r: r.food / max(r.food_capacity, 1)):
                if recv.id == donor.id: continue
                space = recv.food_capacity - recv.food
                give = min(surplus, space)
                if give <= 0: continue
                recv.food += give; donor.food -= give; surplus -= give
                if surplus <= 0: break
        
        camp_turns += 1
    
    total_camp += camp_turns
    pos += leg['waypoint']
    
    bishop_str = ', '.join(f'{b.name}={b.food:.0f}/{b.food_capacity}' for b in bishops)
    queen_turns = queen.food / queen.diet if queen.diet > 0 else float('inf')
    print(f"Leg {leg_num}: march {leg['waypoint']} hexes → hex {wp_hex}, "
          f"camp {camp_turns}t, queen={queen.food:.1f}/{queen.food_capacity} "
          f"({queen_turns:.1f}t), bishops: [{bishop_str}]")
    
    legs.append({**leg, 'camp_turns': camp_turns, 'camp': camp, 'leg': leg_num})

# ── Final redistribute before setting up destination camp ──
pre_march_distribute(squad)

# ── Setup destination: use supply_assign to station feeders ──
camp = squad.supply_assign(236, grid, elevations, food_tiers, countries)

if camp.get('feasible'):
    for p in squad.alive:
        tile = camp.get('tile_for', {}).get(p.id)
        if tile is not None:
            # Feeder: pathfind to tile, face queen, harvest/give loop
            tp = p.pathfind(tile, grid, elevations, countries)
            rules = tp.to_rules(grid, p.facing) if tp else []
            facing = _facing_toward(grid, tile, 236)
            # Rotation
            cur = p.facing if not rules else _facing_toward(
                grid, tp.hexes[-2] if tp and len(tp.hexes) >= 2 else p.location, tile)
            diff = (facing - cur) % 6
            if diff <= 3: rules.extend([Instruction.ROT_R.value] * diff)
            else: rules.extend([Instruction.ROT_L.value] * (6 - diff))
            rules.extend([Instruction.HARVEST.value, Instruction.GIVE.value])
            p.instructions = InstructionList(rules, cursor=0, patrol=True)
            print(f"  🌾 {p.name:10s} → tile {tile} (feeder, {len(rules)} instr)")
        else:
            # Non-feeder (queen): pause
            p.instructions = InstructionList(
                [Instruction.PAUSE.value], cursor=0, patrol=True)
            print(f"  👑 {p.name:10s} → pause (fed by others)")
else:
    # Fallback: everyone harvest/give
    print("  ⚠ No feasible camp — everyone self-sustaining")
    for p in squad.alive:
        p.instructions = InstructionList(
            [Instruction.HARVEST.value, Instruction.GIVE.value],
            cursor=0, patrol=True)

# ── Summary ──
print(f"\n{'='*60}")
print(f"Leapfrog: {len(legs)} legs, {total_camp} total camp turns")
for p in squad.alive:
    t = p.food / p.diet if p.diet > 0 else float('inf')
    print(f"  {p.name:10s} ({p.piece_type.name:6s}): "
          f"food={p.food:.1f}/{p.food_capacity} ({t:.1f} turns)")
```

Then verify sustainability with the sim:

```python
# ── Simulate to confirm sustainability at destination ──
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()
food_tiers = fy.tiers

sim = FoodSimulator(
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=food_tiers,
    pieces=myStuff.queenGuard.alive,
    countries=myStuff.terr.fields.get('country'),
    squads=[myStuff.queenGuard],
    callbacks={myStuff.queenGuard.id: food_triage},
)

sim.run(num_turns=40)
show(sim)
```

The key changes from what you had before:

1. **Camp exit checks everyone** — `min_food_turns >= 4` ensures the queen isn't starving when the squad departs
2. **`supply_assign` at destination** — pawns get stationed on fertile tiles *facing the queen*, running `HARVEST→GIVE` loops. The queen just PAUSEs and receives food.
3. **The MIP picks which pawns feed** — minimizing combat value lost, so if you later add knights/rooks the best fighters stay free

In [ ]:
# ── Reset fresh ──
demoTerr()
myStuff.apply_food_profile(soil_scale=7)
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()
food_tiers = fy.tiers

squad = myStuff.queenGuard
queen = myStuff.queen
grid = myStuff.grid
elevations = myStuff.terr.elevations
countries = myStuff.terr.fields.get('country')

leader_path = queen.pathfind(236, grid, elevations, countries)
print(f"Leader path: {len(leader_path)} hexes, cost {leader_path.cost:.1f}\n")

# ── Leapfrog march ──
pos = 0
legs = []
total_camp = 0
MIN_DEPART_TURNS = 4  # nobody leaves camp with fewer turns of food than this

for leg_num in range(20):
    if pos >= len(leader_path) - 1:
        print(f"🏁 Arrived at hex 236!")
        break
    
    pre_march_distribute(squad)
    
    sub_hexes = leader_path.hexes[pos:]
    sub_cost = sum(_move_cost(elevations, sub_hexes[k], sub_hexes[k+1])
                   for k in range(len(sub_hexes) - 1))
    sub_path = TroopPath(sub_hexes, sub_cost)
    
    leg = squad.march_range(sub_path, grid, elevations,
                            formation=FormationType.BOX, countries=countries,
                            food_tiers=food_tiers, min_food_tier=2, leader=queen)
    
    if not leg['feasible'] or leg['waypoint'] == 0:
        print(f"❌ Stuck at pos {pos}, hex {leader_path[pos]}")
        break
    
    wp_hex = leg['hex_idx']
    
    # ── Deduct travel food, move everyone ──
    for p in squad.alive:
        tp = leg['paths'].get(p.id)
        turns = math.ceil(tp.cost / max(p.move_strength, 1)) if tp else 0
        p.food = max(0, p.food - p.diet * turns)
        p.location = wp_hex
    
    # ── Camp MIP ──
    camp = squad.supply_assign(wp_hex, grid, elevations, food_tiers, countries)
    
    if not camp.get('feasible'):
        print(f"  ⚠ Leg {leg_num}: can't sustain at hex {wp_hex}, pressing on")
        pos += leg['waypoint']
        legs.append({**leg, 'camp_turns': 0, 'leg': leg_num})
        continue
    
    # ── Camp until BOTH conditions met: ──
    #    1) Every piece has >= MIN_DEPART_TURNS of food
    #    2) Bishops are >= 95% capacity
    bishops = [p for p in squad.alive if p.piece_type == PieceType.BISHOP]
    camp_turns = 0
    max_camp = 80
    
    while camp_turns < max_camp:
        # Check exit: everyone safe AND bishops full
        min_food_turns = min(
            (p.food / p.diet for p in squad.alive if p.diet > 0), default=999)
        bishops_full = not bishops or all(
            b.food >= b.food_capacity * 0.95 for b in bishops)
        
        if min_food_turns >= MIN_DEPART_TURNS and bishops_full:
            break
        
        # Feeders harvest
        for p in squad.alive:
            tile = camp.get('tile_for', {}).get(p.id)
            if tile is not None:
                tier = int(food_tiers[tile])
                p.food = min(p.food + p.harvest_strength * tier, p.food_capacity)
        
        # Everyone eats
        for p in squad.alive:
            p.food = max(0, p.food - p.diet)
        
        # Surplus → neediest
        for donor in sorted(squad.alive, key=lambda d: -d.food):
            reserve = donor.diet * 2
            surplus = max(0, donor.food - reserve)
            if surplus <= 0: continue
            for recv in sorted(squad.alive, key=lambda r: r.food / max(r.food_capacity, 1)):
                if recv.id == donor.id: continue
                space = recv.food_capacity - recv.food
                give = min(surplus, space)
                if give <= 0: continue
                recv.food += give; donor.food -= give; surplus -= give
                if surplus <= 0: break
        
        camp_turns += 1
    
    total_camp += camp_turns
    pos += leg['waypoint']
    
    bishop_str = ', '.join(f'{b.name}={b.food:.0f}/{b.food_capacity}' for b in bishops)
    queen_turns = queen.food / queen.diet if queen.diet > 0 else float('inf')
    print(f"Leg {leg_num}: march {leg['waypoint']} hexes → hex {wp_hex}, "
          f"camp {camp_turns}t, queen={queen.food:.1f}/{queen.food_capacity} "
          f"({queen_turns:.1f}t), bishops: [{bishop_str}]")
    
    legs.append({**leg, 'camp_turns': camp_turns, 'camp': camp, 'leg': leg_num})

# ── Final redistribute before setting up destination camp ──
pre_march_distribute(squad)

# ── Setup destination: use supply_assign to station feeders ──
camp = squad.supply_assign(236, grid, elevations, food_tiers, countries)

if camp.get('feasible'):
    for p in squad.alive:
        tile = camp.get('tile_for', {}).get(p.id)
        if tile is not None:
            # Feeder: pathfind to tile, face queen, harvest/give loop
            tp = p.pathfind(tile, grid, elevations, countries)
            rules = tp.to_rules(grid, p.facing) if tp else []
            facing = _facing_toward(grid, tile, 236)
            # Rotation
            cur = p.facing if not rules else _facing_toward(
                grid, tp.hexes[-2] if tp and len(tp.hexes) >= 2 else p.location, tile)
            diff = (facing - cur) % 6
            if diff <= 3: rules.extend([Instruction.ROT_R.value] * diff)
            else: rules.extend([Instruction.ROT_L.value] * (6 - diff))
            rules.extend([Instruction.HARVEST.value, Instruction.GIVE.value])
            p.instructions = InstructionList(rules, cursor=0, patrol=True)
            print(f"  🌾 {p.name:10s} → tile {tile} (feeder, {len(rules)} instr)")
        else:
            # Non-feeder (queen): pause
            p.instructions = InstructionList(
                [Instruction.PAUSE.value], cursor=0, patrol=True)
            print(f"  👑 {p.name:10s} → pause (fed by others)")
else:
    # Fallback: everyone harvest/give
    print("  ⚠ No feasible camp — everyone self-sustaining")
    for p in squad.alive:
        p.instructions = InstructionList(
            [Instruction.HARVEST.value, Instruction.GIVE.value],
            cursor=0, patrol=True)

# ── Summary ──
print(f"\n{'='*60}")
print(f"Leapfrog: {len(legs)} legs, {total_camp} total camp turns")
for p in squad.alive:
    t = p.food / p.diet if p.diet > 0 else float('inf')
    print(f"  {p.name:10s} ({p.piece_type.name:6s}): "
          f"food={p.food:.1f}/{p.food_capacity} ({t:.1f} turns)")


This is hard.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yeah, it really is! 😅 But the output is telling us exactly what's broken. Let me diagnose:

**The squad camped for 80 turns and STILL everyone is nearly dead.** That means the camp isn't actually producing enough food. Look:

- Total diet = 5×2 + 2×2.5 + 7 = **22/turn**
- Only **3 feeders** assigned (MIP minimized combat loss)
- 3 pawns × 2.0 × tier... if tiles are tier 3: **12/turn** ← deficit of 10!

The MIP is trying to be clever (minimize feeders) but 3 pawns can't sustain 8 mouths. You need **all 5 pawns** harvesting. Also, `pre_march_distribute` empties the bishops before marching, so after arrival everyone is near zero — that's 290 total food capacity to refill at a trickle.

The fix is actually much simpler than the MIP. For this squad composition, **every pawn must feed**:

```python
# ── Reset fresh ──
demoTerr()
myStuff.apply_food_profile(soil_scale=7)
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()
food_tiers = fy.tiers

squad = myStuff.queenGuard
queen = myStuff.queen
grid = myStuff.grid
elevations = myStuff.terr.elevations
countries = myStuff.terr.fields.get('country')

# ── March everyone to 236 ──
orders = squad.march_to(
    236, FormationType.BOX,
    grid=grid, elevations=elevations,
    leader=queen, countries=countries,
    food_tiers=food_tiers, sync=True)

# ── Find best food tiles near 236 for ALL pawns ──
from heapq import nlargest

candidate_tiles = []
for ring in range(1, 4):
    for hp in HexPosition.origin().ring(ring):
        idx = grid.hexposition_to_index(hp, 236)
        if idx < 0 or idx in grid.invalidRegion: continue
        if elevations[idx] <= 0: continue
        if countries is not None and countries[idx] < 0: continue
        tier = int(food_tiers[idx]) if 0 <= idx < len(food_tiers) else 0
        if tier >= 1:
            candidate_tiles.append((tier, ring, idx))

candidate_tiles.sort(key=lambda c: (-c[0], c[1]))  # best tier, closest
print(f"Candidate food tiles near 236: {[(t,r,i) for t,r,i in candidate_tiles[:10]]}\n")

# ── Assign every pawn to a food tile, queen/bishops pause ──
claimed = set()
pawns = [p for p in squad.alive if p.piece_type == PieceType.PAWN]
others = [p for p in squad.alive if p.piece_type != PieceType.PAWN]

for p in squad.alive:
    if p.id in orders:
        march_rules = orders[p.id].rules.copy()
    else:
        march_rules = []
    
    if p.piece_type == PieceType.PAWN:
        # Find best unclaimed tile
        tile_idx = None
        for tier, ring, idx in candidate_tiles:
            if idx not in claimed:
                tile_idx = idx
                claimed.add(idx)
                break
        
        if tile_idx is not None:
            # After march: move to food tile, face 236, harvest/give loop
            tp = p.pathfind(tile_idx, grid, elevations, countries)
            if tp:
                station_rules = tp.to_rules(grid, p.facing)
            else:
                station_rules = []
            
            facing = _facing_toward(grid, tile_idx, 236)
            cur = p.facing
            diff = (facing - cur) % 6
            if diff <= 3: station_rules.extend([Instruction.ROT_R.value] * diff)
            else: station_rules.extend([Instruction.ROT_L.value] * (6 - diff))
            
            station_rules.extend([Instruction.HARVEST.value, Instruction.GIVE.value])
            
            # Combine: march first, then station + harvest/give patrol
            all_rules = march_rules + station_rules
            p.instructions = InstructionList(all_rules, cursor=0, patrol=True)
            tier_at = int(food_tiers[tile_idx])
            print(f"  🌾 {p.name:10s} → tile {tile_idx} (tier {tier_at}, "
                  f"net={p.harvest_strength * tier_at - p.diet:+.1f}/t, "
                  f"{len(all_rules)} instr)")
        else:
            march_rules.extend([Instruction.HARVEST.value, Instruction.GIVE.value])
            p.instructions = InstructionList(march_rules, cursor=0, patrol=True)
            print(f"  🌾 {p.name:10s} → no tile, harvest in place")
    else:
        # Queen/Bishop: march then pause (fed by pawns)
        march_rules.append(Instruction.PAUSE.value)
        p.instructions = InstructionList(march_rules, cursor=0, patrol=True)
        print(f"  👑 {p.name:10s} → pause after march (fed by pawns)")

# Production check
total_diet = sum(p.diet for p in squad.alive)
total_prod = sum(p.harvest_strength * int(food_tiers[t[2]])
                 for p, (_, _, tidx) in zip(pawns, candidate_tiles[:len(pawns)])
                 if (t := next((c for c in candidate_tiles if c[2] == tidx), None)))
print(f"\nTotal diet: {total_diet}/turn")
print(f"5 pawns harvesting: surplus goes to queen + bishops")
```

Then sim it:

```python
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()

sim = FoodSimulator(
    grid=grid, elevations=elevations,
    food_tiers=fy.tiers,
    pieces=squad.alive,
    countries=countries,
    squads=[squad],
    callbacks={squad.id: food_triage},
)
sim.run(num_turns=60)
show(sim)
```

The core insight: **for this squad, the MIP is overkill and counterproductive**. The math is simple:

| | Diet | Harvest@tier4 | Net |
|---|---|---|---|
| 5 pawns | 10 | 40 | **+30** |
| 2 bishops | 5 | 0 | -5 |
| 1 queen | 7 | 0 | -7 |
| **Total** | **22** | **40** | **+18** |

With all 5 pawns on tier-4 tiles, you get **+18 surplus/turn** — queen gets fed in 2 turns, bishops fill in ~12 turns. The MIP was trying to keep 2 pawns "free" but there's nothing for them to fight! Every pawn should be a feeder.

The MIP would become useful when you have knights/rooks that you genuinely want to keep combat-ready. For now, brute-force all pawns as feeders.

In [ ]:
# ── Reset fresh ──
demoTerr()
myStuff.apply_food_profile(soil_scale=7)
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()
food_tiers = fy.tiers

squad = myStuff.queenGuard
queen = myStuff.queen
grid = myStuff.grid
elevations = myStuff.terr.elevations
countries = myStuff.terr.fields.get('country')

# ── Phase 1: March (patrol=False — one shot) ──
orders = squad.march_to(
    236, FormationType.BOX,
    grid=grid, elevations=elevations,
    leader=queen, countries=countries,
    food_tiers=food_tiers, sync=True)

for p in squad.alive:
    if p.id in orders:
        il = orders[p.id]
        il.patrol = False
        p.instructions = il

# ── Phase 2: Callback swaps instructions on arrival ──
_camp_set = False

def switch_to_camp(report):
    global _camp_set
    if _camp_set: return
    if queen.location != 236: return
    _camp_set = True
    
    # Find food tiles
    tiles = []
    for ring in range(1, 4):
        for hp in HexPosition.origin().ring(ring):
            idx = grid.hexposition_to_index(hp, 236)
            if idx < 0 or idx in grid.invalidRegion: continue
            if elevations[idx] <= 0: continue
            if countries is not None and countries[idx] < 0: continue
            tier = int(food_tiers[idx]) if 0 <= idx < len(food_tiers) else 0
            if tier >= 1: tiles.append((tier, ring, idx))
    tiles.sort(key=lambda c: (-c[0], c[1]))
    
    claimed = set()
    for p in squad.alive:
        if p.piece_type == PieceType.PAWN:
            # Grab best unclaimed tile
            tile_idx = None
            for tier, ring, idx in tiles:
                if idx not in claimed:
                    tile_idx = idx; claimed.add(idx); break
            
            if tile_idx is not None:
                p.location = tile_idx
                p.facing = _facing_toward(grid, tile_idx, 236)
            
            p.instructions = InstructionList(
                [Instruction.HARVEST.value, Instruction.HARVEST.value, Instruction.GIVE.value],
                cursor=0, patrol=True)
        else:
            p.instructions = InstructionList(
                [Instruction.PAUSE.value], cursor=0, patrol=True)

# ── Run ──
sim = FoodSimulator(
    grid=grid, elevations=elevations,
    food_tiers=food_tiers,
    pieces=squad.alive,
    countries=countries,
    squads=[squad],
    callbacks={squad.id: compose_callbacks(food_triage, switch_to_camp)},
)
sim.run(num_turns=60)
if showDemo:
    show(sim)
else:
    sim.summary()


In [ ]:


TerrainDisplay(
    TerrainOverlay(),
    SettlementOverlay(),
    FoodOverlay(),
    SquadFoodTrailOverlay(sim, myStuff.queenGuard),
    PieceOverlay([myStuff.queenGuard]),
    RiverOverlay(max_width=5),
    terrain=myStuff.terr,
    board=myStuff.board,
    basins=myStuff.basin,
    debug=not showDemo
)



In [ ]:
??InstructionList

So instructionList isn't that complicated of a item. It's just a list of instructions with some extra metadata. so if we want to be in camp mode we should just remove what rules we don't want and then add the ones we do.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Exactly right. When they arrive, just do:

```python
p.instructions = InstructionList(
    [Instruction.HARVEST.value, Instruction.GIVE.value],
    cursor=0, patrol=True)
```

No fancy callback machinery needed. Here's the clean version:

```python
# ── Reset fresh ──
demoTerr()
myStuff.apply_food_profile(soil_scale=7)
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()
food_tiers = fy.tiers

squad = myStuff.queenGuard
queen = myStuff.queen
grid = myStuff.grid
elevations = myStuff.terr.elevations
countries = myStuff.terr.fields.get('country')

# ── Phase 1: March (patrol=False — one shot) ──
orders = squad.march_to(
    236, FormationType.BOX,
    grid=grid, elevations=elevations,
    leader=queen, countries=countries,
    food_tiers=food_tiers, sync=True)

for p in squad.alive:
    if p.id in orders:
        il = orders[p.id]
        il.patrol = False
        p.instructions = il

# ── Phase 2: Callback swaps instructions on arrival ──
_camp_set = False

def switch_to_camp(report):
    global _camp_set
    if _camp_set: return
    if queen.location != 236: return
    _camp_set = True
    
    # Find food tiles
    tiles = []
    for ring in range(1, 4):
        for hp in HexPosition.origin().ring(ring):
            idx = grid.hexposition_to_index(hp, 236)
            if idx < 0 or idx in grid.invalidRegion: continue
            if elevations[idx] <= 0: continue
            if countries is not None and countries[idx] < 0: continue
            tier = int(food_tiers[idx]) if 0 <= idx < len(food_tiers) else 0
            if tier >= 1: tiles.append((tier, ring, idx))
    tiles.sort(key=lambda c: (-c[0], c[1]))
    
    claimed = set()
    for p in squad.alive:
        if p.piece_type == PieceType.PAWN:
            # Grab best unclaimed tile
            tile_idx = None
            for tier, ring, idx in tiles:
                if idx not in claimed:
                    tile_idx = idx; claimed.add(idx); break
            
            if tile_idx is not None:
                p.location = tile_idx
                p.facing = _facing_toward(grid, tile_idx, 236)
            
            p.instructions = InstructionList(
                [Instruction.HARVEST.value, Instruction.GIVE.value],
                cursor=0, patrol=True)
        else:
            p.instructions = InstructionList(
                [Instruction.PAUSE.value], cursor=0, patrol=True)

# ── Run ──
sim = FoodSimulator(
    grid=grid, elevations=elevations,
    food_tiers=food_tiers,
    pieces=squad.alive,
    countries=countries,
    squads=[squad],
    callbacks={squad.id: compose_callbacks(food_triage, switch_to_camp)},
)
sim.run(num_turns=60)
show(sim)
```

That's it. March with `patrol=False`, then on arrival just replace `.instructions` with `[HARVEST, GIVE]` (pawns) or `[PAUSE]` (queen/bishops) and `patrol=True`. The old march rules are just gone — don't need them anymore.

How should we design this?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good question. Let's think about it as a **state machine** with clear phases. The squad is always in exactly one state:

**States:**
1. **CAMP** — pawns on food tiles harvesting/giving, queen pauses
2. **PREP** — at camp, but filling bishops before departure (same instructions as CAMP, different exit condition)  
3. **MARCH** — one-shot movement instructions, patrol=False
4. **ARRIVED** — transition: find food tiles, assign pawns, enter CAMP

**Transitions:**
```
CAMP → PREP       (when: ordered to move somewhere)
PREP → MARCH      (when: min_food_turns >= threshold for ALL pieces, bishops ≥ 95%)
MARCH → ARRIVED   (when: leader.location == target)
ARRIVED → CAMP    (immediate — just reassign instructions)
```

The implementation is just a callback that tracks state:

```python
class SquadMission:
    """State machine for moving a squad and sustaining it."""
    
    CAMP, PREP, MARCH, ARRIVED = range(4)
    
    def __init__(self, squad, target, grid, elevations, food_tiers, countries,
                 min_depart_turns=4):
        self.squad = squad
        self.target = target
        self.grid = grid
        self.elevations = elevations
        self.food_tiers = food_tiers
        self.countries = countries
        self.min_depart_turns = min_depart_turns
        self.state = self.PREP  # start by prepping
        self.leader = squad.by_rank()[0]
        
        # Immediately set up camp at current location to fill up
        self._enter_camp(self.leader.location)
    
    def callback(self, report):
        """Called each sim turn. Handles state transitions."""
        if self.state == self.PREP:
            self._check_prep()
        elif self.state == self.MARCH:
            self._check_arrived()
        elif self.state == self.ARRIVED:
            self._enter_camp(self.target)
            self.state = self.CAMP
    
    # ── State transitions ──
    
    def _check_prep(self):
        """Exit PREP when everyone is fed and bishops are full."""
        alive = [p for p in self.squad.alive if p.diet > 0]
        if not alive: return
        
        min_turns = min(p.food / p.diet for p in alive)
        bishops = [p for p in alive if p.piece_type == PieceType.BISHOP]
        bishops_ready = not bishops or all(
            b.food >= b.food_capacity * 0.95 for b in bishops)
        
        if min_turns >= self.min_depart_turns and bishops_ready:
            self._enter_march()
    
    def _enter_march(self):
        """Generate one-shot march orders."""
        self.state = self.MARCH
        orders = self.squad.march_to(
            self.target, FormationType.BOX,
            grid=self.grid, elevations=self.elevations,
            leader=self.leader, countries=self.countries,
            food_tiers=self.food_tiers, sync=True)
        
        for p in self.squad.alive:
            if p.id in orders:
                il = orders[p.id]
                il.patrol = False  # one-shot
                p.instructions = il
    
    def _check_arrived(self):
        """Transition to ARRIVED when leader reaches target."""
        if self.leader.location == self.target:
            self.state = self.ARRIVED
    
    def _enter_camp(self, location):
        """Set up camp: pawns to food tiles, others pause."""
        self.state = self.CAMP
        
        # Find food tiles near location
        tiles = []
        for ring in range(1, 4):
            for hp in HexPosition.origin().ring(ring):
                idx = self.grid.hexposition_to_index(hp, location)
                if idx < 0 or idx in self.grid.invalidRegion: continue
                if self.elevations[idx] <= 0: continue
                if self.countries is not None and self.countries[idx] < 0: continue
                tier = int(self.food_tiers[idx]) if 0 <= idx < len(self.food_tiers) else 0
                if tier >= 1:
                    tiles.append((tier, ring, idx))
        tiles.sort(key=lambda c: (-c[0], c[1]))
        
        claimed = set()
        for p in self.squad.alive:
            if p.piece_type == PieceType.PAWN:
                tile_idx = None
                for tier, ring, idx in tiles:
                    if idx not in claimed:
                        tile_idx = idx; claimed.add(idx); break
                
                if tile_idx is not None:
                    p.location = tile_idx
                    p.facing = _facing_toward(self.grid, tile_idx, location)
                
                p.instructions = InstructionList(
                    [Instruction.HARVEST.value, Instruction.GIVE.value],
                    cursor=0, patrol=True)
            else:
                p.instructions = InstructionList(
                    [Instruction.PAUSE.value], cursor=0, patrol=True)
```

Usage becomes very clean:

```python
demoTerr()
myStuff.apply_food_profile(soil_scale=7)
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()

squad = myStuff.queenGuard
mission = SquadMission(
    squad, target=236,
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers,
    countries=myStuff.terr.fields.get('country'))

sim = FoodSimulator(
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers,
    pieces=squad.alive,
    countries=myStuff.terr.fields.get('country'),
    squads=[squad],
    callbacks={squad.id: compose_callbacks(food_triage, mission.callback)},
)
sim.run(num_turns=120)
show(sim)
```

The beauty of this design:

- **Each state is obvious** — you can read what the squad is doing at any point
- **Transitions are explicit** — PREP waits for everyone to be fed before marching
- **Instructions are simple** — just `[HARVEST, GIVE]` or `[PAUSE]` or one-shot march orders
- **Extensible** — want to add a LEAPFROG state for long journeys? Just add `_check_leg()` and `_enter_next_leg()`
- **No duplication** — `_enter_camp` is reused for both origin and destination

The one thing to think about: for **long journeys** where the squad can't make it in one march, you'd add a `LEAPFROG` state that breaks the path into legs and cycles through `MARCH → CAMP → MARCH → CAMP` until arrival. But get the simple version working first.

## SquadMission

In [ ]:
class SquadMission:
    """State machine for moving a squad and sustaining it."""
    
    CAMP, PREP, MARCH, ARRIVED = range(4)
    
    def __init__(self, squad, target, grid, elevations, food_tiers, countries,
                 min_depart_turns=4):
        self.squad = squad
        self.target = target
        self.grid = grid
        self.elevations = elevations
        self.food_tiers = food_tiers
        self.countries = countries
        self.min_depart_turns = min_depart_turns
        self.state = self.PREP  # start by prepping
        self.leader = squad.by_rank()[0]
        
        # Immediately set up camp at current location to fill up
        self._enter_camp(self.leader.location)
    
    def callback(self, report):
        """Called each sim turn. Handles state transitions."""
        if self.state == self.PREP:
            self._check_prep()
        elif self.state == self.MARCH:
            self._check_arrived()
        elif self.state == self.ARRIVED:
            self._enter_camp(self.target)
            self.state = self.CAMP
    
    # ── State transitions ──
    
    def _check_prep(self):
        """Exit PREP when everyone is fed and bishops are full."""
        alive = [p for p in self.squad.alive if p.diet > 0]
        if not alive: return
        
        min_turns = min(p.food / p.diet for p in alive)
        bishops = [p for p in alive if p.piece_type == PieceType.BISHOP]
        bishops_ready = not bishops or all(
            b.food >= b.food_capacity * 0.95 for b in bishops)
        
        if min_turns >= self.min_depart_turns and bishops_ready:
            self._enter_march()
    
    def _enter_march(self):
        """Generate one-shot march orders."""
        self.state = self.MARCH
        orders = self.squad.march_to(
            self.target, FormationType.BOX,
            grid=self.grid, elevations=self.elevations,
            leader=self.leader, countries=self.countries,
            food_tiers=self.food_tiers, sync=True)
        
        for p in self.squad.alive:
            if p.id in orders:
                il = orders[p.id]
                il.patrol = False  # one-shot
                p.instructions = il
    
    def _check_arrived(self):
        """Transition to ARRIVED when leader reaches target."""
        if self.leader.location == self.target:
            self.state = self.ARRIVED
    
    def _enter_camp(self, location):
        """Set up camp: pawns to food tiles, others pause."""
        self.state = self.CAMP
        
        # Find food tiles near location
        tiles = []
        for ring in range(1, 4):
            for hp in HexPosition.origin().ring(ring):
                idx = self.grid.hexposition_to_index(hp, location)
                if idx < 0 or idx in self.grid.invalidRegion: continue
                if self.elevations[idx] <= 0: continue
                if self.countries is not None and self.countries[idx] < 0: continue
                tier = int(self.food_tiers[idx]) if 0 <= idx < len(self.food_tiers) else 0
                if tier >= 1:
                    tiles.append((tier, ring, idx))
        tiles.sort(key=lambda c: (-c[0], c[1]))
        
        claimed = set()
        for p in self.squad.alive:
            if p.piece_type == PieceType.PAWN:
                tile_idx = None
                for tier, ring, idx in tiles:
                    if idx not in claimed:
                        tile_idx = idx; claimed.add(idx); break
                
                if tile_idx is not None:
                    p.location = tile_idx
                    p.facing = _facing_toward(self.grid, tile_idx, location)
                
                p.instructions = InstructionList(
                    [Instruction.HARVEST.value, Instruction.GIVE.value],
                    cursor=0, patrol=True)
            else:
                p.instructions = InstructionList(
                    [Instruction.PAUSE.value], cursor=0, patrol=True)


In [ ]:
demoTerr()
myStuff.apply_food_profile(soil_scale=7)
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()

squad = myStuff.queenGuard
mission = SquadMission(
    squad, target=236,
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers,
    countries=myStuff.terr.fields.get('country'))

sim = FoodSimulator(
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers,
    pieces=squad.alive,
    countries=myStuff.terr.fields.get('country'),
    squads=[squad],
    callbacks={squad.id: compose_callbacks(food_triage, mission.callback)},
)
sim.run(num_turns=120)
sim.summary()


And all of sudden it starts working?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

😄 Yep! The state machine nailed it. **8/8 alive at turn 120**, average 12.4 turns of food in reserve.

The key difference from all the earlier attempts: **PREP before MARCH**. 

Every previous version marched immediately and arrived with empty tanks. The queen would show up at hex 236 with 1.3 turns of food and die before the pawns could set up. This time:

1. `__init__` → `_enter_camp(current_location)` — pawns start harvesting *before* anyone moves
2. `_check_prep` waits until bishops are 95% full AND everyone has ≥4 turns — that's ~200 food in the bishops alone
3. **Then** they march — 4 hexes is nothing with full tanks
4. On arrival, `_enter_camp(236)` — pawns immediately start feeding again

The whole problem was never about fancy MIPs or leapfrogging — it was about **not leaving home hungry**. The state machine made that obvious because you can see the states: CAMP → PREP → MARCH → ARRIVED → CAMP. No state where anyone is moving without food.

Should we use some of our optimal placement?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Absolutely — `supply_assign` already exists as a method on Squad, and it would make `_enter_camp` both shorter and smarter. Right now we're hardcoding "pawns feed, everyone else pauses" — but the MIP knows the actual math:

```python
def _enter_camp(self, location):
    """Set up camp using optimal feeder assignment."""
    self.state = self.CAMP
    
    camp = self.squad.supply_assign(
        location, self.grid, self.elevations, 
        self.food_tiers, self.countries)
    
    if camp.get('feasible'):
        feeder_ids = camp.get('feeder_ids', set())
        tile_for = camp.get('tile_for', {})
        
        for p in self.squad.alive:
            tile = tile_for.get(p.id)
            if tile is not None:
                p.location = tile
                p.facing = _facing_toward(self.grid, tile, location)
                p.instructions = InstructionList(
                    [Instruction.HARVEST.value, Instruction.GIVE.value],
                    cursor=0, patrol=True)
            else:
                p.instructions = InstructionList(
                    [Instruction.PAUSE.value], cursor=0, patrol=True)
    else:
        # Fallback: everyone harvest/give in place
        for p in self.squad.alive:
            p.instructions = InstructionList(
                [Instruction.HARVEST.value, Instruction.GIVE.value],
                cursor=0, patrol=True)
```

This buys you:

- **Adapts to any squad composition** — add a rook or knight and the MIP figures out who should feed without code changes
- **Minimizes combat value lost** — keeps your best fighters free, which matters once there are threats
- **Handles edge cases** — if there aren't enough good tiles, the MIP's demand constraint will tell you (`feasible=False`) instead of silently starving
- **One source of truth** — tile finding, feeder selection, and capacity math all live in `supply_assign` instead of being duplicated in the state machine

The greedy fallback is still there for when the MIP can't find a solution (bad terrain, no food tiles). The state machine stays clean — it doesn't care *how* feeders are picked, just that camp gets set up.

So it seems to me that we have a scheduling issue (I am looking at you pyomo) we want to schedule movement as much as we can so we can go forward, but we also have to schedule harvest and feeding for the group. is there such a thing as a schedule frog. a leap looker?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Love that framing. What you're describing is a **Resource-Constrained Shortest Path with Replenishment** — a real OR problem class. The military version is literally called **march-and-forage scheduling**.

The reactive state machine works, but it's greedy — it fills up completely before each march, which might waste turns. The optimal schedule might be: "march 2 hexes, camp 3 turns, march 2 more" instead of "camp 40 turns, march 4." A Pyomo model can find that.

Here's the formulation — for each time step, each piece chooses an action:

```python
def plan_mission(squad, target, grid, elevations, food_tiers, countries,
                 max_turns=80):
    """Joint schedule: when to march, when to camp, who feeds.
    
    Minimizes total turns to get leader to target while keeping everyone alive.
    """
    leader = squad.by_rank()[0]
    alive = squad.alive
    N = len(alive)
    
    # ── Leader's path (fixed route, variable pacing) ──
    lpath = leader.pathfind(target, grid, elevations, countries)
    H = len(lpath)  # waypoints along path
    T = max_turns
    
    # Food tiles at each waypoint
    wp_tiles = {}  # wp_index → [(tier, hex_idx), ...]
    for w, whex in enumerate(lpath.hexes):
        tiles = []
        for ring in range(1, 4):
            for hp in HexPosition.origin().ring(ring):
                idx = grid.hexposition_to_index(hp, whex)
                if idx < 0 or idx in grid.invalidRegion: continue
                if elevations[idx] <= 0: continue
                if countries is not None and countries[idx] < 0: continue
                tier = int(food_tiers[idx]) if 0 <= idx < len(food_tiers) else 0
                if tier >= 1: tiles.append((tier, idx))
        tiles.sort(key=lambda c: -c[0])
        wp_tiles[w] = tiles[:N]  # at most one per piece
    
    K = max(len(v) for v in wp_tiles.values()) if wp_tiles else 0
    
    diet = {i: alive[i].diet for i in range(N)}
    harvest = {i: alive[i].harvest_strength for i in range(N)}
    cap = {i: alive[i].food_capacity for i in range(N)}
    food0 = {i: alive[i].food for i in range(N)}
    is_pawn = {i: alive[i].piece_type == PieceType.PAWN for i in range(N)}
    
    m = pyo.ConcreteModel()
    m.T = pyo.RangeSet(0, T - 1)
    m.P = pyo.RangeSet(0, N - 1)    # pieces
    m.W = pyo.RangeSet(0, H - 1)    # waypoints
    
    # ── Decision variables ──
    
    # march[t] = 1 if squad advances one waypoint at time t
    m.march = pyo.Var(m.T, domain=pyo.Binary)
    
    # position[t] = waypoint index the squad is at after turn t
    m.pos = pyo.Var(m.T, domain=pyo.NonNegativeIntegers, bounds=(0, H - 1))
    
    # feeding[p, t] = 1 if piece p is harvesting at time t
    m.feeding = pyo.Var(m.P, m.T, domain=pyo.Binary)
    
    # food[p, t] = food level of piece p after turn t
    m.food = pyo.Var(m.P, m.T, domain=pyo.NonNegativeReals)
    
    # arrival = first turn the leader reaches target
    m.arrival = pyo.Var(domain=pyo.NonNegativeIntegers, bounds=(0, T))
    
    # arrived[t] = 1 if squad has reached target by turn t
    m.arrived = pyo.Var(m.T, domain=pyo.Binary)
    
    # ── Objective: minimize arrival turn ──
    m.obj = pyo.Objective(expr=m.arrival, sense=pyo.minimize)
    
    # ── Position tracking ──
    m.pos_init = pyo.Constraint(expr=m.pos[0] == m.march[0])
    m.pos_update = pyo.Constraint(m.T, rule=lambda m, t:
        m.pos[t] == sum(m.march[tt] for tt in m.T if tt <= t))
    
    # ── Can't march and feed same turn ──
    m.no_march_feed = pyo.Constraint(m.P, m.T, rule=lambda m, p, t:
        m.feeding[p, t] + m.march[t] <= 1)
    
    # ── Only pawns can feed ──
    m.pawn_only = pyo.Constraint(m.P, m.T, rule=lambda m, p, t:
        m.feeding[p, t] <= (1 if is_pawn[p] else 0))
    
    # ── Food dynamics ──
    # Average tier at current waypoint (simplified)
    avg_tier = {}
    for w in range(H):
        tiles = wp_tiles.get(w, [])
        avg_tier[w] = sum(t for t, _ in tiles) / max(len(tiles), 1) if tiles else 0
    
    # Simplified: food[p,t] = food[p,t-1] - diet + feeding[p,t] * harvest * avg_tier
    def food_rule(m, p, t):
        if t == 0:
            # Sum avg_tier over possible positions (linearization trick)
            prod = harvest[p] * sum(
                avg_tier[w] for w in range(H)) / max(H, 1)  # rough average
            return m.food[p, 0] == food0[p] - diet[p] + m.feeding[p, 0] * prod
        prod = harvest[p] * sum(
            avg_tier[w] for w in range(H)) / max(H, 1)
        return m.food[p, t] == m.food[p, t-1] - diet[p] + m.feeding[p, t] * prod
    m.food_dyn = pyo.Constraint(m.P, m.T, rule=food_rule)
    
    # ── Food capacity ──
    m.food_cap = pyo.Constraint(m.P, m.T, rule=lambda m, p, t:
        m.food[p, t] <= cap[p])
    
    # ── Nobody dies ──
    m.alive = pyo.Constraint(m.P, m.T, rule=lambda m, p, t:
        m.food[p, t] >= 0.5)
    
    # ── Must reach target ──
    m.must_arrive = pyo.Constraint(expr=m.pos[T-1] >= H - 1)
    
    # ── Arrival tracking ──
    m.arr_link = pyo.Constraint(m.T, rule=lambda m, t:
        m.arrival <= t + T * (1 - m.arrived[t]))
    m.arr_pos = pyo.Constraint(m.T, rule=lambda m, t:
        m.pos[t] >= (H - 1) * m.arrived[t])
    
    # ── Solve ──
    solver = pyo.SolverFactory('appsi_highs')
    result = solver.solve(m)
    
    if not _solver_ok(result):
        return None
    
    # ── Extract schedule ──
    schedule = []
    for t in m.T:
        turn = {
            'turn': t,
            'march': pyo.value(m.march[t]) > 0.5,
            'pos': int(round(pyo.value(m.pos[t]))),
            'feeders': [alive[p].name for p in m.P if pyo.value(m.feeding[p, t]) > 0.5],
            'food': {alive[p].name: round(pyo.value(m.food[p, t]), 1) for p in m.P},
        }
        schedule.append(turn)
    
    return schedule
```

But honestly — this is a **rough sketch**. The hard part is the bilinear term: `feeding[p,t] × avg_tier_at_position[t]` — the production rate depends on *where* you are, and where you are depends on march decisions. That makes it nonlinear.

There are three ways to handle it:

| Approach | Complexity | Quality |
|---|---|---|
| **Fix the route, optimize the pacing** | MIP (linear) | Good — route from pathfind, just decide march/camp per turn |
| **Discretize into legs** | Small MIP | Great — "how many camp turns at each waypoint?" |
| **Full joint route+schedule** | MINLP | Optimal but hard to solve |

The middle option is the sweet spot — the **"leap looker"**:

```python
def leap_schedule(squad, path, grid, elevations, food_tiers, countries,
                  max_camp_per_stop=40):
    """For each waypoint on the path, decide how many turns to camp.
    
    Minimize: sum of camp turns (= total journey time beyond pure march)
    Subject to: nobody's food drops to 0 at any point
    """
    # ... waypoints from path
    # Decision var: camp_turns[w] = integer, how long to camp at waypoint w
    # Food is tracked analytically through the schedule
    # Linear MIP — very fast
```

This is your "schedule frog" — it sees the whole journey and decides the **optimal camping schedule** in one shot. Want me to flesh out the discretized leg version? It's the cleanest and most solvable.

## Resource-Constrained Shortest Path with Replenishment

— a real OR problem class. The military version is literally called **march-and-forage scheduling**.

The reactive state machine works, but it's greedy — it fills up completely before each march, which might waste turns. The optimal schedule might be: "march 2 hexes, camp 3 turns, march 2 more" instead of "camp 40 turns, march 4." A Pyomo model can find that.

Here's the formulation — for each time step, each piece chooses an action:

```python
def plan_mission(squad, target, grid, elevations, food_tiers, countries,
                 max_turns=80):
    """Joint schedule: when to march, when to camp, who feeds.
    
    Minimizes total turns to get leader to target while keeping everyone alive.
    """
    leader = squad.by_rank()[0]
    alive = squad.alive
    N = len(alive)
    
    # ── Leader's path (fixed route, variable pacing) ──
    lpath = leader.pathfind(target, grid, elevations, countries)
    H = len(lpath)  # waypoints along path
    T = max_turns
    
    # Food tiles at each waypoint
    wp_tiles = {}  # wp_index → [(tier, hex_idx), ...]
    for w, whex in enumerate(lpath.hexes):
        tiles = []
        for ring in range(1, 4):
            for hp in HexPosition.origin().ring(ring):
                idx = grid.hexposition_to_index(hp, whex)
                if idx < 0 or idx in grid.invalidRegion: continue
                if elevations[idx] <= 0: continue
                if countries is not None and countries[idx] < 0: continue
                tier = int(food_tiers[idx]) if 0 <= idx < len(food_tiers) else 0
                if tier >= 1: tiles.append((tier, idx))
        tiles.sort(key=lambda c: -c[0])
        wp_tiles[w] = tiles[:N]  # at most one per piece
    
    K = max(len(v) for v in wp_tiles.values()) if wp_tiles else 0
    
    diet = {i: alive[i].diet for i in range(N)}
    harvest = {i: alive[i].harvest_strength for i in range(N)}
    cap = {i: alive[i].food_capacity for i in range(N)}
    food0 = {i: alive[i].food for i in range(N)}
    is_pawn = {i: alive[i].piece_type == PieceType.PAWN for i in range(N)}
    
    m = pyo.ConcreteModel()
    m.T = pyo.RangeSet(0, T - 1)
    m.P = pyo.RangeSet(0, N - 1)    # pieces
    m.W = pyo.RangeSet(0, H - 1)    # waypoints
    
    # ── Decision variables ──
    
    # march[t] = 1 if squad advances one waypoint at time t
    m.march = pyo.Var(m.T, domain=pyo.Binary)
    
    # position[t] = waypoint index the squad is at after turn t
    m.pos = pyo.Var(m.T, domain=pyo.NonNegativeIntegers, bounds=(0, H - 1))
    
    # feeding[p, t] = 1 if piece p is harvesting at time t
    m.feeding = pyo.Var(m.P, m.T, domain=pyo.Binary)
    
    # food[p, t] = food level of piece p after turn t
    m.food = pyo.Var(m.P, m.T, domain=pyo.NonNegativeReals)
    
    # arrival = first turn the leader reaches target
    m.arrival = pyo.Var(domain=pyo.NonNegativeIntegers, bounds=(0, T))
    
    # arrived[t] = 1 if squad has reached target by turn t
    m.arrived = pyo.Var(m.T, domain=pyo.Binary)
    
    # ── Objective: minimize arrival turn ──
    m.obj = pyo.Objective(expr=m.arrival, sense=pyo.minimize)
    
    # ── Position tracking ──
    m.pos_init = pyo.Constraint(expr=m.pos[0] == m.march[0])
    m.pos_update = pyo.Constraint(m.T, rule=lambda m, t:
        m.pos[t] == sum(m.march[tt] for tt in m.T if tt <= t))
    
    # ── Can't march and feed same turn ──
    m.no_march_feed = pyo.Constraint(m.P, m.T, rule=lambda m, p, t:
        m.feeding[p, t] + m.march[t] <= 1)
    
    # ── Only pawns can feed ──
    m.pawn_only = pyo.Constraint(m.P, m.T, rule=lambda m, p, t:
        m.feeding[p, t] <= (1 if is_pawn[p] else 0))
    
    # ── Food dynamics ──
    # Average tier at current waypoint (simplified)
    avg_tier = {}
    for w in range(H):
        tiles = wp_tiles.get(w, [])
        avg_tier[w] = sum(t for t, _ in tiles) / max(len(tiles), 1) if tiles else 0
    
    # Simplified: food[p,t] = food[p,t-1] - diet + feeding[p,t] * harvest * avg_tier
    def food_rule(m, p, t):
        if t == 0:
            # Sum avg_tier over possible positions (linearization trick)
            prod = harvest[p] * sum(
                avg_tier[w] for w in range(H)) / max(H, 1)  # rough average
            return m.food[p, 0] == food0[p] - diet[p] + m.feeding[p, 0] * prod
        prod = harvest[p] * sum(
            avg_tier[w] for w in range(H)) / max(H, 1)
        return m.food[p, t] == m.food[p, t-1] - diet[p] + m.feeding[p, t] * prod
    m.food_dyn = pyo.Constraint(m.P, m.T, rule=food_rule)
    
    # ── Food capacity ──
    m.food_cap = pyo.Constraint(m.P, m.T, rule=lambda m, p, t:
        m.food[p, t] <= cap[p])
    
    # ── Nobody dies ──
    m.alive = pyo.Constraint(m.P, m.T, rule=lambda m, p, t:
        m.food[p, t] >= 0.5)
    
    # ── Must reach target ──
    m.must_arrive = pyo.Constraint(expr=m.pos[T-1] >= H - 1)
    
    # ── Arrival tracking ──
    m.arr_link = pyo.Constraint(m.T, rule=lambda m, t:
        m.arrival <= t + T * (1 - m.arrived[t]))
    m.arr_pos = pyo.Constraint(m.T, rule=lambda m, t:
        m.pos[t] >= (H - 1) * m.arrived[t])
    
    # ── Solve ──
    solver = pyo.SolverFactory('appsi_highs')
    result = solver.solve(m)
    
    if not _solver_ok(result):
        return None
    
    # ── Extract schedule ──
    schedule = []
    for t in m.T:
        turn = {
            'turn': t,
            'march': pyo.value(m.march[t]) > 0.5,
            'pos': int(round(pyo.value(m.pos[t]))),
            'feeders': [alive[p].name for p in m.P if pyo.value(m.feeding[p, t]) > 0.5],
            'food': {alive[p].name: round(pyo.value(m.food[p, t]), 1) for p in m.P},
        }
        schedule.append(turn)
    
    return schedule
```

But honestly — this is a **rough sketch**. The hard part is the bilinear term: `feeding[p,t] × avg_tier_at_position[t]` — the production rate depends on *where* you are, and where you are depends on march decisions. That makes it nonlinear.

There are three ways to handle it:

| Approach | Complexity | Quality |
|---|---|---|
| **Fix the route, optimize the pacing** | MIP (linear) | Good — route from pathfind, just decide march/camp per turn |
| **Discretize into legs** | Small MIP | Great — "how many camp turns at each waypoint?" |
| **Full joint route+schedule** | MINLP | Optimal but hard to solve |

The middle option is the sweet spot — the **"leap looker"**:

```python
def leap_schedule(squad, path, grid, elevations, food_tiers, countries,
                  max_camp_per_stop=40):
    """For each waypoint on the path, decide how many turns to camp.
    
    Minimize: sum of camp turns (= total journey time beyond pure march)
    Subject to: nobody's food drops to 0 at any point
    """
    # ... waypoints from path
    # Decision var: camp_turns[w] = integer, how long to camp at waypoint w
    # Food is tracked analytically through the schedule
    # Linear MIP — very fast
```

This is your "schedule frog" — it sees the whole journey and decides the **optimal camping schedule** in one shot. Want me to flesh out the discretized leg version? It's the cleanest and most solvable.

Can you build out the optimal camping schedule? Should this be patched on Plan Mission

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes — `leap_schedule` as a method on `SquadMission` that plans the whole journey upfront. The state machine then just *executes* the plan. Here's the full thing:

```python
def leap_schedule(squad, target, grid, elevations, food_tiers, countries,
                  min_food_turns=2.0, max_camp_per_stop=60):
    """Optimal camping schedule over a fixed route.
    
    Minimizes total camp turns while keeping squad food above a safety margin.
    Returns list of {waypoint, hex, camp_turns, net_prod, arrival_food, departure_food}.
    """
    leader = squad.by_rank()[0]
    alive = squad.alive
    
    lpath = leader.pathfind(target, grid, elevations, countries)
    if not lpath: return None
    
    W = len(lpath.hexes)
    total_diet = sum(p.diet for p in alive)
    total_cap = sum(p.food_capacity for p in alive)
    food0 = sum(p.food for p in alive)
    min_food = total_diet * min_food_turns  # safety margin
    
    # ── Precompute net production at each waypoint ──
    feeders = sorted([p for p in alive if p.harvest_strength > 0],
                     key=lambda p: -p.harvest_strength)
    
    net_prod = []
    for w, whex in enumerate(lpath.hexes):
        tiles = []
        for ring in range(1, 4):
            for hp in HexPosition.origin().ring(ring):
                idx = grid.hexposition_to_index(hp, whex)
                if idx < 0 or idx in grid.invalidRegion: continue
                if elevations[idx] <= 0: continue
                if countries is not None and countries[idx] < 0: continue
                tier = int(food_tiers[idx]) if 0 <= idx < len(food_tiers) else 0
                if tier >= 1: tiles.append(tier)
        tiles.sort(reverse=True)
        total_harvest = sum(f.harvest_strength * tiles[i]
                           for i, f in enumerate(feeders) if i < len(tiles))
        net_prod.append(total_harvest - total_diet)
    
    # ── March food cost per leg ──
    march_cost = []
    for w in range(W - 1):
        mc = _move_cost(elevations, lpath.hexes[w], lpath.hexes[w + 1])
        min_ms = min((p.move_strength for p in alive if p.move_strength > 0), default=1)
        est_turns = max(1, math.ceil(mc / min_ms))
        march_cost.append(total_diet * est_turns)
    
    # ── Pyomo MIP ──
    m = pyo.ConcreteModel()
    m.W = pyo.RangeSet(0, W - 1)
    
    # Decision: how many turns to camp at each waypoint
    m.camp = pyo.Var(m.W, domain=pyo.NonNegativeIntegers,
                     bounds=(0, max_camp_per_stop))
    
    # State: arrival food (before camping) and departure food (after camping)
    m.A = pyo.Var(m.W, domain=pyo.NonNegativeReals, bounds=(0, total_cap))
    m.F = pyo.Var(m.W, domain=pyo.NonNegativeReals, bounds=(0, total_cap))
    
    # ── Objective: minimize total camp turns ──
    m.obj = pyo.Objective(expr=sum(m.camp[w] for w in m.W), sense=pyo.minimize)
    
    # ── Food flow ──
    # Arrival at start = current squad food
    m.a_init = pyo.Constraint(expr=m.A[0] == food0)
    
    # Arrival at w = departure from w-1 minus march cost
    m.a_flow = pyo.Constraint(range(1, W), rule=lambda m, w:
        m.A[w] == m.F[w - 1] - march_cost[w - 1])
    
    # Departure = arrival + camp_turns × net_production
    m.f_camp = pyo.Constraint(m.W, rule=lambda m, w:
        m.F[w] == m.A[w] + m.camp[w] * net_prod[w])
    
    # ── Safety: never arrive starving ──
    m.survive = pyo.Constraint(m.W, rule=lambda m, w:
        m.A[w] >= min_food)
    
    # ── Capacity: can't store more than squad can carry ──
    m.capacity = pyo.Constraint(m.W, rule=lambda m, w:
        m.F[w] <= total_cap)
    
    # ── Don't camp where it hurts ──
    # If net_prod <= 0, camping only costs food
    for w in range(W):
        if net_prod[w] <= 0:
            m.camp[w].fix(0)
    
    # ── Solve ──
    solver = pyo.SolverFactory('appsi_highs')
    result = solver.solve(m)
    
    if not _solver_ok(result):
        return None
    
    # ── Extract schedule ──
    schedule = []
    total_camp_turns = 0
    for w in range(W):
        ct = int(round(pyo.value(m.camp[w])))
        total_camp_turns += ct
        schedule.append({
            'waypoint': w,
            'hex': lpath.hexes[w],
            'camp_turns': ct,
            'net_prod': net_prod[w],
            'arrival_food': round(pyo.value(m.A[w]), 1),
            'departure_food': round(pyo.value(m.F[w]), 1),
            'march_cost': march_cost[w] if w < W - 1 else 0,
        })
    
    total_march = sum(1 for w in range(W - 1))  # 1 move action per leg
    print(f"Schedule: {W} waypoints, {total_camp_turns} camp turns, "
          f"~{total_camp_turns + total_march} total turns")
    for s in schedule:
        if s['camp_turns'] > 0:
            print(f"  wp{s['waypoint']} hex {s['hex']}: "
                  f"camp {s['camp_turns']}t (net {s['net_prod']:+.1f}/t), "
                  f"food {s['arrival_food']}→{s['departure_food']}")
    
    return schedule
```

Then `SquadMission` uses it as the plan:

```python
class SquadMission:
    CAMP, MARCH, ARRIVED = range(3)
    
    def __init__(self, squad, target, grid, elevations, food_tiers, countries,
                 min_food_turns=2.0):
        self.squad = squad
        self.target = target
        self.grid = grid
        self.elevations = elevations
        self.food_tiers = food_tiers
        self.countries = countries
        self.leader = squad.by_rank()[0]
        
        # ── Plan the whole journey upfront ──
        self.schedule = leap_schedule(
            squad, target, grid, elevations, food_tiers, countries,
            min_food_turns=min_food_turns)
        
        self.leg = 0            # current schedule index
        self.camp_remaining = 0 # camp turns left at current waypoint
        
        # Start camping at first waypoint (if schedule says to)
        if self.schedule and self.schedule[0]['camp_turns'] > 0:
            self.camp_remaining = self.schedule[0]['camp_turns']
            self._enter_camp(self.schedule[0]['hex'])
            self.state = self.CAMP
        else:
            self._enter_march_to_next()
    
    def callback(self, report):
        if self.state == self.CAMP:
            self.camp_remaining -= 1
            if self.camp_remaining <= 0:
                self._enter_march_to_next()
        
        elif self.state == self.MARCH:
            if self.leader.location == self.schedule[self.leg]['hex']:
                # Arrived at this waypoint
                ct = self.schedule[self.leg]['camp_turns']
                if ct > 0:
                    self.camp_remaining = ct
                    self._enter_camp(self.schedule[self.leg]['hex'])
                    self.state = self.CAMP
                else:
                    self._enter_march_to_next()
        
        elif self.state == self.ARRIVED:
            self._enter_camp(self.target)
    
    def _enter_march_to_next(self):
        """March to the next waypoint in the schedule."""
        self.leg += 1
        if self.leg >= len(self.schedule):
            self.state = self.ARRIVED
            self._enter_camp(self.target)
            return
        
        self.state = self.MARCH
        next_hex = self.schedule[self.leg]['hex']
        orders = self.squad.march_to(
            next_hex, FormationType.BOX,
            grid=self.grid, elevations=self.elevations,
            leader=self.leader, countries=self.countries,
            food_tiers=self.food_tiers, sync=True)
        for p in self.squad.alive:
            if p.id in orders:
                il = orders[p.id]
                il.patrol = False
                p.instructions = il
    
    def _enter_camp(self, location):
        """Pawns to food tiles, others pause."""
        self.state = self.CAMP
        camp = self.squad.supply_assign(
            location, self.grid, self.elevations,
            self.food_tiers, self.countries)
        
        if camp.get('feasible'):
            for p in self.squad.alive:
                tile = camp.get('tile_for', {}).get(p.id)
                if tile is not None:
                    p.location = tile
                    p.facing = _facing_toward(self.grid, tile, location)
                    p.instructions = InstructionList(
                        [Instruction.HARVEST.value, Instruction.GIVE.value],
                        cursor=0, patrol=True)
                else:
                    p.instructions = InstructionList(
                        [Instruction.PAUSE.value], cursor=0, patrol=True)
        else:
            for p in self.squad.alive:
                p.instructions = InstructionList(
                    [Instruction.HARVEST.value, Instruction.GIVE.value],
                    cursor=0, patrol=True)
```

The beauty of this design:

- **`leap_schedule` is pure math** — no simulation, no state. It's a small linear MIP with just `2W` variables (camp turns + food levels at each waypoint). Solves in milliseconds.
- **No bilinearity** — production rate depends on *waypoint* (fixed parameter), not on decisions. `camp[w] × net_prod[w]` is linear because `net_prod[w]` is a constant.
- **SquadMission just follows the plan** — the state machine is trivial: camp N turns, march to next hex, camp M turns, repeat.
- **Optimal** — it finds the minimum total camp time. Maybe "camp 3 at waypoint 1, march 2, camp 5 at waypoint 3" is faster than "camp 40, march 4."
- **`supply_assign`** handles the *who feeds where* at each camp — the schedule handles the *when and how long*

I lover this and want to make it more graphic so the player understands what is going on. I have 2 things
1. We could make leap_schedule return a panda dataframe. I don't know if it should be wrapped with some meta data like the path so we should have a schedule class
2. we should have an overlay that demostrates the schedule

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Love it. Here's the design:

**1. `LeapSchedule` — data class wrapping the plan**

```python
class LeapSchedule:
    """Optimal camping schedule for a squad march."""
    
    def __init__(self, squad, path, df, net_prod, total_diet, total_cap):
        self.squad = squad
        self.path = path          # TroopPath
        self.df = df              # pandas DataFrame
        self.net_prod = net_prod  # per-waypoint net production rates
        self.total_diet = total_diet
        self.total_cap = total_cap
    
    @property
    def total_camp(self): return int(self.df['camp_turns'].sum())
    
    @property
    def total_turns(self): return self.total_camp + len(self.df) - 1
    
    @property
    def camp_stops(self): return self.df[self.df['camp_turns'] > 0]
    
    def __repr__(self):
        stops = len(self.camp_stops)
        return (f"LeapSchedule({len(self.df)} waypoints, "
                f"{stops} camps, {self.total_camp} camp turns, "
                f"~{self.total_turns} total)")
    
    def _repr_html_(self):
        """Rich display in solveit."""
        from io import StringIO
        buf = StringIO()
        buf.write(f"<h4>🐸 Leap Schedule</h4>")
        buf.write(f"<p>{len(self.df)} waypoints · "
                  f"{len(self.camp_stops)} camp stops · "
                  f"{self.total_camp} camp turns · "
                  f"~{self.total_turns} total turns</p>")
        buf.write(f"<p>Diet: {self.total_diet:.1f}/t · "
                  f"Capacity: {self.total_cap:.0f}</p>")
        
        # Style the dataframe
        styled = (self.df.style
            .format({
                'net_prod': '{:+.1f}',
                'arrival_food': '{:.0f}',
                'departure_food': '{:.0f}',
                'march_cost': '{:.0f}',
                'food_pct': '{:.0%}',
            })
            .bar(subset=['camp_turns'], color='#66BB6A', vmin=0)
            .background_gradient(subset=['food_pct'], cmap='RdYlGn', vmin=0, vmax=1)
        )
        buf.write(styled.to_html())
        return buf.getvalue()
```

Update `leap_schedule` to return it:

```python
def leap_schedule(squad, target, grid, elevations, food_tiers, countries,
                  min_food_turns=2.0, max_camp_per_stop=60):
    """Optimal camping schedule. Returns a LeapSchedule."""
    leader = squad.by_rank()[0]
    alive = squad.alive
    
    lpath = leader.pathfind(target, grid, elevations, countries)
    if not lpath: return None
    
    W = len(lpath.hexes)
    total_diet = sum(p.diet for p in alive)
    total_cap = sum(p.food_capacity for p in alive)
    food0 = sum(p.food for p in alive)
    min_food = total_diet * min_food_turns
    
    # ── Net production at each waypoint ──
    feeders = sorted([p for p in alive if p.harvest_strength > 0],
                     key=lambda p: -p.harvest_strength)
    
    net_prod = []
    for w, whex in enumerate(lpath.hexes):
        tiles = []
        for ring in range(1, 4):
            for hp in HexPosition.origin().ring(ring):
                idx = grid.hexposition_to_index(hp, whex)
                if idx < 0 or idx in grid.invalidRegion: continue
                if elevations[idx] <= 0: continue
                if countries is not None and countries[idx] < 0: continue
                tier = int(food_tiers[idx]) if 0 <= idx < len(food_tiers) else 0
                if tier >= 1: tiles.append(tier)
        tiles.sort(reverse=True)
        total_harvest = sum(f.harvest_strength * tiles[i]
                           for i, f in enumerate(feeders) if i < len(tiles))
        net_prod.append(total_harvest - total_diet)
    
    # ── March cost per leg ──
    march_cost = []
    for w in range(W - 1):
        mc = _move_cost(elevations, lpath.hexes[w], lpath.hexes[w + 1])
        min_ms = min((p.move_strength for p in alive if p.move_strength > 0), default=1)
        est_turns = max(1, math.ceil(mc / min_ms))
        march_cost.append(total_diet * est_turns)
    
    # ── Pyomo MIP ──
    m = pyo.ConcreteModel()
    m.W = pyo.RangeSet(0, W - 1)
    m.camp = pyo.Var(m.W, domain=pyo.NonNegativeIntegers,
                     bounds=(0, max_camp_per_stop))
    m.A = pyo.Var(m.W, domain=pyo.NonNegativeReals, bounds=(0, total_cap))
    m.F = pyo.Var(m.W, domain=pyo.NonNegativeReals, bounds=(0, total_cap))
    
    m.obj = pyo.Objective(expr=sum(m.camp[w] for w in m.W), sense=pyo.minimize)
    
    m.a_init = pyo.Constraint(expr=m.A[0] == food0)
    m.a_flow = pyo.Constraint(range(1, W), rule=lambda m, w:
        m.A[w] == m.F[w - 1] - march_cost[w - 1])
    m.f_camp = pyo.Constraint(m.W, rule=lambda m, w:
        m.F[w] == m.A[w] + m.camp[w] * net_prod[w])
    m.survive = pyo.Constraint(m.W, rule=lambda m, w:
        m.A[w] >= min_food)
    m.capacity = pyo.Constraint(m.W, rule=lambda m, w:
        m.F[w] <= total_cap)
    
    for w in range(W):
        if net_prod[w] <= 0:
            m.camp[w].fix(0)
    
    solver = pyo.SolverFactory('appsi_highs')
    result = solver.solve(m)
    if not _solver_ok(result): return None
    
    # ── Build DataFrame ──
    rows = []
    for w in range(W):
        ct = int(round(pyo.value(m.camp[w])))
        af = pyo.value(m.A[w])
        df = pyo.value(m.F[w])
        rows.append({
            'waypoint': w,
            'hex': lpath.hexes[w],
            'camp_turns': ct,
            'net_prod': net_prod[w],
            'arrival_food': af,
            'departure_food': df,
            'march_cost': march_cost[w] if w < W - 1 else 0,
            'food_pct': af / total_cap if total_cap > 0 else 0,
        })
    
    return LeapSchedule(
        squad=squad, path=lpath,
        df=pd.DataFrame(rows),
        net_prod=net_prod,
        total_diet=total_diet,
        total_cap=total_cap)
```

**2. `ScheduleOverlay` — visualize the plan on the map**

```python
class ScheduleOverlay(Overlay):
    """Visualize a LeapSchedule on the terrain map."""
    
    def __init__(self, schedule: LeapSchedule, 
                 camp_color='#66BB6A', march_color='#dee5a5',
                 danger_color='#e74c3c', label=True):
        self.schedule = schedule
        self.camp_color = camp_color
        self.march_color = march_color
        self.danger_color = danger_color
        self.label = label
    
    def render(self, ctx: OverlayContext):
        s = self.schedule
        grid = ctx.grid
        layer = ctx.layer('schedule')
        
        hexes = s.path.hexes
        df = s.df
        
        # ── Route line ──
        points = []
        for whex in hexes:
            cx, cy = grid.hex_center(whex)
            points.append(f"{cx},{cy}")
        
        if len(points) >= 2:
            layer.add(SvgElement('polyline', attrs={
                'points': ' '.join(points),
                'fill': 'none',
                'stroke': self.march_color,
                'stroke-width': '3',
                'stroke-dasharray': '8,4',
                'opacity': '0.7',
            }))
        
        # ── Waypoint markers ──
        for _, row in df.iterrows():
            w = int(row['waypoint'])
            whex = int(row['hex'])
            cx, cy = grid.hex_center(whex)
            ct = int(row['camp_turns'])
            fp = row['food_pct']
            
            # Color by food level
            if fp < 0.15:
                color = self.danger_color
            elif ct > 0:
                color = self.camp_color
            else:
                color = self.march_color
            
            # Size by camp duration (min 4, max 16)
            r = max(4, min(16, 4 + ct * 0.5))
            
            # Outer ring
            layer.add(SvgElement('circle', attrs={
                'cx': str(cx), 'cy': str(cy), 'r': str(r + 2),
                'fill': 'none', 'stroke': color,
                'stroke-width': '2', 'opacity': '0.8',
            }))
            
            # Inner fill — opacity proportional to food
            layer.add(SvgElement('circle', attrs={
                'cx': str(cx), 'cy': str(cy), 'r': str(r),
                'fill': color, 'opacity': str(max(0.3, fp)),
            }))
            
            # ── Labels ──
            if self.label and ct > 0:
                # Camp turns badge
                layer.add(SvgElement('text', text=f"⛺{ct}", attrs={
                    'x': str(cx + r + 4), 'y': str(cy - 4),
                    'font-size': '10', 'fill': color,
                    'font-weight': 'bold', 'font-family': 'monospace',
                }))
                # Food level
                layer.add(SvgElement('text', 
                    text=f"{row['arrival_food']:.0f}→{row['departure_food']:.0f}",
                    attrs={
                        'x': str(cx + r + 4), 'y': str(cy + 8),
                        'font-size': '8', 'fill': color,
                        'opacity': '0.7', 'font-family': 'monospace',
                    }))
            
            # Waypoint number
            layer.add(SvgElement('text', text=str(w), attrs={
                'x': str(cx), 'y': str(cy + 3),
                'font-size': '9', 'fill': 'white',
                'text-anchor': 'middle', 'font-weight': 'bold',
                'font-family': 'monospace',
            }))
```

**Usage:**

```python
sched = leap_schedule(squad, 236, grid, elevations, food_tiers, countries)
sched  # rich HTML table in solveit

TerrainDisplay(
    TerrainOverlay(),
    FoodOverlay(),
    ScheduleOverlay(sched),
    PieceOverlay([squad]),
    terrain=myStuff.terr,
    board=myStuff.board,
    basins=myStuff.basin,
)
```

The player sees:
- **Dashed line** along the route
- **Green circles** at camp stops, sized by duration — big circle = long camp
- **⛺12** labels showing camp turns
- **44→188** showing food before/after camping
- **Red circles** where food gets dangerously low
- The **DataFrame** with green/red gradient showing food levels at each waypoint

The `SquadMission` would take a `LeapSchedule` instead of computing it internally — same execution logic, but the player can *see* the plan before committing.

@patch
def _enter_camp(self:SquadMission, location):
    """Set up camp. Does NOT change state — caller manages that."""
    camp = self.squad.supply_assign(
        location, self.grid, self.elevations,
        self.food_tiers, self.countries)
    
    if camp.get('feasible'):
        tile_for = camp.get('tile_for', {})
        for p in self.squad.alive:
            tile = tile_for.get(p.id)
            if tile is not None:
                p.location = tile
                p.facing = _facing_toward(self.grid, tile, location)
            # EVERYONE harvests and gives — not just feeders
            p.instructions = InstructionList(
                [Instruction.HARVEST.value, Instruction.GIVE.value],
                cursor=0, patrol=True)
    else:
        for p in self.squad.alive:
            p.instructions = InstructionList(
                [Instruction.HARVEST.value, Instruction.GIVE.value],
                cursor=0, patrol=True)


In [ ]:
def _enter_camp(self:SquadMission, location):
    """Set up camp. Does NOT change state — caller manages that."""
    # During PREP, force ALL pawns as feeders for max surplus
    tiles = []
    for ring in range(1, 4):
        for hp in HexPosition.origin().ring(ring):
            idx = self.grid.hexposition_to_index(hp, location)
            if idx < 0 or idx in self.grid.invalidRegion: continue
            if self.elevations[idx] <= 0: continue
            if self.countries is not None and self.countries[idx] < 0: continue
            tier = int(self.food_tiers[idx]) if 0 <= idx < len(self.food_tiers) else 0
            if tier >= 1: tiles.append((tier, ring, idx))
    tiles.sort(key=lambda c: (-c[0], c[1]))
    
    claimed = set()
    for p in self.squad.alive:
        if p.piece_type == PieceType.PAWN:
            for tier, ring, idx in tiles:
                if idx not in claimed:
                    p.location = idx; claimed.add(idx)
                    p.facing = _facing_toward(self.grid, idx, location)
                    break
        p.instructions = InstructionList(
            [Instruction.HARVEST.value, Instruction.GIVE.value],
            cursor=0, patrol=True)


In [ ]:
@patch
def _enter_camp(self:SquadMission, location):
    """Set up camp. Does NOT change state — caller manages that."""
    tiles = []
    for ring in range(1, 4):
        for hp in HexPosition.origin().ring(ring):
            idx = self.grid.hexposition_to_index(hp, location)
            if idx < 0 or idx in self.grid.invalidRegion: continue
            if self.elevations[idx] <= 0: continue
            if self.countries is not None and self.countries[idx] < 0: continue
            tier = int(self.food_tiers[idx]) if 0 <= idx < len(self.food_tiers) else 0
            if tier >= 1: tiles.append((tier, ring, idx))
    tiles.sort(key=lambda c: (-c[0], c[1]))
    
    claimed = set()
    for p in self.squad.alive:
        if p.piece_type == PieceType.PAWN:
            for tier, ring, idx in tiles:
                if idx not in claimed:
                    p.location = idx; claimed.add(idx)
                    p.facing = _facing_toward(self.grid, idx, location)
                    break
        p.instructions = InstructionList(
            [Instruction.HARVEST.value, Instruction.GIVE.value],
            cursor=0, patrol=True)


In [ ]:
demoTerr()
myStuff.apply_food_profile(soil_scale=7)
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()

squad = myStuff.queenGuard
mission = SquadMission(
    squad, target=236,
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers,
    countries=myStuff.terr.fields.get('country'))

sim = FoodSimulator(
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers,
    pieces=squad.alive,
    countries=myStuff.terr.fields.get('country'),
    squads=[squad],
    callbacks={squad.id: compose_callbacks(food_triage, mission.callback)},
)
sim.run(num_turns=120)
sim.summary()


In [ ]:
myStuff.queen.location

the queen doesn't make it across?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The queen is still at 158 — she never marched. The bug is in the original `_enter_camp`:

```python
def _enter_camp(self, location):
    self.state = self.CAMP  # ← THIS is the problem
    ...
```

Here's the sequence:
1. `__init__`: `self.state = self.PREP`
2. `__init__`: `self._enter_camp(...)` → **overwrites state to CAMP**
3. `callback`: sees CAMP → does nothing forever

The state machine gets stuck in CAMP and never transitions to MARCH. Your note had the right fix (`_enter_camp` should NOT change state), but it's a note — it wasn't executed. And the code cell version isn't `@patch`ed so it didn't replace the method.

Quick fix — make the `@patch` version live in a code cell:

```python
@patch
def _enter_camp(self:SquadMission, location):
    """Set up camp. Does NOT change state — caller manages that."""
    tiles = []
    for ring in range(1, 4):
        for hp in HexPosition.origin().ring(ring):
            idx = self.grid.hexposition_to_index(hp, location)
            if idx < 0 or idx in self.grid.invalidRegion: continue
            if self.elevations[idx] <= 0: continue
            if self.countries is not None and self.countries[idx] < 0: continue
            tier = int(self.food_tiers[idx]) if 0 <= idx < len(self.food_tiers) else 0
            if tier >= 1: tiles.append((tier, ring, idx))
    tiles.sort(key=lambda c: (-c[0], c[1]))
    
    claimed = set()
    for p in self.squad.alive:
        if p.piece_type == PieceType.PAWN:
            for tier, ring, idx in tiles:
                if idx not in claimed:
                    p.location = idx; claimed.add(idx)
                    p.facing = _facing_toward(self.grid, idx, location)
                    break
        p.instructions = InstructionList(
            [Instruction.HARVEST.value, Instruction.GIVE.value],
            cursor=0, patrol=True)
```

Then re-run. The flow will be: PREP (camp + fill) → MARCH (when ready) → ARRIVED → CAMP (at 236).

In [ ]:
# Reset to fresh state
demoTerr()
fy = myStuff.apply_food_profile(soil_scale=7)
food_tiers = fy.tiers

grid = myStuff.grid
elevs = myStuff.terr.elevations
countries = myStuff.terr.fields.get('country')

# Save original state (leapfrog_plan mutates food/location)
orig = {p.id: (p.food, p.location, p.facing) for p in myStuff.queenGuard.alive}

# Plan the leapfrog
route = myStuff.queen.pathfind(236, grid, elevs, countries)
print(f"Route to 236: {len(route)} hexes, cost {route.cost:.0f}")

recovery_turns = 4
legs = myStuff.queenGuard.leapfrog_plan(
    route, grid, elevs,
    formation=FormationType.BOX,
    countries=countries,
    food_tiers=food_tiers,
    recovery_turns=recovery_turns)

for leg in legs:
    fc = leg['feed_cover']
    print(f"  Leg {leg['leg_number']}: {leg['waypoint']} hexes → hex {leg['hex_idx']} "
          f"(tier {leg['recovery_tier']}, feeders: {[p.name for p in fc['feeders']]})")

# Restore original state
for p in myStuff.queenGuard.pieces:
    if p.id in orig:
        p.food, p.location, p.facing = orig[p.id]

# Build chained instructions: march → recovery → march → recovery → ...
recovery = [Instruction.HARVEST.value, Instruction.GIVE.value] * recovery_turns

for p in myStuff.queenGuard.alive:
    all_rules = []
    facing = p.facing

    for leg in legs:
        tp = leg['paths'].get(p.id)
        if tp and len(tp) > 1:
            move_rules = tp.to_rules(grid, start_facing=facing)
            all_rules.extend(move_rules)
            facing = _facing_toward(grid, tp[-2], tp[-1])
        # Rest & resupply at each waypoint
        all_rules.extend(recovery)

    p.instructions = InstructionList(all_rules, cursor=0, patrol=False)
    print(f"{p.name:10s} ({p.piece_type.name:6s}): {len(all_rules)} instr across {len(legs)} legs")

# Simulate
sim = FoodSimulator(
    grid=grid, elevations=elevs, food_tiers=food_tiers,
    pieces=myStuff.queenGuard.alive,
    countries=countries,
    squads=[myStuff.queenGuard],
    callbacks={myStuff.queenGuard.id: food_triage},
)
sim.run(num_turns=60)
sim.summary()


Can you rewrite this so that it uses a leap_frog plan

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Here's the full leapfrog approach — plan waypoints, then chain march + recovery instructions per leg:

```python
# Reset to fresh state
demoTerr()
fy = myStuff.apply_food_profile(soil_scale=7)
food_tiers = fy.tiers

grid = myStuff.grid
elevs = myStuff.terr.elevations
countries = myStuff.terr.fields.get('country')

# Save original state (leapfrog_plan mutates food/location)
orig = {p.id: (p.food, p.location, p.facing) for p in myStuff.queenGuard.alive}

# Plan the leapfrog
route = myStuff.queen.pathfind(236, grid, elevs, countries)
print(f"Route to 236: {len(route)} hexes, cost {route.cost:.0f}")

recovery_turns = 4
legs = myStuff.queenGuard.leapfrog_plan(
    route, grid, elevs,
    formation=FormationType.BOX,
    countries=countries,
    food_tiers=food_tiers,
    recovery_turns=recovery_turns)

for leg in legs:
    fc = leg['feed_cover']
    print(f"  Leg {leg['leg_number']}: {leg['waypoint']} hexes → hex {leg['hex_idx']} "
          f"(tier {leg['recovery_tier']}, feeders: {[p.name for p in fc['feeders']]})")

# Restore original state
for p in myStuff.queenGuard.pieces:
    if p.id in orig:
        p.food, p.location, p.facing = orig[p.id]

# Build chained instructions: march → recovery → march → recovery → ...
recovery = [Instruction.HARVEST.value, Instruction.GIVE.value] * recovery_turns

for p in myStuff.queenGuard.alive:
    all_rules = []
    facing = p.facing

    for leg in legs:
        tp = leg['paths'].get(p.id)
        if tp and len(tp) > 1:
            move_rules = tp.to_rules(grid, start_facing=facing)
            all_rules.extend(move_rules)
            facing = _facing_toward(grid, tp[-2], tp[-1])
        # Rest & resupply at each waypoint
        all_rules.extend(recovery)

    p.instructions = InstructionList(all_rules, cursor=0, patrol=False)
    print(f"{p.name:10s} ({p.piece_type.name:6s}): {len(all_rules)} instr across {len(legs)} legs")

# Simulate
sim = FoodSimulator(
    grid=grid, elevations=elevs, food_tiers=food_tiers,
    pieces=myStuff.queenGuard.alive,
    countries=countries,
    squads=[myStuff.queenGuard],
    callbacks={myStuff.queenGuard.id: food_triage},
)
sim.run(num_turns=60)
sim.summary()
```

The key differences from the straight `march_to`:
- **Leapfrog plans waypoints** where the squad can sustainably refuel, rather than one long death march
- **Recovery cycles** (HARVEST + GIVE × 4) are inserted between each leg so pawns restock the queen
- **`patrol=False`** — these are one-shot orders that end after the final leg (you could add a trailing harvest/give loop if you want them to sustain at the destination too)

If BOX formation limits range too much, try `formation=None` for broken formation — it typically reaches further per leg.

In [ ]:
# Run the simulator with food_triage so the squad stops to feed
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()
food_tiers = fy.tiers

sim = FoodSimulator(
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=food_tiers,
    pieces=myStuff.queenGuard.alive,
    countries=myStuff.terr.fields.get('country'),
    squads=[myStuff.queenGuard],
    callbacks={myStuff.queenGuard.id: food_triage},
)

sim.run(num_turns=40)
sim.summary()


In [ ]:
show(sim)

Once we arrive can we remove instructions and then give everybody except the queen to surround and start to feed the hungriest pieces. The queen will defend.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Absolutely — after the march completes, clear instructions and assign roles. The GIVE action already targets the hungriest visible ally, so pawns/bishops harvesting and giving will naturally funnel food to the queen (and each other).

```python
# Reset to fresh state
demoTerr()
fy = myStuff.apply_food_profile(soil_scale=7)
food_tiers = fy.tiers

grid = myStuff.grid
elevs = myStuff.terr.elevations
countries = myStuff.terr.fields.get('country')

# Save original state (leapfrog_plan mutates food/location)
orig = {p.id: (p.food, p.location, p.facing) for p in myStuff.queenGuard.alive}

# Plan the leapfrog
route = myStuff.queen.pathfind(236, grid, elevs, countries)
print(f"Route to 236: {len(route)} hexes, cost {route.cost:.0f}")

recovery_turns = 4
legs = myStuff.queenGuard.leapfrog_plan(
    route, grid, elevs,
    formation=FormationType.BOX,
    countries=countries,
    food_tiers=food_tiers,
    recovery_turns=recovery_turns)

for leg in legs:
    fc = leg['feed_cover']
    print(f"  Leg {leg['leg_number']}: {leg['waypoint']} hexes → hex {leg['hex_idx']} "
          f"(tier {leg['recovery_tier']}, feeders: {[p.name for p in fc['feeders']]})")

# Restore original state
for p in myStuff.queenGuard.pieces:
    if p.id in orig:
        p.food, p.location, p.facing = orig[p.id]

# Build chained instructions: march → recovery → march → ... → arrival role
recovery = [Instruction.HARVEST.value, Instruction.GIVE.value] * recovery_turns

queen = myStuff.queen
for p in myStuff.queenGuard.alive:
    all_rules = []
    facing = p.facing

    # March legs with recovery stops
    for leg in legs:
        tp = leg['paths'].get(p.id)
        if tp and len(tp) > 1:
            move_rules = tp.to_rules(grid, start_facing=facing)
            all_rules.extend(move_rules)
            facing = _facing_toward(grid, tp[-2], tp[-1])
        all_rules.extend(recovery)

    # === Post-arrival roles ===
    if p.id == queen.id:
        # Queen defends in place
        arrival_role = [Instruction.DEFEND.value]
    else:
        # Everyone else: face toward queen, then harvest+give loop
        target_facing = _facing_toward(grid, p.location, queen.location)
        diff = (target_facing - facing) % 6
        if diff <= 3:
            rot = [Instruction.ROT_R.value] * diff
        else:
            rot = [Instruction.ROT_L.value] * (6 - diff)
        arrival_role = rot + [Instruction.HARVEST.value, Instruction.GIVE.value]

    all_rules.extend(arrival_role)
    # Patrol=True so the arrival role loops forever
    p.instructions = InstructionList(all_rules, cursor=0, patrol=True)
    role = "DEFEND" if p.id == queen.id else "HARVEST+GIVE"
    print(f"{p.name:10s} ({p.piece_type.name:6s}): {len(all_rules)} instr, arrival={role}")

# Simulate
sim = FoodSimulator(
    grid=grid, elevations=elevs, food_tiers=food_tiers,
    pieces=myStuff.queenGuard.alive,
    countries=countries,
    squads=[myStuff.queenGuard],
    callbacks={myStuff.queenGuard.id: food_triage},
)
sim.run(num_turns=60)
sim.summary()
```

The key bits:
- **Queen** loops DEFEND forever after arriving — she won't move or harvest
- **Everyone else** rotates to face the queen's hex, then loops HARVEST → GIVE — this means their GIVE cone points at the queen, funnelling food to whichever ally is hungriest (which will usually be the queen since her diet is 7.0 and she's not harvesting)
- `patrol=True` on the whole `InstructionList` means the march instructions run once, then the arrival role loops indefinitely

Note: the rotation-to-face-queen uses `p.location` which is the *original* location before the sim runs. If the formation slots differ significantly, you might want to compute the facing from the leg's target hex instead. But for a BOX formation around hex 236, it should be close enough.

In [ ]:
# Reset to fresh state
demoTerr()
fy = myStuff.apply_food_profile(soil_scale=7)
food_tiers = fy.tiers

grid = myStuff.grid
elevs = myStuff.terr.elevations
countries = myStuff.terr.fields.get('country')

# Save original state (leapfrog_plan mutates food/location)
orig = {p.id: (p.food, p.location, p.facing) for p in myStuff.queenGuard.alive}

# Plan the leapfrog
route = myStuff.queen.pathfind(236, grid, elevs, countries)
print(f"Route to 236: {len(route)} hexes, cost {route.cost:.0f}")

recovery_turns = 4
legs = myStuff.queenGuard.leapfrog_plan(
    route, grid, elevs,
    formation=FormationType.BOX,
    countries=countries,
    food_tiers=food_tiers,
    recovery_turns=recovery_turns)

for leg in legs:
    fc = leg['feed_cover']
    print(f"  Leg {leg['leg_number']}: {leg['waypoint']} hexes → hex {leg['hex_idx']} "
          f"(tier {leg['recovery_tier']}, feeders: {[p.name for p in fc['feeders']]})")

# Restore original state
for p in myStuff.queenGuard.pieces:
    if p.id in orig:
        p.food, p.location, p.facing = orig[p.id]

# Build chained instructions: march → recovery → march → ... → arrival role
recovery = [Instruction.HARVEST.value, Instruction.GIVE.value] * recovery_turns

queen = myStuff.queen
for p in myStuff.queenGuard.alive:
    all_rules = []
    facing = p.facing

    # March legs with recovery stops
    for leg in legs:
        tp = leg['paths'].get(p.id)
        if tp and len(tp) > 1:
            move_rules = tp.to_rules(grid, start_facing=facing)
            all_rules.extend(move_rules)
            facing = _facing_toward(grid, tp[-2], tp[-1])
        all_rules.extend(recovery)

    # === Post-arrival roles ===
    if p.id == queen.id:
        # Queen defends in place
        arrival_role = [Instruction.DEFEND.value]
    else:
        # Everyone else: face toward queen, then harvest+give loop
        target_facing = _facing_toward(grid, p.location, queen.location)
        diff = (target_facing - facing) % 6
        if diff <= 3:
            rot = [Instruction.ROT_R.value] * diff
        else:
            rot = [Instruction.ROT_L.value] * (6 - diff)
        arrival_role = rot + [Instruction.HARVEST.value, Instruction.GIVE.value]

    all_rules.extend(arrival_role)
    # Patrol=True so the arrival role loops forever
    p.instructions = InstructionList(all_rules, cursor=0, patrol=True)
    role = "DEFEND" if p.id == queen.id else "HARVEST+GIVE"
    print(f"{p.name:10s} ({p.piece_type.name:6s}): {len(all_rules)} instr, arrival={role}")

# Simulate
sim = FoodSimulator(
    grid=grid, elevations=elevs, food_tiers=food_tiers,
    pieces=myStuff.queenGuard.alive,
    countries=countries,
    squads=[myStuff.queenGuard],
    callbacks={myStuff.queenGuard.id: food_triage},
)
sim.run(num_turns=60)
sim.summary()


In [ ]:
show(sim)


In [ ]:
FoodSimulator.__ft__??

can you rewrite FoodSimulator.__ft__ so that it does it by rank (so there would be a line for all of the pawns, a line for all of the queens, etc)?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Sure! You'll need two changes: (1) store per-piece data in each snapshot during `run()`, and (2) the new `__ft__`.

**Step 1** — In your snapshot recording (inside `run()`), add a `piece_snaps` field:

```python
# Add this wherever you build the snapshot dict in FoodSimulator.run():
snap['piece_snaps'] = [(p.piece_type.name, p.health, p.food, p.diet) for p in self.pieces]
```

**Step 2** — The new `__ft__`:

```python
@patch
def __ft__(self: FoodSimulator):
    """Line charts by piece rank: alive count, avg health, food turns remaining."""
    if not self.snapshots:
        return P("No turns simulated yet", cls="text-sm opacity-50")
    if 'piece_snaps' not in self.snapshots[0]:
        return P("No per-piece data in snapshots — add piece_snaps to snapshot recording", cls="text-sm text-red-500")

    rank_colors = {
        'PAWN': '#2ecc71', 'BISHOP': '#3498db', 'KNIGHT': '#9b59b6',
        'ROOK': '#e67e22', 'QUEEN': '#e74c3c', 'KING': '#f1c40f',
    }
    rank_symbols = {
        'PAWN': '♟', 'BISHOP': '♝', 'KNIGHT': '♞',
        'ROOK': '♜', 'QUEEN': '♛', 'KING': '♚',
    }
    rank_order = ['PAWN', 'BISHOP', 'KNIGHT', 'ROOK', 'QUEEN', 'KING']

    # Discover which ranks are present
    ranks_present = sorted(
        {ps[0] for snap in self.snapshots for ps in snap['piece_snaps']},
        key=lambda r: rank_order.index(r) if r in rank_order else 99)

    # Build per-rank time series
    turns = [s['turn'] for s in self.snapshots]
    rank_data = {r: {'alive': [], 'avg_health': [], 'avg_food_turns': []} for r in ranks_present}

    for snap in self.snapshots:
        by_rank = {}
        for (rname, health, food, diet) in snap['piece_snaps']:
            by_rank.setdefault(rname, []).append((health, food, diet))
        for r in ranks_present:
            pieces = by_rank.get(r, [])
            alive = [(h, f, d) for h, f, d in pieces if h > 0]
            rank_data[r]['alive'].append(len(alive))
            if alive:
                rank_data[r]['avg_health'].append(sum(h for h, f, d in alive) / len(alive))
                rank_data[r]['avg_food_turns'].append(
                    sum(f / max(d, 0.01) for h, f, d in alive) / len(alive))
            else:
                rank_data[r]['avg_health'].append(0)
                rank_data[r]['avg_food_turns'].append(0)

    fig, axes = plt.subplots(1, 3, figsize=(11, 3))

    for r in ranks_present:
        c = rank_colors.get(r, '#95a5a6')
        sym = rank_symbols.get(r, '?')
        label = f"{sym} {r.capitalize()}"
        axes[0].plot(turns, rank_data[r]['alive'], 'o-', color=c, lw=2, ms=3, label=label)
        axes[1].plot(turns, rank_data[r]['avg_health'], 'o-', color=c, lw=2, ms=3, label=label)
        axes[2].plot(turns, rank_data[r]['avg_food_turns'], 'o-', color=c, lw=2, ms=3, label=label)

    axes[0].set_title('Alive (by rank)', fontweight='bold', fontsize=10)
    axes[0].yaxis.set_major_locator(plt.MaxNLocator(integer=True))
    axes[1].set_title('Avg Health (by rank)', fontweight='bold', fontsize=10)
    axes[2].set_title('Food turns left (by rank)', fontweight='bold', fontsize=10)
    axes[2].axhline(FOOD_RESERVE_TURNS, color='#aaa', ls=':', lw=1, label=f'reserve ({FOOD_RESERVE_TURNS})')

    for ax in axes:
        ax.set_xlabel('Turn')
        ax.set_ylim(bottom=0)
        ax.legend(fontsize=7, loc='best')
        ax.grid(axis='y', alpha=0.3, lw=0.5)
        ax.tick_params(labelsize=8)

    fig.tight_layout()
    buf = io.StringIO()
    fig.savefig(buf, format='svg', bbox_inches='tight')
    plt.close(fig)

    return Div(
        NotStr(buf.getvalue()),
        P(f"Simulated {self.turn} turns — {len(self.alive)}/{len(self.pieces)} alive",
          cls="text-xs opacity-60 text-center"),
        cls="space-y-1")
```

Each rank gets its own colored line with a chess symbol label (♟ Pawn, ♝ Bishop, ♛ Queen, etc). The three panels show:
- **Alive** — integer count per rank (so you'll see the 5 pawns drop vs the 1 queen)
- **Avg Health** — average health of living members of that rank
- **Food turns left** — average food/diet for living members of that rank

The `max(d, 0.01)` guard prevents division-by-zero if diet is ever 0. The `piece_snaps` line in step 1 is the only change needed in `run()` — just add it right after where you currently build the snapshot dict.

If these were employees they would be on strike (though probably not a hunger one) am I applying the proper food_yield or it probably doesn't matter since things are binned. should we start with much higher capacity so they can go three hexes without food?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Ha! 😄 Fair point about the strike.

The `soil_scale=7` is being applied correctly — `apply_food_profile` returns the scaled `fy` and you're using its tiers. But you're right that it doesn't really matter much because the problem isn't **at the destination** — it's **during transit**.

Here's the math that kills them:

| Piece | Capacity | Diet | Turns of food |
|-------|----------|------|---------------|
| Queen | 30 | 7.0 | **4.3 turns** |
| Pawn | 12 | 2.0 | 6 turns |
| Bishop | 100 | 2.5 | 40 turns |

The queen starves by turn 5 no matter what. Nobody is harvesting or giving during the march — they're all moving. And `sync_paths` pads early arrivals with PAUSE (not HARVEST), so they burn food doing nothing.

**Two complementary fixes:**

1. **Bump capacities** so the squad can actually survive a march:

```python
PROFILES = {
    PieceType.PAWN:   (2.0,  2.0,     24),   # was 12 → 12 turns
    PieceType.BISHOP: (2.5,  1.0,    100),   # already fine
    PieceType.KNIGHT: (3.0,  1.5,     40),   # was 20
    PieceType.ROOK:   (3.5,  1.0,     60),   # was 40
    PieceType.QUEEN:  (7.0,  0.5,     70),   # was 30 → 10 turns
    PieceType.KING:   (6.0,  0.5,     60),   # was 25 → 10 turns
}
```

2. **Use `leapfrog_plan`** instead of a single `march_to` — it was designed exactly for this. It chains march legs with feeding stops in between, using `pre_march_distribute` (bishop tops everyone up) and `squad_recover` (harvest at waypoints). That's the "stop to feed" behavior you actually want.

The capacity bump is the quick fix. The leapfrog is the proper architectural solution. I'd do both — bump capacity so short marches (3-4 hexes) don't need stops, and use leapfrog for longer expeditions.

In [ ]:
# What's actually happening? Check the death log
deaths = sim.df[sim.df.event == 'died']
print("=== DEATHS ===")
print(deaths[['turn', 'piece_name', 'src_hex']].to_string(index=False))

print(f"\n=== Food tier at destination 236: {int(food_tiers[236])}")
print("=== Nearby food tiers:")
for nb in myStuff.grid.neighborsOf(236):
    print(f"  hex {nb}: tier {int(food_tiers[nb])}")

# Were they harvesting at all?
harvests = sim.df[sim.df.event == 'harvested']
print(f"\n=== Total harvest events: {len(harvests)}")
print(harvests.groupby('piece_name')['amount'].sum().to_string())

# And gives?
gives = sim.df[sim.df.event == 'gave']
print(f"\n=== Total give events: {len(gives)}")
if not gives.empty:
    print(gives.groupby('piece_name')['amount'].sum().to_string())


Not enough food?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Exactly — hex 236 is a **food desert**. Tier 2 destination with tier 0-1 neighbors. And the queen dies on **turn 2** — she arrives first and starves before anyone can feed her!

Look at the harvest totals — the pawns gathered pitiful amounts (6-14 food over the whole sim). With tier 1-2 tiles, `harvest_strength × tier` gives almost nothing per turn.

Two problems to solve:

1. **Queen shouldn't go first to a barren tile** — she should arrive *after* feeders are set up, or they should all arrive together (which `sync=True` does, but she still can't self-sustain at tier 2)

2. **The destination itself is hostile** — no amount of clever facing fixes a tier-2 hex. The squad needs to stop at food-rich waypoints along the route.

This is exactly what `leapfrog_plan` was designed for. Let's check what it thinks:

```python
demoTerr()
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()
food_tiers = fy.tiers

route = myStuff.queen.pathfind(236, myStuff.grid, myStuff.terr.elevations,
                                myStuff.terr.fields.get('country'))

print(f"Route length: {len(route)} hexes")
print(f"Food tiers along route:")
for i, h in enumerate(route.hexes):
    print(f"  step {i:2d}: hex {h:3d} → tier {int(food_tiers[h])}")
```

This will show us whether there are *any* fertile tiles (tier 4+) along the path where the squad could camp and resupply. If the whole corridor is barren, hex 236 is simply unreachable without bringing food — like marching across a desert without supply wagons! 🏜️

In [ ]:
demoTerr()
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()
food_tiers = fy.tiers

route = myStuff.queen.pathfind(236, myStuff.grid, myStuff.terr.elevations,
                                myStuff.terr.fields.get('country'))

print(f"Route length: {len(route)} hexes")
print(f"Food tiers along route:")
for i, h in enumerate(route.hexes):
    print(f"  step {i:2d}: hex {h:3d} → tier {int(food_tiers[h])}")


In [ ]:
??FoodOverlay

So a key strategic question is where can you go? and what would be a stable position. what I am thinking about for an overlay. is how many pawns it would take to support a queen there. then we should only try to do paths along those hexes.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

That's a brilliant strategic concept — a **sustainability cost map**. Here's how I'd model it:

**Per hex, how many pawns to sustain a queen?**

- A pawn cycling HARVEST/GIVE gets ~0.5 harvests/turn (budget alternates)
- Net exportable food per pawn per turn: `0.5 × harvest_strength × best_ring1_tier - pawn_diet`
- Pawns needed: `ceil(queen_diet / net_exportable)`
- If `net_exportable ≤ 0` → **unsustainable** (no number of pawns works)

The overlay would look at each hex as a potential camp site, checking the **best food tier in ring-1** (since pawns sit around the queen, not on her hex):

```python
def sustain_cost(hex_idx, grid, food_tiers, elevations,
                 queen_diet=1.9, pawn_diet=1.9, pawn_harvest=2):
    """How many pawns needed to sustain a queen camped at hex_idx.
    Returns int (1-99) or -1 for unsustainable.
    """
    # Best food tier in ring-1 (where pawns would sit)
    best_tier = 0
    for nb in grid.neighborsOf(hex_idx):
        if 0 <= nb < len(food_tiers) and elevations[nb] > 0:
            best_tier = max(best_tier, int(food_tiers[nb]))
    
    # Net food a pawn can export per turn (harvest every other turn)
    net = 0.5 * pawn_harvest * best_tier - pawn_diet
    
    if net <= 0:
        return -1  # unsustainable
    
    return math.ceil(queen_diet / net)
```

Then the overlay paints each hex by cost:

```python
def SustainOverlay(queen, pawn_template=None, **kw) -> OverlaySpec:
    """Color hexes by number of pawns needed to sustain the queen there.
    
    Green (1-2), Yellow (3-4), Orange (5-6), Red (7+), Gray (impossible).
    """
    COLORS = {
        1: '#27ae60', 2: '#2ecc71',  # great
        3: '#f1c40f', 4: '#f39c12',  # expensive
        5: '#e67e22', 6: '#d35400',  # costly
    }
    DEFAULT = '#e74c3c'  # 7+
    IMPOSSIBLE = '#444444'
    
    def render(ctx):
        grid = ctx.grid
        fy = FoodYield(ctx.terrain, ctx.basins)
        fy.compute()
        food_tiers = fy.tiers
        elevs = ctx.terrain.elevations
        
        parts = []
        for i in range(len(elevs)):
            if elevs[i] <= 0:
                continue
            cost = sustain_cost(i, grid, food_tiers, elevs,
                                queen_diet=queen.diet)
            if cost == -1:
                color = IMPOSSIBLE
            else:
                color = COLORS.get(cost, DEFAULT)
            
            h = grid.hexes[i]
            pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in h.vertices())
            parts.append(
                f'<polygon points="{pts}" fill="{color}" opacity="0.3"/>')
        
        return '\n'.join(parts)
    
    return OverlaySpec("sustain_cost", render, priority=45)
```

For **pathfinding**, you'd modify `_move_cost` to treat unsustainable hexes as impassable (or very expensive):

```python
def _sustain_move_cost(elevations, food_tiers, grid, from_idx, to_idx, max_pawns=5):
    """Movement cost that penalizes food deserts."""
    base = _move_cost(elevations, from_idx, to_idx)
    cost = sustain_cost(to_idx, grid, food_tiers, elevations)
    if cost == -1 or cost > max_pawns:
        return base * 10  # heavy penalty, not infinite (allow passing through)
    return base
```

This way the squad naturally routes through the green/yellow corridor. Want me to write this up as a proper class you can drop into your overlay system?

In [ ]:
TerrainDisplay(
    TerrainOverlay(),
    SettlementOverlay(),
    SustainOverlay(myStuff.queen),
    RiverOverlay(max_width=5),
    terrain=myStuff.terr,
    board=myStuff.board,
    basins=myStuff.basin,
    debug=not showDemo
    
    )

So a couple of things come clear. We should have a better placement spiral. The queen should be put on a green hex.
We also need to figure out if a game is *playable* that is to say there must be some path between settlements of high enough levels. I do wonder if we need to adjust our food levels to a certain percentage to make a random board be playable.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Really sharp observations — you're essentially defining the **strategic topology** of the game. Two distinct problems:

**1. Smarter placement spiral**

The queen should anchor on the best `sustain_cost` hex, then pawns fill around her on the fertile ring:

```python
def smart_placement(settlement, squad, terrain, food_tiers, grid, max_ring=10):
    """Place queen on best sustainable hex, pawns on fertile neighbors."""
    center = settlement.location
    elevs = terrain.elevations
    
    # Score all candidate hexes by sustain_cost
    queen = squad.by_rank()[0]
    spiral = HexPosition.origin().spiral(max_ring)
    
    # Find best hex for the queen (lowest sustain_cost > 0)
    best_idx, best_cost = center, 99
    for hp in spiral:
        idx = grid.hexposition_to_index(hp, origin_index=center)
        if idx < 0 or elevs[idx] <= 0:
            continue
        cost = sustain_cost(idx, grid, food_tiers, elevs, queen_diet=queen.diet)
        if 0 < cost < best_cost:
            best_cost = cost
            best_idx = idx
    
    queen.location = best_idx
    
    # Place pawns on fertile ring-1 hexes around queen, sorted by food tier
    occupied = {best_idx}
    pawns = [p for p in squad.pieces if p.id != queen.id]
    
    neighbors = []
    for hp in HexPosition.origin().spiral(3):
        idx = grid.hexposition_to_index(hp, origin_index=best_idx)
        if idx < 0 or elevs[idx] <= 0 or idx in occupied:
            continue
        neighbors.append((int(food_tiers[idx]), idx))
    neighbors.sort(reverse=True)  # best food first
    
    placed = 0
    for tier, idx in neighbors:
        if placed >= len(pawns):
            break
        pawns[placed].location = idx
        pawns[placed].facing = _facing_toward(grid, idx, best_idx)  # face queen!
        occupied.add(idx)
        placed += 1
    
    return best_cost
```

**2. Playability validation**

This is the bigger design question. I'd approach it in layers:

```python
def playability_check(terrain, basins, grid, settlements, max_pawns=5):
    """Check if all settlements are connected via sustainable corridors.
    
    Returns dict with:
      - connected: bool
      - coverage: float (fraction of land that's sustainable)
      - corridors: dict of settlement pairs → path exists
      - suggestion: str (what to tweak if not playable)
    """
    fy = FoodYield(terrain, basins); fy.compute()
    food_tiers = fy.tiers
    elevs = terrain.elevations
    
    # Build sustainable hex set
    sustainable = set()
    for i in range(len(elevs)):
        if elevs[i] <= 0:
            continue
        cost = sustain_cost(i, grid, food_tiers, elevs)
        if 0 < cost <= max_pawns:
            sustainable.add(i)
    
    land = sum(1 for e in elevs if e > 0)
    coverage = len(sustainable) / max(land, 1)
    
    # BFS connectivity between settlements using only sustainable hexes
    def bfs_connected(start, end):
        if start not in sustainable or end not in sustainable:
            return False
        visited = {start}
        queue = [start]
        while queue:
            current = queue.pop(0)
            if current == end:
                return True
            for nb in grid.neighborsOf(current):
                if nb in sustainable and nb not in visited:
                    visited.add(nb)
                    queue.append(nb)
        return False
    
    corridors = {}
    locs = [s.location for s in settlements]
    all_connected = True
    for i in range(len(locs)):
        for j in range(i+1, len(locs)):
            ok = bfs_connected(locs[i], locs[j])
            corridors[(i,j)] = ok
            if not ok:
                all_connected = False
    
    return {
        'connected': all_connected,
        'coverage': coverage,
        'corridors': corridors,
        'sustainable_count': len(sustainable),
        'land_count': land,
    }
```

**3. The calibration question**

I think you're right — rather than hoping random terrain works, **target a coverage percentage**. Something like:

- **Minimum 30% sustainable** for a playable map
- **40-60% sustainable** for balanced gameplay
- **70%+** makes food trivial (boring)

The lever is the `FoodYield` params. You could auto-calibrate after generation:

```python
def calibrate_food(terrain, basins, grid, target_coverage=0.45, max_iters=10):
    """Adjust soil_mult scaling until sustainable coverage hits target."""
    scale = 1.0
    for _ in range(max_iters):
        fy = FoodYield(terrain, basins)
        fy.soil_mult = [s * scale for s in fy.soil_mult]
        fy.compute()
        
        check = playability_check(terrain, basins, grid, [], max_pawns=5)
        coverage = check['coverage']
        
        if abs(coverage - target_coverage) < 0.05:
            return scale, coverage
        
        # Binary search direction
        if coverage < target_coverage:
            scale *= 1.3
        else:
            scale *= 0.8
    
    return scale, coverage
```

Want to start with the `playability_check` to see what your current map looks like? That'll tell us whether we need calibration or if the terrain params just need a nudge.

In [ ]:
def playability_check(terrain, basins, grid, settlements, max_pawns=5):
    """Check if all settlements are connected via sustainable corridors.
    
    Returns dict with:
      - connected: bool
      - coverage: float (fraction of land that's sustainable)
      - corridors: dict of settlement pairs → path exists
      - suggestion: str (what to tweak if not playable)
    """
    fy = FoodYield(terrain, basins); fy.compute()
    food_tiers = fy.tiers
    elevs = terrain.elevations
    
    # Build sustainable hex set
    sustainable = set()
    for i in range(len(elevs)):
        if elevs[i] <= 0:
            continue
        cost = sustain_cost(i, grid, food_tiers, elevs)
        if 0 < cost <= max_pawns:
            sustainable.add(i)
    
    land = sum(1 for e in elevs if e > 0)
    coverage = len(sustainable) / max(land, 1)
    
    # BFS connectivity between settlements using only sustainable hexes
    def bfs_connected(start, end):
        if start not in sustainable or end not in sustainable:
            return False
        visited = {start}
        queue = [start]
        while queue:
            current = queue.pop(0)
            if current == end:
                return True
            for nb in grid.neighborsOf(current):
                if nb in sustainable and nb not in visited:
                    visited.add(nb)
                    queue.append(nb)
        return False
    
    corridors = {}
    locs = [s.location for s in settlements]
    all_connected = True
    for i in range(len(locs)):
        for j in range(i+1, len(locs)):
            ok = bfs_connected(locs[i], locs[j])
            corridors[(i,j)] = ok
            if not ok:
                all_connected = False
    
    return {
        'connected': all_connected,
        'coverage': coverage,
        'corridors': corridors,
        'sustainable_count': len(sustainable),
        'land_count': land,
    }


can you make playability_check an @patch on gameboard.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->



```python
@patch
def playability_check(self: GameBoard, max_pawns=5):
    """Check if all settlements are connected via sustainable corridors.
    
    Returns dict with:
      - connected: bool
      - coverage: float (fraction of land that's sustainable)
      - corridors: dict of settlement pairs → path exists
      - sustainable_count / land_count: raw numbers
    """
    fy = FoodYield(self.terrain, self.basins); fy.compute()
    food_tiers = fy.tiers
    elevs = self.terrain.elevations
    grid = self.grid
    
    # Build sustainable hex set
    sustainable = set()
    for i in range(len(elevs)):
        if elevs[i] <= 0:
            continue
        cost = sustain_cost(i, grid, food_tiers, elevs)
        if 0 < cost <= max_pawns:
            sustainable.add(i)
    
    land = sum(1 for e in elevs if e > 0)
    coverage = len(sustainable) / max(land, 1)
    
    # BFS connectivity between settlements using only sustainable hexes
    def bfs_connected(start, end):
        if start not in sustainable or end not in sustainable:
            return False
        visited = {start}
        queue = [start]
        while queue:
            current = queue.pop(0)
            if current == end:
                return True
            for nb in grid.neighborsOf(current):
                if nb in sustainable and nb not in visited:
                    visited.add(nb)
                    queue.append(nb)
        return False
    
    settlements = self.settlements
    corridors = {}
    locs = [s.location for s in settlements]
    all_connected = True
    for i in range(len(locs)):
        for j in range(i+1, len(locs)):
            ok = bfs_connected(locs[i], locs[j])
            corridors[(i, j)] = ok
            if not ok:
                all_connected = False
    
    return {
        'connected': all_connected,
        'coverage': coverage,
        'corridors': corridors,
        'sustainable_count': len(sustainable),
        'land_count': land,
    }
```

This assumes `GameBoard` has `.terrain`, `.basins`, `.grid`, and `.settlements` attributes. Try it with:

```python
result = myStuff.board.playability_check()
print(f"Connected: {result['connected']}")
print(f"Coverage:  {result['coverage']:.1%} ({result['sustainable_count']}/{result['land_count']} hexes)")
for (i, j), ok in result['corridors'].items():
    print(f"  Settlement {i} ↔ {j}: {'✅' if ok else '❌'}")
```

In [ ]:
settlements = [country.settlements[0] for country in myStuff.board.kingdoms]

playability_check(myStuff.terr, myStuff.basin, myStuff.grid, settlements)

Not a good map

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

49% coverage is actually decent — the problem is it's **fragmented**. The green zones don't form a connected corridor between settlements. It's like two oases with a desert between them.

Two approaches:

**Quick diagnostic — where's the gap?**

```python
# Visualize the sustainable vs unsustainable hexes
# and see where the corridor breaks

fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()
food_tiers = fy.tiers
elevs = myStuff.terr.elevations
grid = myStuff.grid

# What's the max sustain_cost we'd need to connect them?
for threshold in range(1, 20):
    sustainable = set()
    for i in range(len(elevs)):
        if elevs[i] <= 0: continue
        cost = sustain_cost(i, grid, food_tiers, elevs)
        if 0 < cost <= threshold:
            sustainable.add(i)
    
    # Quick BFS
    s0, s1 = settlements[0].location, settlements[1].location
    if s0 not in sustainable or s1 not in sustainable:
        print(f"  max_pawns={threshold:2d}: {len(sustainable):3d} hexes, settlements not sustainable")
        continue
    visited = {s0}
    queue = [s0]
    while queue:
        c = queue.pop(0)
        if c == s1: break
        for nb in grid.neighborsOf(c):
            if nb in sustainable and nb not in visited:
                visited.add(nb); queue.append(nb)
    connected = s1 in visited
    print(f"  max_pawns={threshold:2d}: {len(sustainable):3d} hexes, connected={connected}")
    if connected:
        print(f"  → Need {threshold} pawns to be willing to spend for a viable corridor")
        break
```

This tells you the **minimum squad size** needed to make the map work. If it never connects even at 20 pawns, the map is genuinely unplayable.

**Proper fix — calibrate until connected:**

```python
def calibrate_food(terrain, basins, grid, settlements,
                   target_coverage=0.50, max_pawns=5, max_iters=12):
    """Binary-search soil_mult scaling until settlements are connected."""
    lo, hi = 0.5, 5.0
    best_scale, best_result = 1.0, None
    
    for i in range(max_iters):
        scale = (lo + hi) / 2
        fy = FoodYield(terrain, basins)
        fy.soil_mult = [s * scale for s in fy.soil_mult]
        fy.compute()
        
        check = playability_check(terrain, basins, grid, settlements, max_pawns)
        check['scale'] = scale
        print(f"  iter {i}: scale={scale:.2f}, coverage={check['coverage']:.1%}, connected={check['connected']}")
        
        if check['connected']:
            best_scale = scale
            best_result = check
            hi = scale  # try less boost
        else:
            lo = scale  # need more boost
    
    return best_scale, best_result
```

I'd run the diagnostic first — if it connects at `max_pawns=8`, you might just want a slightly bigger squad rather than retuning the whole world. But if it never connects, the map needs calibration or regeneration.

In [ ]:
# Visualize the sustainable vs unsustainable hexes
# and see where the corridor breaks

fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()
food_tiers = fy.tiers
elevs = myStuff.terr.elevations
grid = myStuff.grid

# What's the max sustain_cost we'd need to connect them?
for threshold in range(1, 20):
    sustainable = set()
    for i in range(len(elevs)):
        if elevs[i] <= 0: continue
        cost = sustain_cost(i, grid, food_tiers, elevs)
        if 0 < cost <= threshold:
            sustainable.add(i)
    
    # Quick BFS
    s0, s1 = settlements[0].location, settlements[1].location
    if s0 not in sustainable or s1 not in sustainable:
        print(f"  max_pawns={threshold:2d}: {len(sustainable):3d} hexes, settlements not sustainable")
        continue
    visited = {s0}
    queue = [s0]
    while queue:
        c = queue.pop(0)
        if c == s1: break
        for nb in grid.neighborsOf(c):
            if nb in sustainable and nb not in visited:
                visited.add(nb); queue.append(nb)
    connected = s1 in visited
    print(f"  max_pawns={threshold:2d}: {len(sustainable):3d} hexes, connected={connected}")
    if connected:
        print(f"  → Need {threshold} pawns to be willing to spend for a viable corridor")
        break


The otherway of balancing is making the queen less hungry I guess

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Exactly — there are two sides to the equation. Let's see what queen diet makes this specific map playable:

```python
# What queen diet connects the map at max_pawns=5?
for diet in [1.9, 1.5, 1.2, 1.0, 0.8, 0.5, 0.3]:
    sustainable = set()
    for i in range(len(elevs)):
        if elevs[i] <= 0: continue
        cost = sustain_cost(i, grid, food_tiers, elevs, queen_diet=diet)
        if 0 < cost <= 5:
            sustainable.add(i)
    
    s0, s1 = settlements[0].location, settlements[1].location
    if s0 not in sustainable or s1 not in sustainable:
        print(f"  diet={diet:.1f}: {len(sustainable):3d} hexes, settlement not in range")
        continue
    visited = {s0}
    queue = [s0]
    while queue:
        c = queue.pop(0)
        if c == s1: break
        for nb in grid.neighborsOf(c):
            if nb in sustainable and nb not in visited:
                visited.add(nb); queue.append(nb)
    connected = s1 in visited
    print(f"  diet={diet:.1f}: {len(sustainable):3d} hexes, connected={connected}")
```

But honestly, there are **four knobs** you could turn — and they all multiply together:

| Knob | Effect | Gameplay feel |
|------|--------|---------------|
| ↓ Queen diet | More hexes viable | Queen feels lighter, less dramatic |
| ↑ Pawn harvest | More food per turn | Pawns feel productive |
| ↑ Food tiers (soil_mult) | Richer land | Lush world |
| ↓ Pawn diet | More net export | Pawns self-sustain easier |

Rather than picking one, I'd think about it as a **ratio**: `pawn_net_export / queen_diet`. If that ratio is too low, you need 19 pawns. The current math is:

- net per pawn ≈ `0.5 × 2 × best_tier - 1.9`
- At tier 3: `3.0 - 1.9 = 1.1` → need 2 pawns ✅
- At tier 2: `2.0 - 1.9 = 0.1` → need **19 pawns** 😱
- At tier 1: `1.0 - 1.9 = -0.9` → **impossible**

That tier 2→3 cliff is the killer. The bottleneck hexes in the corridor are tier 2, so the whole map breaks. A tiny bump — either making tier 2 yield slightly more, or reducing pawn diet to 1.5 — would collapse those 19-pawn hexes down to 3-4 pawns and the map connects.

Run the diagnostic and see where the sweet spot is?

I feel we have a pyomo/ optymo balancing problem

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

100%. You've got Optuna and Pyomo both imported already — this is a textbook multi-objective black-box optimization:

**Optuna outer loop** (search game params across random maps):

```python
def balance_objective(trial):
    """Optuna objective: find params where most random maps are playable."""
    # Decision variables
    queen_diet    = trial.suggest_float('queen_diet', 0.5, 3.0)
    pawn_diet     = trial.suggest_float('pawn_diet', 0.5, 2.5)
    pawn_harvest  = trial.suggest_float('pawn_harvest', 1.0, 5.0)
    soil_scale    = trial.suggest_float('soil_scale', 0.5, 4.0)
    
    n_maps = 8
    scores = []
    
    for seed in range(n_maps):
        # Generate random terrain
        terr = TerraDemo(seed=seed).japan_korea_map()
        grid = terr.hexGrid
        terr.carve_to_ocean(num_lakes=0)
        basin = DrainageBasins(terr)
        board = GameBoard(terr, 2)
        
        # Apply soil scaling
        fy = FoodYield(terr, basin)
        fy.soil_mult = [s * soil_scale for s in fy.soil_mult]
        fy.compute()
        food_tiers = fy.tiers
        elevs = terr.elevations
        
        settlements = [k.settlements[0] for k in board.kingdoms]
        
        # Coverage
        sustainable = set()
        for i in range(len(elevs)):
            if elevs[i] <= 0: continue
            cost = sustain_cost(i, grid, food_tiers, elevs,
                                queen_diet=queen_diet,
                                pawn_diet=pawn_diet,
                                pawn_harvest=pawn_harvest)
            if 0 < cost <= 5:
                sustainable.add(i)
        
        land = max(1, sum(1 for e in elevs if e > 0))
        coverage = len(sustainable) / land
        
        # Connectivity
        s0, s1 = settlements[0].location, settlements[1].location
        connected = False
        if s0 in sustainable and s1 in sustainable:
            visited = {s0}
            queue = [s0]
            while queue:
                c = queue.pop(0)
                if c == s1:
                    connected = True; break
                for nb in grid.neighborsOf(c):
                    if nb in sustainable and nb not in visited:
                        visited.add(nb); queue.append(nb)
        
        # Score this map
        map_score = 0.0
        if connected: map_score += 1.0
        
        # Bonus for coverage in sweet spot (35-55%)
        if 0.35 <= coverage <= 0.55:
            map_score += 0.5
        elif 0.25 <= coverage <= 0.65:
            map_score += 0.2
        
        # Penalty: queen should need ≥ 1 pawn (not self-sustaining)
        if queen_diet <= pawn_diet * 0.3:
            map_score -= 0.5  # too easy
        
        scores.append(map_score)
    
    return sum(scores) / len(scores)

study = optuna.create_study(direction='maximize')
study.optimize(balance_objective, n_trials=100)

print(f"\nBest params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v:.2f}")
print(f"Score: {study.best_value:.2f}")
```

The beauty is you can add more objectives to the score:

- **Strategic tension**: queen needs 2-3 pawns (not 1, not 10)
- **Corridor narrowness**: paths exist but aren't everywhere (chokepoints!)
- **Asymmetry**: some maps should slightly favor one side

And if you want to go full Pyomo for a **single map** — "given this terrain, what's the minimum diet adjustment to connect settlements?":

```python
def min_diet_adjustment(terrain, basins, grid, settlements):
    """Pyomo: smallest diet reduction that connects settlements."""
    m = pyo.ConcreteModel()
    m.diet_scale = pyo.Var(bounds=(0.1, 1.0))  # multiplier on current diet
    
    # ... but this is hard to linearize since connectivity is graph-theoretic
```

That's where it gets tricky — connectivity is inherently combinatorial. Optuna is the right tool for the outer loop. Pyomo shines for the **inner** problems you already use it for (feed_cover assignment, Hungarian matching).

Want to run the Optuna search and see what balance emerges?

So the very similar model I want to run is based upon the terro demo world that a queen and 5 pawns could traverse between the settlements or a queen and bishop could do it. but fewer pawns or no bishop wouldn't work.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

That's a beautifully crisp game design constraint. The bishop becomes strategically valuable — it's the "supply officer" that unlocks routes a pawn army can't manage alone. Here's the Optuna model:

```python
def traversal_check(terrain, basins, grid, settlements,
                    queen_diet, feeder_diets, feeder_harvests, soil_scale=1.0):
    """Can this squad composition traverse between settlements?
    
    feeder_diets/feeder_harvests: lists, one per feeder in the squad.
    Returns (connected, coverage, sustainable_set).
    """
    fy = FoodYield(terrain, basins)
    fy.soil_mult = [s * soil_scale for s in fy.soil_mult]
    fy.compute()
    food_tiers = fy.tiers
    elevs = terrain.elevations
    
    n_feeders = len(feeder_diets)
    
    # Per hex: can this squad sustain here?
    sustainable = set()
    for i in range(len(elevs)):
        if elevs[i] <= 0:
            continue
        # Best food tier in ring-1
        best_tier = 0
        for nb in grid.neighborsOf(i):
            if 0 <= nb < len(food_tiers) and elevs[nb] > 0:
                best_tier = max(best_tier, int(food_tiers[nb]))
        
        # Total net export from all feeders
        total_net = sum(
            0.5 * h * best_tier - d
            for h, d in zip(feeder_harvests, feeder_diets)
        )
        
        if total_net >= queen_diet:
            sustainable.add(i)
    
    land = max(1, sum(1 for e in elevs if e > 0))
    coverage = len(sustainable) / land
    
    # BFS connectivity
    s0, s1 = settlements[0].location, settlements[1].location
    connected = False
    if s0 in sustainable and s1 in sustainable:
        visited = {s0}
        queue = [s0]
        while queue:
            c = queue.pop(0)
            if c == s1:
                connected = True; break
            for nb in grid.neighborsOf(c):
                if nb in sustainable and nb not in visited:
                    visited.add(nb); queue.append(nb)
    
    return connected, coverage, sustainable


def balance_objective(trial):
    """Find params where queen+5pawns OR queen+bishop works,
    but queen+4pawns or queen+pawns(no bishop) doesn't."""
    
    queen_diet     = trial.suggest_float('queen_diet', 0.8, 3.0)
    pawn_diet      = trial.suggest_float('pawn_diet', 0.5, 2.5)
    pawn_harvest   = trial.suggest_float('pawn_harvest', 1.0, 4.0)
    bishop_diet    = trial.suggest_float('bishop_diet', 0.5, 3.0)
    bishop_harvest = trial.suggest_float('bishop_harvest', 2.0, 8.0)
    soil_scale     = trial.suggest_float('soil_scale', 0.5, 4.0)
    
    n_maps = 6
    score = 0.0
    
    for seed in range(n_maps):
        terr = TerraDemo(seed=seed).japan_korea_map()
        grid = terr.hexGrid
        terr.carve_to_ocean(num_lakes=0)
        basin = DrainageBasins(terr)
        board = GameBoard(terr, 2)
        setts = [k.settlements[0] for k in board.kingdoms]
        
        # Squad compositions to test
        # (label, feeder_diets, feeder_harvests, should_connect)
        scenarios = [
            ("queen+5pawns",
             [pawn_diet]*5, [pawn_harvest]*5, True),
            ("queen+4pawns",
             [pawn_diet]*4, [pawn_harvest]*4, False),
            ("queen+bishop",
             [bishop_diet], [bishop_harvest], True),
            ("queen+3pawns",
             [pawn_diet]*3, [pawn_harvest]*3, False),
            ("queen+1pawn",
             [pawn_diet]*1, [pawn_harvest]*1, False),
        ]
        
        for label, diets, harvests, should in scenarios:
            connected, coverage, _ = traversal_check(
                terr, basin, grid, setts,
                queen_diet, diets, harvests, soil_scale)
            
            if connected == should:
                score += 1.0   # correct!
            else:
                score -= 1.5   # wrong — penalize harder
        
        # Bonus: coverage in interesting range (30-55%)
        _, cov5, _ = traversal_check(
            terr, basin, grid, setts,
            queen_diet, [pawn_diet]*5, [pawn_harvest]*5, soil_scale)
        if 0.30 <= cov5 <= 0.55:
            score += 0.5  # strategic corridors, not open plains
    
    return score / n_maps


optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize')
study.optimize(balance_objective, n_trials=80, show_progress_bar=True)

print(f"\n{'='*50}")
print(f"Best score: {study.best_value:.2f}")
print(f"\nBalanced params:")
for k, v in study.best_params.items():
    print(f"  {k:20s}: {v:.3f}")

# Interpret the result
bp = study.best_params
print(f"\nGameplay implications:")
print(f"  Bishop harvest is {bp['bishop_harvest']/bp['pawn_harvest']:.1f}× a pawn's")
print(f"  Bishop diet is {bp['bishop_diet']/bp['pawn_diet']:.1f}× a pawn's")
print(f"  Queen diet is {bp['queen_diet']/bp['pawn_diet']:.1f}× a pawn's")
```

The key insight in the scoring: we **reward** correct outcomes and **penalize** wrong ones asymmetrically (1.0 vs -1.5), so the optimizer strongly avoids params where the wrong squad succeeds or the right one fails.

The five scenarios encode your exact design intent:
- ✅ 5 pawns: the "brute force" army approach
- ❌ 4 pawns: just barely not enough — tension!
- ✅ 1 bishop: the "specialist" approach — bishops are worth it
- ❌ 3 pawns: clearly insufficient
- ❌ 1 pawn: hopeless

This should converge in ~80 trials to the sweet spot where bishops feel like strategic assets, not luxury pieces. 🎯

In [ ]:
import pyomo.environ as pyo
import optuna


In [ ]:
world = TerrainFactory.create_world(
        bounds= MapRect(MapCord(0, 0), MapSize(300, 300)),
        preset='temperate',
        name='Maiden Lane',
        radius=15,
        lon_span=10.0,
        num_plates=8,
        subdivisions=3,
        ocean_fraction=0.3,
        oceanic_sides=['N'],  # Ocean on east and west
        terrain_age='young',  # Sharp, dramatic features
        formation_type='ridge',  # Creates ridge formations
        elevation_scale=1.5,  # Exaggerate the heights
        erosion_age=0.1,  # Minimal erosion for sharp peaks
        num_lakes=0,
        seed=23,
        debug=True
    )

sampleMap = world.terrain
sampleMap.repalette()
myStuff.loadGeology(world)
myStuff.grid.adjustRadius(40)

terr = world.terrain
grid = terr.hexGrid
terr.carve_to_ocean(num_lakes=0)
basin = DrainageBasins(terr)
board = GameBoard(terr, 2)
setts = [k.settlements[0] for k in board.kingdoms]

def traversal_check(terrain, basins, grid, settlements,
                    queen_diet, feeder_diets, feeder_harvests, soil_scale=1.0):
    """Can this squad composition traverse between settlements?
    
    feeder_diets/feeder_harvests: lists, one per feeder in the squad.
    Returns (connected, coverage, sustainable_set).
    """
    fy = FoodYield(terrain, basins)
    fy.soil_mult = [s * soil_scale for s in fy.soil_mult]
    fy.compute()
    food_tiers = fy.tiers
    elevs = terrain.elevations
    
    n_feeders = len(feeder_diets)
    
    # Per hex: can this squad sustain here?
    sustainable = set()
    for i in range(len(elevs)):
        if elevs[i] <= 0:
            continue
        # Best food tier in ring-1
        best_tier = 0
        for nb in grid.neighborsOf(i):
            if 0 <= nb < len(food_tiers) and elevs[nb] > 0:
                best_tier = max(best_tier, int(food_tiers[nb]))
        
        # Total net export from all feeders
        total_net = sum(
            0.5 * h * best_tier - d
            for h, d in zip(feeder_harvests, feeder_diets)
        )
        
        if total_net >= queen_diet:
            sustainable.add(i)
    
    land = max(1, sum(1 for e in elevs if e > 0))
    coverage = len(sustainable) / land
    
    # BFS connectivity
    s0, s1 = settlements[0].location, settlements[1].location
    connected = False
    if s0 in sustainable and s1 in sustainable:
        visited = {s0}
        queue = [s0]
        while queue:
            c = queue.pop(0)
            if c == s1:
                connected = True; break
            for nb in grid.neighborsOf(c):
                if nb in sustainable and nb not in visited:
                    visited.add(nb); queue.append(nb)
    
    return connected, coverage, sustainable


def balance_objective(trial):
    """Find params where queen+5pawns OR queen+bishop works,
    but queen+4pawns or queen+pawns(no bishop) doesn't."""
    
    queen_diet     = trial.suggest_float('queen_diet', 0.8, 3.0)
    pawn_diet      = trial.suggest_float('pawn_diet', 0.5, 2.5)
    pawn_harvest   = trial.suggest_float('pawn_harvest', 1.0, 4.0)
    bishop_diet    = trial.suggest_float('bishop_diet', 0.5, 3.0)
    bishop_harvest = trial.suggest_float('bishop_harvest', 2.0, 8.0)
    soil_scale     = trial.suggest_float('soil_scale', 0.5, 4.0)
    
    n_maps = 6
    score = 0.0
    
    for seed in range(n_maps):

        
        
        # Squad compositions to test
        # (label, feeder_diets, feeder_harvests, should_connect)
        scenarios = [
            ("queen+5pawns",
             [pawn_diet]*5, [pawn_harvest]*5, True),
            ("queen+4pawns",
             [pawn_diet]*4, [pawn_harvest]*4, False),
            ("queen+bishop",
             [bishop_diet], [bishop_harvest], True),
            ("queen+3pawns",
             [pawn_diet]*3, [pawn_harvest]*3, False),
            ("queen+1pawn",
             [pawn_diet]*1, [pawn_harvest]*1, False),
        ]
        
        for label, diets, harvests, should in scenarios:
            connected, coverage, _ = traversal_check(
                terr, basin, grid, setts,
                queen_diet, diets, harvests, soil_scale)
            
            if connected == should:
                score += 1.0   # correct!
            else:
                score -= 1.5   # wrong — penalize harder
        
        # Bonus: coverage in interesting range (30-55%)
        _, cov5, _ = traversal_check(
            terr, basin, grid, setts,
            queen_diet, [pawn_diet]*5, [pawn_harvest]*5, soil_scale)
        if 0.30 <= cov5 <= 0.55:
            score += 0.5  # strategic corridors, not open plains
    
    return score / n_maps


optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize')
study.optimize(balance_objective, n_trials=80, show_progress_bar=True)

print(f"\n{'='*50}")
print(f"Best score: {study.best_value:.2f}")
print(f"\nBalanced params:")
for k, v in study.best_params.items():
    print(f"  {k:20s}: {v:.3f}")

# Interpret the result
bp = study.best_params
print(f"\nGameplay implications:")
print(f"  Bishop harvest is {bp['bishop_harvest']/bp['pawn_harvest']:.1f}× a pawn's")
print(f"  Bishop diet is {bp['bishop_diet']/bp['pawn_diet']:.1f}× a pawn's")
print(f"  Queen diet is {bp['queen_diet']/bp['pawn_diet']:.1f}× a pawn's")


We need to add some contraints. bishops need to be bad at harvesting. we need to put the food_capacity so that a queen can make it across the board being fed by the bishop

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great refinement! The bishop's value shouldn't be "harvests a lot" — it should be **"carries food across barren gaps"**. That's the food_capacity play. The model changes from "can you sustain at every hex?" to "can you refuel and carry enough to cross the desert between oases?"

```python
def traversal_check_v2(terrain, basins, grid, settlements,
                       queen_diet, feeder_diets, feeder_harvests,
                       feeder_capacities, soil_scale=1.0):
    """Connectivity check with carry capacity — squads can cross barren gaps."""
    fy = FoodYield(terrain, basins)
    fy.soil_mult = [s * soil_scale for s in fy.soil_mult]
    fy.compute()
    food_tiers = fy.tiers
    elevs = terrain.elevations
    
    # Total burn rate per hex when nobody harvests (crossing barren land)
    total_diet = queen_diet + sum(feeder_diets)
    
    # Max barren hexes the squad can cross on full reserves
    total_capacity = sum(feeder_capacities)
    max_gap = int(total_capacity / max(total_diet, 0.01))
    
    # Fertile = squad can net-refuel here (not just sustain — actually gain food)
    fertile = set()
    passable = set()
    for i in range(len(elevs)):
        if elevs[i] <= 0: continue
        passable.add(i)
        best_tier = 0
        for nb in grid.neighborsOf(i):
            if 0 <= nb < len(food_tiers) and elevs[nb] > 0:
                best_tier = max(best_tier, int(food_tiers[nb]))
        total_net = sum(0.5 * h * best_tier - d
                        for h, d in zip(feeder_harvests, feeder_diets))
        if total_net > 0:
            fertile.add(i)
    
    land = max(1, len(passable))
    coverage = len(fertile) / land
    
    # BFS with fuel: track remaining gap budget
    # State: (hex_idx, gap_remaining)
    s0, s1 = settlements[0].location, settlements[1].location
    if s0 not in passable or s1 not in passable:
        return False, coverage, fertile
    
    # BFS: at fertile hexes, reset gap counter to max_gap
    from collections import deque
    visited = {}  # hex_idx → best gap_remaining seen
    queue = deque()
    start_gap = max_gap if s0 in fertile else 0
    queue.append((s0, start_gap))
    visited[s0] = start_gap
    connected = False
    
    while queue:
        current, fuel = queue.popleft()
        if current == s1:
            connected = True; break
        
        for nb in grid.neighborsOf(current):
            if nb not in passable:
                continue
            # Arriving at fertile hex? Refuel to max
            nb_fuel = max_gap if nb in fertile else fuel - 1
            if nb_fuel < 0:
                continue  # can't reach — out of food
            if nb in visited and visited[nb] >= nb_fuel:
                continue  # already reached with more fuel
            visited[nb] = nb_fuel
            queue.append((nb, nb_fuel))
    
    return connected, coverage, fertile


def balance_objective_v2(trial):
    """Bishop = low harvest, high capacity (supply wagon).
    Pawn = decent harvest, low capacity (field worker)."""
    
    queen_diet       = trial.suggest_float('queen_diet', 0.8, 3.0)
    pawn_diet        = trial.suggest_float('pawn_diet', 0.5, 2.5)
    pawn_harvest     = trial.suggest_float('pawn_harvest', 1.0, 4.0)
    pawn_capacity    = trial.suggest_float('pawn_capacity', 5, 30)
    bishop_diet      = trial.suggest_float('bishop_diet', 0.5, 3.0)
    bishop_harvest   = trial.suggest_float('bishop_harvest', 0.3, 2.0)  # BAD at harvest!
    bishop_capacity  = trial.suggest_float('bishop_capacity', 30, 200)  # BIG backpack
    soil_scale       = trial.suggest_float('soil_scale', 0.5, 4.0)
    
    score = 0.0
    
    scenarios = [
        # (label, feeder_diets, feeder_harvests, feeder_capacities, should_connect)
        ("queen+5pawns",
         [pawn_diet]*5, [pawn_harvest]*5, [pawn_capacity]*5, True),
        ("queen+4pawns",
         [pawn_diet]*4, [pawn_harvest]*4, [pawn_capacity]*4, False),
        ("queen+bishop",
         [bishop_diet], [bishop_harvest], [bishop_capacity], True),
        ("queen+3pawns",
         [pawn_diet]*3, [pawn_harvest]*3, [pawn_capacity]*3, False),
        ("queen+1pawn",
         [pawn_diet]*1, [pawn_harvest]*1, [pawn_capacity]*1, False),
        # Bishop without capacity advantage shouldn't work
        ("queen+bishop(small bag)",
         [bishop_diet], [bishop_harvest], [pawn_capacity], False),
    ]
    
    for label, diets, harvests, caps, should in scenarios:
        connected, coverage, _ = traversal_check_v2(
            terr, basin, grid, setts,
            queen_diet, diets, harvests, caps, soil_scale)
        
        if connected == should:
            score += 1.0
        else:
            score -= 1.5
    
    # Coverage sweet spot for the 5-pawn case
    _, cov5, _ = traversal_check_v2(
        terr, basin, grid, setts,
        queen_diet, [pawn_diet]*5, [pawn_harvest]*5,
        [pawn_capacity]*5, soil_scale)
    if 0.30 <= cov5 <= 0.55:
        score += 0.5
    
    return score


optuna.logging.set_verbosity(optuna.logging.WARNING)
study2 = optuna.create_study(direction='maximize')
study2.optimize(balance_objective_v2, n_trials=100, show_progress_bar=True)

bp = study2.best_params
print(f"\n{'='*50}")
print(f"Best score: {study2.best_value:.2f}")
print(f"\nBalanced params:")
for k, v in bp.items():
    print(f"  {k:20s}: {v:.3f}")

print(f"\nGameplay story:")
print(f"  Queen eats {bp['queen_diet']:.1f}/turn")
print(f"  Pawn: eats {bp['pawn_diet']:.1f}, harvests {bp['pawn_harvest']:.1f}, carries {bp['pawn_capacity']:.0f}")
print(f"  Bishop: eats {bp['bishop_diet']:.1f}, harvests {bp['bishop_harvest']:.1f}, carries {bp['bishop_capacity']:.0f}")
print(f"  Bishop carries {bp['bishop_capacity']/bp['pawn_capacity']:.1f}× a pawn's food")
print(f"  Bishop crosses ~{int(bp['bishop_capacity']/(bp['bishop_diet']+bp['queen_diet']))} barren hexes solo")
```

The 6th scenario is the sneaky one — "queen+bishop but with a pawn-sized backpack" should **fail**. This forces the optimizer to make the bishop's *capacity* the thing that matters, not some other stat. The bishop earns its keep as a supply wagon 🎒, not a farmer.

In [ ]:
def traversal_check_v2(terrain, basins, grid, settlements,
                       queen_diet, feeder_diets, feeder_harvests,
                       feeder_capacities, soil_scale=1.0):
    """Connectivity check with carry capacity — squads can cross barren gaps."""
    fy = FoodYield(terrain, basins)
    fy.soil_mult = [s * soil_scale for s in fy.soil_mult]
    fy.compute()
    food_tiers = fy.tiers
    elevs = terrain.elevations
    
    # Total burn rate per hex when nobody harvests (crossing barren land)
    total_diet = queen_diet + sum(feeder_diets)
    
    # Max barren hexes the squad can cross on full reserves
    total_capacity = sum(feeder_capacities)
    max_gap = int(total_capacity / max(total_diet, 0.01))
    
    # Fertile = squad can net-refuel here (not just sustain — actually gain food)
    fertile = set()
    passable = set()
    for i in range(len(elevs)):
        if elevs[i] <= 0: continue
        passable.add(i)
        best_tier = 0
        for nb in grid.neighborsOf(i):
            if 0 <= nb < len(food_tiers) and elevs[nb] > 0:
                best_tier = max(best_tier, int(food_tiers[nb]))
        total_net = sum(0.5 * h * best_tier - d
                        for h, d in zip(feeder_harvests, feeder_diets))
        if total_net > 0:
            fertile.add(i)
    
    land = max(1, len(passable))
    coverage = len(fertile) / land
    
    # BFS with fuel: track remaining gap budget
    # State: (hex_idx, gap_remaining)
    s0, s1 = settlements[0].location, settlements[1].location
    if s0 not in passable or s1 not in passable:
        return False, coverage, fertile
    
    # BFS: at fertile hexes, reset gap counter to max_gap
    from collections import deque
    visited = {}  # hex_idx → best gap_remaining seen
    queue = deque()
    start_gap = max_gap if s0 in fertile else 0
    queue.append((s0, start_gap))
    visited[s0] = start_gap
    connected = False
    
    while queue:
        current, fuel = queue.popleft()
        if current == s1:
            connected = True; break
        
        for nb in grid.neighborsOf(current):
            if nb not in passable:
                continue
            # Arriving at fertile hex? Refuel to max
            nb_fuel = max_gap if nb in fertile else fuel - 1
            if nb_fuel < 0:
                continue  # can't reach — out of food
            if nb in visited and visited[nb] >= nb_fuel:
                continue  # already reached with more fuel
            visited[nb] = nb_fuel
            queue.append((nb, nb_fuel))
    
    return connected, coverage, fertile


def balance_objective_v2(trial):
    """Bishop = low harvest, high capacity (supply wagon).
    Pawn = decent harvest, low capacity (field worker)."""
    
    queen_diet       = trial.suggest_float('queen_diet', 0.8, 3.0)
    pawn_diet        = trial.suggest_float('pawn_diet', 0.5, 2.5)
    pawn_harvest     = trial.suggest_float('pawn_harvest', 1.0, 4.0)
    pawn_capacity    = trial.suggest_float('pawn_capacity', 5, 30)
    bishop_diet      = trial.suggest_float('bishop_diet', 0.5, 3.0)
    bishop_harvest   = trial.suggest_float('bishop_harvest', 0.3, 2.0)  # BAD at harvest!
    bishop_capacity  = trial.suggest_float('bishop_capacity', 30, 200)  # BIG backpack
    soil_scale       = trial.suggest_float('soil_scale', 0.5, 4.0)
    
    score = 0.0
    
    scenarios = [
        # (label, feeder_diets, feeder_harvests, feeder_capacities, should_connect)
        ("queen+5pawns",
         [pawn_diet]*5, [pawn_harvest]*5, [pawn_capacity]*5, True),
        ("queen+4pawns",
         [pawn_diet]*4, [pawn_harvest]*4, [pawn_capacity]*4, False),
        ("queen+bishop",
         [bishop_diet], [bishop_harvest], [bishop_capacity], True),
        ("queen+3pawns",
         [pawn_diet]*3, [pawn_harvest]*3, [pawn_capacity]*3, False),
        ("queen+1pawn",
         [pawn_diet]*1, [pawn_harvest]*1, [pawn_capacity]*1, False),
        # Bishop without capacity advantage shouldn't work
        ("queen+bishop(small bag)",
         [bishop_diet], [bishop_harvest], [pawn_capacity], False),
    ]
    
    for label, diets, harvests, caps, should in scenarios:
        connected, coverage, _ = traversal_check_v2(
            terr, basin, grid, setts,
            queen_diet, diets, harvests, caps, soil_scale)
        
        if connected == should:
            score += 1.0
        else:
            score -= 1.5
    
    # Coverage sweet spot for the 5-pawn case
    _, cov5, _ = traversal_check_v2(
        terr, basin, grid, setts,
        queen_diet, [pawn_diet]*5, [pawn_harvest]*5,
        [pawn_capacity]*5, soil_scale)
    if 0.30 <= cov5 <= 0.55:
        score += 0.5
    
    return score


optuna.logging.set_verbosity(optuna.logging.WARNING)
study2 = optuna.create_study(direction='maximize')
study2.optimize(balance_objective_v2, n_trials=100, show_progress_bar=True)

bp = study2.best_params
print(f"\n{'='*50}")
print(f"Best score: {study2.best_value:.2f}")
print(f"\nBalanced params:")
for k, v in bp.items():
    print(f"  {k:20s}: {v:.3f}")

print(f"\nGameplay story:")
print(f"  Queen eats {bp['queen_diet']:.1f}/turn")
print(f"  Pawn: eats {bp['pawn_diet']:.1f}, harvests {bp['pawn_harvest']:.1f}, carries {bp['pawn_capacity']:.0f}")
print(f"  Bishop: eats {bp['bishop_diet']:.1f}, harvests {bp['bishop_harvest']:.1f}, carries {bp['bishop_capacity']:.0f}")
print(f"  Bishop carries {bp['bishop_capacity']/bp['pawn_capacity']:.1f}× a pawn's food")
print(f"  Bishop crosses ~{int(bp['bishop_capacity']/(bp['bishop_diet']+bp['queen_diet']))} barren hexes solo")


So the queen needs to each much more than a pawn.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Right — the optimizer gamed it! Queen eats 1.8 but pawns eat 2.1? That's backwards. The queen should be the expensive mouth to feed — that's what makes her a strategic liability you need to protect and supply.

Add the constraint:

```python
def balance_objective_v3(trial):
    """Bishop = supply wagon. Queen = hungry royalty."""
    
    pawn_diet        = trial.suggest_float('pawn_diet', 0.5, 2.0)
    queen_mult       = trial.suggest_float('queen_diet_mult', 1.5, 4.0)  # queen eats MORE
    queen_diet       = pawn_diet * queen_mult
    
    pawn_harvest     = trial.suggest_float('pawn_harvest', 1.0, 4.0)
    pawn_capacity    = trial.suggest_float('pawn_capacity', 5, 30)
    
    bishop_diet_mult = trial.suggest_float('bishop_diet_mult', 0.8, 1.5)  # similar to pawn
    bishop_diet      = pawn_diet * bishop_diet_mult
    bishop_harvest   = trial.suggest_float('bishop_harvest', 0.3, 1.5)  # BAD harvester
    bishop_capacity  = trial.suggest_float('bishop_capacity', 30, 200)
    
    soil_scale       = trial.suggest_float('soil_scale', 0.5, 4.0)
    
    score = 0.0
    
    scenarios = [
        ("queen+5pawns",
         [pawn_diet]*5, [pawn_harvest]*5, [pawn_capacity]*5, True),
        ("queen+4pawns",
         [pawn_diet]*4, [pawn_harvest]*4, [pawn_capacity]*4, False),
        ("queen+bishop",
         [bishop_diet], [bishop_harvest], [bishop_capacity], True),
        ("queen+3pawns",
         [pawn_diet]*3, [pawn_harvest]*3, [pawn_capacity]*3, False),
        ("queen+1pawn",
         [pawn_diet]*1, [pawn_harvest]*1, [pawn_capacity]*1, False),
        ("queen+bishop(small bag)",
         [bishop_diet], [bishop_harvest], [pawn_capacity], False),
    ]
    
    for label, diets, harvests, caps, should in scenarios:
        connected, coverage, _ = traversal_check_v2(
            terr, basin, grid, setts,
            queen_diet, diets, harvests, caps, soil_scale)
        
        if connected == should:
            score += 1.0
        else:
            score -= 1.5
    
    _, cov5, _ = traversal_check_v2(
        terr, basin, grid, setts,
        queen_diet, [pawn_diet]*5, [pawn_harvest]*5,
        [pawn_capacity]*5, soil_scale)
    if 0.30 <= cov5 <= 0.55:
        score += 0.5
    
    return score


optuna.logging.set_verbosity(optuna.logging.WARNING)
study3 = optuna.create_study(direction='maximize')
study3.optimize(balance_objective_v3, n_trials=100, show_progress_bar=True)

bp = study3.best_params
queen_diet = bp['pawn_diet'] * bp['queen_diet_mult']
bishop_diet = bp['pawn_diet'] * bp['bishop_diet_mult']

print(f"\n{'='*50}")
print(f"Best score: {study3.best_value:.2f}")
print(f"\nDerived params:")
print(f"  Queen diet:     {queen_diet:.2f}  ({bp['queen_diet_mult']:.1f}× pawn)")
print(f"  Pawn diet:      {bp['pawn_diet']:.2f}")
print(f"  Pawn harvest:   {bp['pawn_harvest']:.2f}")
print(f"  Pawn capacity:  {bp['pawn_capacity']:.0f}")
print(f"  Bishop diet:    {bishop_diet:.2f}  ({bp['bishop_diet_mult']:.1f}× pawn)")
print(f"  Bishop harvest: {bp['bishop_harvest']:.2f}")
print(f"  Bishop capacity:{bp['bishop_capacity']:.0f}")
print(f"  Soil scale:     {bp['soil_scale']:.2f}")
print(f"\nGameplay:")
print(f"  Queen is {bp['queen_diet_mult']:.1f}× hungrier than a pawn")
print(f"  Bishop carries {bp['bishop_capacity']/bp['pawn_capacity']:.0f}× a pawn's food")
print(f"  Bishop crosses ~{int(bp['bishop_capacity']/(bishop_diet+queen_diet))} barren hexes feeding the queen")
```

The key change: instead of independent `queen_diet`, we use `queen_diet_mult` (forced ≥ 1.5×) so the queen is *always* the hungriest piece. Now the optimizer has to make the food economy work around an expensive queen — which is exactly the strategic tension you want. 👑🍽️

In [ ]:
def balance_objective_v3(trial):
    """Bishop = supply wagon. Queen = hungry royalty."""
    
    pawn_diet        = trial.suggest_float('pawn_diet', 0.5, 2.0)
    queen_mult       = trial.suggest_float('queen_diet_mult', 1.5, 4.0)  # queen eats MORE
    queen_diet       = pawn_diet * queen_mult
    
    pawn_harvest     = trial.suggest_float('pawn_harvest', 1.0, 4.0)
    pawn_capacity    = trial.suggest_float('pawn_capacity', 5, 30)
    
    bishop_diet_mult = trial.suggest_float('bishop_diet_mult', 0.8, 1.5)  # similar to pawn
    bishop_diet      = pawn_diet * bishop_diet_mult
    bishop_harvest   = trial.suggest_float('bishop_harvest', 0.3, 1.5)  # BAD harvester
    bishop_capacity  = trial.suggest_float('bishop_capacity', 30, 200)
    
    soil_scale       = trial.suggest_float('soil_scale', 0.5, 4.0)
    
    score = 0.0
    
    scenarios = [
        ("queen+5pawns",
         [pawn_diet]*5, [pawn_harvest]*5, [pawn_capacity]*5, True),
        ("queen+4pawns",
         [pawn_diet]*4, [pawn_harvest]*4, [pawn_capacity]*4, False),
        ("queen+bishop",
         [bishop_diet], [bishop_harvest], [bishop_capacity], True),
        ("queen+3pawns",
         [pawn_diet]*3, [pawn_harvest]*3, [pawn_capacity]*3, False),
        ("queen+1pawn",
         [pawn_diet]*1, [pawn_harvest]*1, [pawn_capacity]*1, False),
        ("queen+bishop(small bag)",
         [bishop_diet], [bishop_harvest], [pawn_capacity], False),
    ]
    
    for label, diets, harvests, caps, should in scenarios:
        connected, coverage, _ = traversal_check_v2(
            terr, basin, grid, setts,
            queen_diet, diets, harvests, caps, soil_scale)
        
        if connected == should:
            score += 1.0
        else:
            score -= 1.5
    
    _, cov5, _ = traversal_check_v2(
        terr, basin, grid, setts,
        queen_diet, [pawn_diet]*5, [pawn_harvest]*5,
        [pawn_capacity]*5, soil_scale)
    if 0.30 <= cov5 <= 0.55:
        score += 0.5
    
    return score


optuna.logging.set_verbosity(optuna.logging.WARNING)
study3 = optuna.create_study(direction='maximize')
study3.optimize(balance_objective_v3, n_trials=100, show_progress_bar=True)

bp = study3.best_params
queen_diet = bp['pawn_diet'] * bp['queen_diet_mult']
bishop_diet = bp['pawn_diet'] * bp['bishop_diet_mult']

print(f"\n{'='*50}")
print(f"Best score: {study3.best_value:.2f}")
print(f"\nDerived params:")
print(f"  Queen diet:     {queen_diet:.2f}  ({bp['queen_diet_mult']:.1f}× pawn)")
print(f"  Pawn diet:      {bp['pawn_diet']:.2f}")
print(f"  Pawn harvest:   {bp['pawn_harvest']:.2f}")
print(f"  Pawn capacity:  {bp['pawn_capacity']:.0f}")
print(f"  Bishop diet:    {bishop_diet:.2f}  ({bp['bishop_diet_mult']:.1f}× pawn)")
print(f"  Bishop harvest: {bp['bishop_harvest']:.2f}")
print(f"  Bishop capacity:{bp['bishop_capacity']:.0f}")
print(f"  Soil scale:     {bp['soil_scale']:.2f}")
print(f"\nGameplay:")
print(f"  Queen is {bp['queen_diet_mult']:.1f}× hungrier than a pawn")
print(f"  Bishop carries {bp['bishop_capacity']/bp['pawn_capacity']:.0f}× a pawn's food")
print(f"  Bishop crosses ~{int(bp['bishop_capacity']/(bishop_diet+queen_diet))} barren hexes feeding the queen")


This seems reasonable. Are we making more desert?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes! `soil_scale = 0.51` — the optimizer is **halving** the world's fertility to create the knife-edge between 4 and 5 pawns. That works mathematically but might make the world feel bleak.

The question is: do you want the tension to come from **harsh terrain** or from **expensive queens**? Right now it's a bit of both. You could pin soil_scale to something reasonable and force the balance elsewhere:

```python
def balance_objective_v4(trial):
    """Same as v3 but soil_scale constrained to ≥ 1.0 — world stays lush."""
    
    pawn_diet        = trial.suggest_float('pawn_diet', 0.5, 2.0)
    queen_mult       = trial.suggest_float('queen_diet_mult', 1.5, 4.0)
    queen_diet       = pawn_diet * queen_mult
    
    pawn_harvest     = trial.suggest_float('pawn_harvest', 1.0, 4.0)
    pawn_capacity    = trial.suggest_float('pawn_capacity', 5, 30)
    
    bishop_diet_mult = trial.suggest_float('bishop_diet_mult', 0.8, 1.5)
    bishop_diet      = pawn_diet * bishop_diet_mult
    bishop_harvest   = trial.suggest_float('bishop_harvest', 0.3, 1.5)
    bishop_capacity  = trial.suggest_float('bishop_capacity', 30, 200)
    
    soil_scale       = trial.suggest_float('soil_scale', 1.0, 3.0)  # world stays fertile!
    
    score = 0.0
    
    scenarios = [
        ("queen+5pawns",
         [pawn_diet]*5, [pawn_harvest]*5, [pawn_capacity]*5, True),
        ("queen+4pawns",
         [pawn_diet]*4, [pawn_harvest]*4, [pawn_capacity]*4, False),
        ("queen+bishop",
         [bishop_diet], [bishop_harvest], [bishop_capacity], True),
        ("queen+3pawns",
         [pawn_diet]*3, [pawn_harvest]*3, [pawn_capacity]*3, False),
        ("queen+1pawn",
         [pawn_diet]*1, [pawn_harvest]*1, [pawn_capacity]*1, False),
        ("queen+bishop(small bag)",
         [bishop_diet], [bishop_harvest], [pawn_capacity], False),
    ]
    
    for label, diets, harvests, caps, should in scenarios:
        connected, coverage, _ = traversal_check_v2(
            terr, basin, grid, setts,
            queen_diet, diets, harvests, caps, soil_scale)
        
        if connected == should:
            score += 1.0
        else:
            score -= 1.5
    
    _, cov5, _ = traversal_check_v2(
        terr, basin, grid, setts,
        queen_diet, [pawn_diet]*5, [pawn_harvest]*5,
        [pawn_capacity]*5, soil_scale)
    if 0.30 <= cov5 <= 0.55:
        score += 0.5
    
    return score


study4 = optuna.create_study(direction='maximize')
study4.optimize(balance_objective_v4, n_trials=100, show_progress_bar=True)

bp = study4.best_params
queen_diet = bp['pawn_diet'] * bp['queen_diet_mult']
bishop_diet = bp['pawn_diet'] * bp['bishop_diet_mult']

print(f"\n{'='*50}")
print(f"Best score: {study4.best_value:.2f}")
print(f"\nDerived params:")
print(f"  Queen diet:     {queen_diet:.2f}  ({bp['queen_diet_mult']:.1f}× pawn)")
print(f"  Pawn diet:      {bp['pawn_diet']:.2f}")
print(f"  Pawn harvest:   {bp['pawn_harvest']:.2f}")
print(f"  Pawn capacity:  {bp['pawn_capacity']:.0f}")
print(f"  Bishop diet:    {bishop_diet:.2f}  ({bp['bishop_diet_mult']:.1f}× pawn)")
print(f"  Bishop harvest: {bp['bishop_harvest']:.2f}")
print(f"  Bishop capacity:{bp['bishop_capacity']:.0f}")
print(f"  Soil scale:     {bp['soil_scale']:.2f}")
print(f"\nGameplay:")
print(f"  Queen is {bp['queen_diet_mult']:.1f}× hungrier than a pawn")
print(f"  Bishop carries {bp['bishop_capacity']/bp['pawn_capacity']:.0f}× a pawn's food")
print(f"  Bishop crosses ~{int(bp['bishop_capacity']/(bishop_diet+queen_diet))} barren hexes feeding the queen")
```

Now the optimizer can't cheap out by scorching the earth — it has to make the *piece economics* do the work. A lush world where queens are just really expensive to feed is a much more interesting game narrative: the land is rich, but royalty has *standards* 👑🍷

In [ ]:
def balance_objective_v4(trial):
    """Same as v3 but soil_scale constrained to ≥ 1.0 — world stays lush."""
    
    pawn_diet        = trial.suggest_float('pawn_diet', 0.5, 2.0)
    queen_mult       = trial.suggest_float('queen_diet_mult', 1.5, 4.0)
    queen_diet       = pawn_diet * queen_mult
    
    pawn_harvest     = trial.suggest_float('pawn_harvest', 1.0, 4.0)
    pawn_capacity    = trial.suggest_float('pawn_capacity', 5, 30)
    
    bishop_diet_mult = trial.suggest_float('bishop_diet_mult', 0.8, 1.5)
    bishop_diet      = pawn_diet * bishop_diet_mult
    bishop_harvest   = trial.suggest_float('bishop_harvest', 0.3, 1.5)
    bishop_capacity  = trial.suggest_float('bishop_capacity', 30, 200)
    
    soil_scale       = trial.suggest_float('soil_scale', 1.0, 3.0)  # world stays fertile!
    
    score = 0.0
    
    scenarios = [
        ("queen+5pawns",
         [pawn_diet]*5, [pawn_harvest]*5, [pawn_capacity]*5, True),
        ("queen+4pawns",
         [pawn_diet]*4, [pawn_harvest]*4, [pawn_capacity]*4, False),
        ("queen+bishop",
         [bishop_diet], [bishop_harvest], [bishop_capacity], True),
        ("queen+3pawns",
         [pawn_diet]*3, [pawn_harvest]*3, [pawn_capacity]*3, False),
        ("queen+1pawn",
         [pawn_diet]*1, [pawn_harvest]*1, [pawn_capacity]*1, False),
        ("queen+bishop(small bag)",
         [bishop_diet], [bishop_harvest], [pawn_capacity], False),
    ]
    
    for label, diets, harvests, caps, should in scenarios:
        connected, coverage, _ = traversal_check_v2(
            terr, basin, grid, setts,
            queen_diet, diets, harvests, caps, soil_scale)
        
        if connected == should:
            score += 1.0
        else:
            score -= 1.5
    
    _, cov5, _ = traversal_check_v2(
        terr, basin, grid, setts,
        queen_diet, [pawn_diet]*5, [pawn_harvest]*5,
        [pawn_capacity]*5, soil_scale)
    if 0.30 <= cov5 <= 0.55:
        score += 0.5
    
    return score


study4 = optuna.create_study(direction='maximize')
study4.optimize(balance_objective_v4, n_trials=100, show_progress_bar=True)

bp = study4.best_params
queen_diet = bp['pawn_diet'] * bp['queen_diet_mult']
bishop_diet = bp['pawn_diet'] * bp['bishop_diet_mult']

print(f"\n{'='*50}")
print(f"Best score: {study4.best_value:.2f}")
print(f"\nDerived params:")
print(f"  Queen diet:     {queen_diet:.2f}  ({bp['queen_diet_mult']:.1f}× pawn)")
print(f"  Pawn diet:      {bp['pawn_diet']:.2f}")
print(f"  Pawn harvest:   {bp['pawn_harvest']:.2f}")
print(f"  Pawn capacity:  {bp['pawn_capacity']:.0f}")
print(f"  Bishop diet:    {bishop_diet:.2f}  ({bp['bishop_diet_mult']:.1f}× pawn)")
print(f"  Bishop harvest: {bp['bishop_harvest']:.2f}")
print(f"  Bishop capacity:{bp['bishop_capacity']:.0f}")
print(f"  Soil scale:     {bp['soil_scale']:.2f}")
print(f"\nGameplay:")
print(f"  Queen is {bp['queen_diet_mult']:.1f}× hungrier than a pawn")
print(f"  Bishop carries {bp['bishop_capacity']/bp['pawn_capacity']:.0f}× a pawn's food")
print(f"  Bishop crosses ~{int(bp['bishop_capacity']/(bishop_diet+queen_diet))} barren hexes feeding the queen")


I think somewhere close to this version. I don't want the game completely about food dynamics (we have after all combat), but enough where it should matter. Is there good food profile that I could update these piece that is only slightly more agressive than this? an @patch for GameParts?
```
erived params:
  Queen diet:     6.59  (3.3× pawn)
  Pawn diet:      1.98
  Pawn harvest:   1.80
  Pawn capacity:  10
  Bishop diet:    2.91  (1.5× pawn)
  Bishop harvest: 1.49
  Bishop capacity:90
  Soil scale:     0.51
```

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Here's a clean food profile with slightly more aggressive numbers — nice round values that tell a clear gameplay story:

```python
@patch
def apply_food_profile(self: GameParts, soil_scale: float = 1.0):
    """Set food stats on all pieces based on balanced profile.
    
    Design intent:
      - Queen is expensive to feed (3.5× pawn), can't self-sustain
      - Pawns are field workers: decent harvest, small pack
      - Bishops are supply wagons: bad harvest, huge capacity
      - Knights are fast but hungry scouts
      - Rooks are heavy but carry well
      - King eats like a queen (royalty!)
    """
    PROFILES = {
        #                  diet  harvest  capacity
        PieceType.PAWN:   (2.0,  2.0,     12),
        PieceType.BISHOP: (2.5,  1.0,    100),
        PieceType.KNIGHT: (3.0,  1.5,     20),
        PieceType.ROOK:   (3.5,  1.0,     40),
        PieceType.QUEEN:  (7.0,  0.5,     30),
        PieceType.KING:   (6.0,  0.5,     25),
    }
    
    count = {t.name: 0 for t in PieceType}
    for piece in self.board.all_pieces():
        profile = PROFILES.get(piece.piece_type)
        if profile is None:
            continue
        piece.diet, piece.harvest_strength, piece.food_capacity = profile
        piece.food = piece.food_capacity  # start full
        count[piece.piece_type.name] += 1
    
    # Apply soil scaling if needed
    if soil_scale != 1.0 and hasattr(self, 'terr'):
        fy = FoodYield(self.terr, self.basin)
        fy.soil_mult = [s * soil_scale for s in fy.soil_mult]
        fy.compute()
    
    print(f"Food profile applied:")
    for ptype, (d, h, c) in PROFILES.items():
        n = count.get(ptype.name, 0)
        net_t5 = 0.5 * h * 5 - d  # net at tier 5
        label = "✅" if net_t5 > 0 else "❌"
        print(f"  {ptype.name:8s} ×{n:2d}: diet={d:.1f} harvest={h:.1f} "
              f"cap={c:3d}  net@tier5={net_t5:+.1f} {label}")
    
    queen_d = PROFILES[PieceType.QUEEN][0]
    pawn_net = 0.5 * PROFILES[PieceType.PAWN][1] * 5 - PROFILES[PieceType.PAWN][0]
    bish_gap = PROFILES[PieceType.BISHOP][2] / (PROFILES[PieceType.BISHOP][0] + queen_d)
    print(f"\n  Pawns to sustain queen @tier5: {math.ceil(queen_d / max(pawn_net, 0.01))}")
    print(f"  Bishop gap crossing: ~{int(bish_gap)} barren hexes")
```

The narrative:

| Piece | Role | Diet | Harvest | Capacity | Story |
|-------|------|------|---------|----------|-------|
| Pawn | Field worker | 2.0 | 2.0 | 12 | Self-sustains on good land, small pack |
| Bishop | Supply wagon | 2.5 | 1.0 | 100 | Can't farm, but carries enough for 10+ hex gaps |
| Knight | Fast scout | 3.0 | 1.5 | 20 | Burns hot, forage OK, travels light |
| Rook | Heavy unit | 3.5 | 1.0 | 40 | Expensive, moderate storage, needs supply line |
| Queen | Royalty | 7.0 | 0.5 | 30 | Helpless alone, needs ~3 pawns or 1 bishop |
| King | Royalty | 6.0 | 0.5 | 25 | Slightly cheaper queen, still needs escort |

Queen at 7.0 diet with pawns netting ~3.0 at tier 5 means you need **3 pawns** on good land — leaving food as a real constraint without being the whole game. The bishop's 100 capacity lets a queen+bishop pair cross ~10 barren hexes, making bishops genuinely valuable for expeditions. Food matters, but you still have 2 pawns free to fight. 🗡️🌾

## Plan Mission

Love that framing. What you're describing is a **Resource-Constrained Shortest Path with Replenishment** — a real OR problem class. The military version is literally called **march-and-forage scheduling**.

The reactive state machine works, but it's greedy — it fills up completely before each march, which might waste turns. The optimal schedule might be: "march 2 hexes, camp 3 turns, march 2 more" instead of "camp 40 turns, march 4." A Pyomo model can find that.

Here's the formulation — for each time step, each piece chooses an action:

```python
def plan_mission(squad, target, grid, elevations, food_tiers, countries,
                 max_turns=80):
    """Joint schedule: when to march, when to camp, who feeds.
    
    Minimizes total turns to get leader to target while keeping everyone alive.
    """
    leader = squad.by_rank()[0]
    alive = squad.alive
    N = len(alive)
    
    # ── Leader's path (fixed route, variable pacing) ──
    lpath = leader.pathfind(target, grid, elevations, countries)
    H = len(lpath)  # waypoints along path
    T = max_turns
    
    # Food tiles at each waypoint
    wp_tiles = {}  # wp_index → [(tier, hex_idx), ...]
    for w, whex in enumerate(lpath.hexes):
        tiles = []
        for ring in range(1, 4):
            for hp in HexPosition.origin().ring(ring):
                idx = grid.hexposition_to_index(hp, whex)
                if idx < 0 or idx in grid.invalidRegion: continue
                if elevations[idx] <= 0: continue
                if countries is not None and countries[idx] < 0: continue
                tier = int(food_tiers[idx]) if 0 <= idx < len(food_tiers) else 0
                if tier >= 1: tiles.append((tier, idx))
        tiles.sort(key=lambda c: -c[0])
        wp_tiles[w] = tiles[:N]  # at most one per piece
    
    K = max(len(v) for v in wp_tiles.values()) if wp_tiles else 0
    
    diet = {i: alive[i].diet for i in range(N)}
    harvest = {i: alive[i].harvest_strength for i in range(N)}
    cap = {i: alive[i].food_capacity for i in range(N)}
    food0 = {i: alive[i].food for i in range(N)}
    is_pawn = {i: alive[i].piece_type == PieceType.PAWN for i in range(N)}
    
    m = pyo.ConcreteModel()
    m.T = pyo.RangeSet(0, T - 1)
    m.P = pyo.RangeSet(0, N - 1)    # pieces
    m.W = pyo.RangeSet(0, H - 1)    # waypoints
    
    # ── Decision variables ──
    
    # march[t] = 1 if squad advances one waypoint at time t
    m.march = pyo.Var(m.T, domain=pyo.Binary)
    
    # position[t] = waypoint index the squad is at after turn t
    m.pos = pyo.Var(m.T, domain=pyo.NonNegativeIntegers, bounds=(0, H - 1))
    
    # feeding[p, t] = 1 if piece p is harvesting at time t
    m.feeding = pyo.Var(m.P, m.T, domain=pyo.Binary)
    
    # food[p, t] = food level of piece p after turn t
    m.food = pyo.Var(m.P, m.T, domain=pyo.NonNegativeReals)
    
    # arrival = first turn the leader reaches target
    m.arrival = pyo.Var(domain=pyo.NonNegativeIntegers, bounds=(0, T))
    
    # arrived[t] = 1 if squad has reached target by turn t
    m.arrived = pyo.Var(m.T, domain=pyo.Binary)
    
    # ── Objective: minimize arrival turn ──
    m.obj = pyo.Objective(expr=m.arrival, sense=pyo.minimize)
    
    # ── Position tracking ──
    m.pos_init = pyo.Constraint(expr=m.pos[0] == m.march[0])
    m.pos_update = pyo.Constraint(m.T, rule=lambda m, t:
        m.pos[t] == sum(m.march[tt] for tt in m.T if tt <= t))
    
    # ── Can't march and feed same turn ──
    m.no_march_feed = pyo.Constraint(m.P, m.T, rule=lambda m, p, t:
        m.feeding[p, t] + m.march[t] <= 1)
    
    # ── Only pawns can feed ──
    m.pawn_only = pyo.Constraint(m.P, m.T, rule=lambda m, p, t:
        m.feeding[p, t] <= (1 if is_pawn[p] else 0))
    
    # ── Food dynamics ──
    # Average tier at current waypoint (simplified)
    avg_tier = {}
    for w in range(H):
        tiles = wp_tiles.get(w, [])
        avg_tier[w] = sum(t for t, _ in tiles) / max(len(tiles), 1) if tiles else 0
    
    # Simplified: food[p,t] = food[p,t-1] - diet + feeding[p,t] * harvest * avg_tier
    def food_rule(m, p, t):
        if t == 0:
            # Sum avg_tier over possible positions (linearization trick)
            prod = harvest[p] * sum(
                avg_tier[w] for w in range(H)) / max(H, 1)  # rough average
            return m.food[p, 0] == food0[p] - diet[p] + m.feeding[p, 0] * prod
        prod = harvest[p] * sum(
            avg_tier[w] for w in range(H)) / max(H, 1)
        return m.food[p, t] == m.food[p, t-1] - diet[p] + m.feeding[p, t] * prod
    m.food_dyn = pyo.Constraint(m.P, m.T, rule=food_rule)
    
    # ── Food capacity ──
    m.food_cap = pyo.Constraint(m.P, m.T, rule=lambda m, p, t:
        m.food[p, t] <= cap[p])
    
    # ── Nobody dies ──
    m.alive = pyo.Constraint(m.P, m.T, rule=lambda m, p, t:
        m.food[p, t] >= 0.5)
    
    # ── Must reach target ──
    m.must_arrive = pyo.Constraint(expr=m.pos[T-1] >= H - 1)
    
    # ── Arrival tracking ──
    m.arr_link = pyo.Constraint(m.T, rule=lambda m, t:
        m.arrival <= t + T * (1 - m.arrived[t]))
    m.arr_pos = pyo.Constraint(m.T, rule=lambda m, t:
        m.pos[t] >= (H - 1) * m.arrived[t])
    
    # ── Solve ──
    solver = pyo.SolverFactory('appsi_highs')
    result = solver.solve(m)
    
    if not _solver_ok(result):
        return None
    
    # ── Extract schedule ──
    schedule = []
    for t in m.T:
        turn = {
            'turn': t,
            'march': pyo.value(m.march[t]) > 0.5,
            'pos': int(round(pyo.value(m.pos[t]))),
            'feeders': [alive[p].name for p in m.P if pyo.value(m.feeding[p, t]) > 0.5],
            'food': {alive[p].name: round(pyo.value(m.food[p, t]), 1) for p in m.P},
        }
        schedule.append(turn)
    
    return schedule
```

But honestly — this is a **rough sketch**. The hard part is the bilinear term: `feeding[p,t] × avg_tier_at_position[t]` — the production rate depends on *where* you are, and where you are depends on march decisions. That makes it nonlinear.

There are three ways to handle it:

| Approach | Complexity | Quality |
|---|---|---|
| **Fix the route, optimize the pacing** | MIP (linear) | Good — route from pathfind, just decide march/camp per turn |
| **Discretize into legs** | Small MIP | Great — "how many camp turns at each waypoint?" |
| **Full joint route+schedule** | MINLP | Optimal but hard to solve |

The middle option is the sweet spot — the **"leap looker"**:

```python
def leap_schedule(squad, path, grid, elevations, food_tiers, countries,
                  max_camp_per_stop=40):
    """For each waypoint on the path, decide how many turns to camp.
    
    Minimize: sum of camp turns (= total journey time beyond pure march)
    Subject to: nobody's food drops to 0 at any point
    """
    # ... waypoints from path
    # Decision var: camp_turns[w] = integer, how long to camp at waypoint w
    # Food is tracked analytically through the schedule
    # Linear MIP — very fast
```

This is your "schedule frog" — it sees the whole journey and decides the **optimal camping schedule** in one shot. Want me to flesh out the discretized leg version? It's the cleanest and most solvable.